In [1]:
print('start')

start


In [2]:
import numpy as np
import pandas as pd
import re
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression  # LogisticRegression is not used for regression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [3]:
def train_and_test_predict(models, X_train, y_train, X_test, y_test):
    kf = KFold(n_splits=5, shuffle=True, random_state=101)
    results = {}
    predictions = []  

    for model in models:
        model_name = model.__class__.__name__
        predictions_train = []
        actual_y_train = []

        test_predictions_folds = []

        

        for train_index, val_index in kf.split(X_train):
            X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
            y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

            model.fit(X_train_fold, y_train_fold)

            y_pred_fold = model.predict(X_val_fold)
            y_pred_fold = np.clip(y_pred_fold, -10, -4.0)
            predictions_train.extend(y_pred_fold)
            actual_y_train.extend(y_val_fold)

            predictions_test_fold = model.predict(X_test)
            predictions_test_fold = np.clip(predictions_test_fold, -10, -4.0)
            test_predictions_folds.append(predictions_test_fold)


        mse_train = mean_squared_error(actual_y_train, predictions_train)
        mae_train = mean_absolute_error(actual_y_train, predictions_train)
        rmse_train = np.sqrt(mse_train)
        r2_train = r2_score(actual_y_train, predictions_train)
        pearson_train, _ = pearsonr(actual_y_train, predictions_train)
        spearman_train, _ = spearmanr(actual_y_train, predictions_train)


        predictions_test_mean = np.mean(test_predictions_folds, axis=0)
        predictions_test_std = np.std(test_predictions_folds, axis=0)

        mse_test = mean_squared_error(y_test, predictions_test_mean)
        mae_test = mean_absolute_error(y_test, predictions_test_mean)
        rmse_test = np.sqrt(mse_test)
        r2_test = r2_score(y_test, predictions_test_mean)
        print(r2_test)
        pearson_test, _ = pearsonr(y_test, predictions_test_mean)
        spearman_test, _ = spearmanr(y_test, predictions_test_mean)
        
        

        predictions.append({
            'Model': model_name,
            'Y Train pred': predictions_train,
            'Y Test actual': y_test,
            'Test prediction folds': test_predictions_folds,
            'Test Predictions Mean': predictions_test_mean,
            'Test Predictions Std': predictions_test_std,

        })

        results[model_name] = {
            'Train MSE (5 fold cv)': f"{mse_train:.4f}",
            'Train MAE (5 fold cv)': f"{mae_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train R2 (5 fold cv)': f"{r2_train:.4f}",
            'Train PCC (5 fold cv)': f"{pearson_train:.4f}",
            'Train SCC (5 fold cv)': f"{spearman_train:.4f}",
            'Test MSE': f"{mse_test:.4f}",
            'Test MAE': f"{mae_test:.4f}",
            'Test RMSE': f"{rmse_test:.4f}",
            'Test R2': f"{r2_test:.4f}",
            'Test Pearson Correlation': f"{pearson_test:.4f}",
            'Test Spearman Correlation': f"{spearman_test:.4f}",
        }

    results_df = pd.DataFrame(results).T
    predictions_df = pd.DataFrame(predictions)

    return results_df, predictions_df



In [4]:
#Monomeric models
def clean_feature_names(df):
    def clean_name(name):
        return re.sub(r'[^a-zA-Z0-9_]', '_', name)

    df.columns = [clean_name(col) for col in df.columns]
    return df

In [5]:
#Monomer composition
df_mc_train = pd.read_csv('features/Monomeric/Train_mon_comp_MDCK.csv')
df_mc_train = clean_feature_names(df_mc_train)
X_train = df_mc_train.drop(['ID','SMILES','Permeability'], axis=1)
y_train = df_mc_train['Permeability']
print(X_train.shape)
print(y_train.shape)
df_mc_test = pd.read_csv('features/Monomeric/Test_mon_comp_MDCK.csv')
df_mc_test = clean_feature_names(df_mc_test)
X_test = df_mc_test.drop(['ID','SMILES','Permeability'], axis=1)
y_test = df_mc_test['Permeability']
print(X_test.shape)
print(y_test.shape)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    xgb.XGBRegressor(random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    SVR(),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3), 
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

(51, 385)
(51,)
(13, 385)
(13,)
0.24137470795774052
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.074661 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 1
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

-1.4021185217272363


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
ExtraTreesRegressor,0.4179,0.5002,0.6465,0.2058,0.5443,0.3419,0.5425,0.5566,0.7366,0.2414,0.4917,0.3034
LGBMRegressor,0.5254,0.5894,0.7248,0.0015,0.1082,0.0861,0.6872,0.6663,0.8290,0.0390,0.2219,0.1983
XGBRegressor,0.4153,0.5114,0.6444,0.2107,0.5572,0.3560,0.4822,0.4825,0.6944,0.3258,0.5812,0.4828
DecisionTreeRegressor,0.4898,0.5322,0.6998,0.0692,0.5283,0.3758,0.5158,0.4853,0.7182,0.2787,0.5317,0.4579
RandomForestRegressor,0.3577,0.4852,0.5981,0.3202,0.5659,0.4127,0.5061,0.5453,0.7114,0.2923,0.6037,0.5021
GradientBoostingRegressor,0.3867,0.5063,0.6218,0.2651,0.5674,0.3542,0.5421,0.5059,0.7363,0.2420,0.5325,0.3117
AdaBoostRegressor,0.3451,0.4701,0.5874,0.3442,0.5984,0.3925,0.5033,0.5332,0.7095,0.2961,0.6215,0.5967
SVR,0.3890,0.5044,0.6237,0.2606,0.5213,0.4857,0.4893,0.5235,0.6995,0.3158,0.7591,0.7172
LinearRegression,0.6761,0.6349,0.8222,-0.2849,0.4455,0.4633,0.6205,0.5154,0.7877,0.1323,0.5933,0.6152
KNeighborsRegressor,0.4823,0.5648,0.6944,0.0834,0.4117,0.3694,0.4557,0.4959,0.6750,0.3628,0.7068,0.6740


In [6]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,ExtraTreesRegressor,"[-6.126140111860008, -5.955900000000001, -5.47...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.013480737620001, -6.005418124430003, -5.3...","[-5.945573893316, -5.731207322658002, -5.49313...","[0.1916020808904238, 0.16583922517478042, 0.33..."
1,LGBMRegressor,"[-5.552886250226585, -5.671700068563222, -5.67...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.552886250226585, -5.552886250226585, -5.6...","[-5.4345869291739834, -5.4345869291739834, -5....","[0.12878170475806108, 0.12878170475806108, 0.1..."
2,XGBRegressor,"[-6.2152042, -6.3192315, -5.3157883, -5.444396...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.037485, -5.9965076, -6.291835, -5.2926407...","[-5.668134, -5.4515257, -5.833188, -5.35652, -...","[0.3809242, 0.31747642, 0.4418105, 0.12930405,..."
3,DecisionTreeRegressor,"[-6.25, -6.3, -5.32, -5.493665733, -7.69897000...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.25, -6.25, -6.4, -5.289036881, -5.1857524...","[-5.7986888299, -5.599688829900001, -6.0840000...","[0.38621153532374525, 0.40243198719451273, 0.6..."
4,RandomForestRegressor,"[-5.906521078232675, -6.016399999999996, -5.63...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.851980502656002, -5.781486805702673, -5.6...","[-5.612494837101869, -5.504497738960203, -5.59...","[0.1765006622543483, 0.1638605998984399, 0.095..."
5,GradientBoostingRegressor,"[-6.11046621474244, -6.186649156985524, -5.731...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.630876598629007, -5.551326527085098, -5.4...","[-5.401987871929767, -5.360877744659817, -5.61...","[0.24636405284723464, 0.12500477084802666, 0.1..."
6,AdaBoostRegressor,"[-5.789744656320002, -6.21, -5.511535693291665...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.615060854615384, -5.615060854615384, -5.2...","[-5.4982361934959325, -5.430308046137226, -5.4...","[0.11984158142805817, 0.11394788750222858, 0.1..."
7,SVR,"[-5.470219418986547, -5.599631534055634, -5.78...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.576693769842984, -5.48283127657874, -5.29...","[-5.475733689316651, -5.613632710966476, -5.33...","[0.15849712786487555, 0.06819570636252266, 0.0..."
8,LinearRegression,"[-4.819502825783285, -4.451005595778877, -5.92...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.0, -4.187784298384316, -5.792675095231238...","[-4.287270887621956, -4.851696895252494, -6.28...","[0.5745417752439131, 0.5192652583283262, 0.399..."
9,KNeighborsRegressor,"[-5.1866666666666665, -5.433333333333334, -5.4...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.426642538, -5.0966666...","[-5.528666666666667, -5.289537972266666, -5.01...","[0.1545689346393881, 0.10181842386894503, 0.15..."


In [7]:
result_df.to_csv('results/Monomeric/Monomer_comp_results_MDCK.csv')
prediction_df.to_csv('results/Monomeric/Monomer_comp_prediction_data_MDCK.csv')

In [8]:
#Removal of constant columns
def remove_constant_columns(df):
    constant_columns = [col for col in df.columns if df[col].nunique() <= 1]
    
    df_cleaned = df.drop(columns=constant_columns)
    
    return df_cleaned, constant_columns

In [9]:
df_mc_train = pd.read_csv('features/Monomeric/Train_mon_comp_MDCK.csv')
df_mc_train = clean_feature_names(df_mc_train)
df_mc_train, const_col = remove_constant_columns(df_mc_train)
X_train = df_mc_train.drop(['ID','SMILES','Permeability'], axis=1)
y_train = df_mc_train['Permeability']
print(X_train.shape)
print(y_train.shape)
df_mc_test = pd.read_csv('features/Monomeric/Test_mon_comp_MDCK.csv')
df_mc_test = clean_feature_names(df_mc_test)
X_test = df_mc_test.drop(['ID','SMILES','Permeability'], axis=1)
X_test = X_test.drop(const_col, axis=1)
y_test = df_mc_test['Permeability']
print(X_test.shape)
print(y_test.shape)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    xgb.XGBRegressor(random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    SVR(),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3), 
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

(51, 33)
(51,)
(13, 33)
(13,)
0.23321748973727008
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.046297 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 1
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Wa

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
ExtraTreesRegressor,0.4071,0.4954,0.6381,0.2262,0.5569,0.3767,0.5483,0.5597,0.7405,0.2332,0.4841,0.3034
LGBMRegressor,0.5254,0.5894,0.7248,0.0015,0.1082,0.0861,0.6872,0.6663,0.8290,0.0390,0.2219,0.1983
XGBRegressor,0.4153,0.5114,0.6444,0.2107,0.5572,0.3560,0.4822,0.4825,0.6944,0.3258,0.5812,0.4828
DecisionTreeRegressor,0.4748,0.5165,0.6891,0.0976,0.5292,0.3557,0.4672,0.4633,0.6836,0.3466,0.5907,0.5269
RandomForestRegressor,0.3583,0.4867,0.5986,0.3190,0.5649,0.4020,0.4990,0.5428,0.7064,0.3022,0.6131,0.5131
GradientBoostingRegressor,0.3828,0.5050,0.6187,0.2725,0.5693,0.3588,0.5279,0.5006,0.7266,0.2618,0.5506,0.4166
AdaBoostRegressor,0.3508,0.4713,0.5923,0.3333,0.5931,0.3582,0.5351,0.5608,0.7315,0.2517,0.5437,0.4193
SVR,0.3890,0.5045,0.6237,0.2606,0.5213,0.4857,0.4893,0.5235,0.6995,0.3158,0.7591,0.7172
LinearRegression,0.6761,0.6349,0.8222,-0.2849,0.4455,0.4633,0.6205,0.5154,0.7877,0.1323,0.5933,0.6152
KNeighborsRegressor,0.4761,0.5648,0.6900,0.0951,0.4195,0.3716,0.4688,0.5027,0.6847,0.3445,0.6968,0.6740


In [10]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,ExtraTreesRegressor,"[-6.09670000000001, -6.0676999999999985, -5.53...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.939036657330002, -5.947883932280004, -5.4...","[-5.931154331832, -5.744788147007001, -5.46681...","[0.2013143635874024, 0.13268640664966033, 0.28..."
1,LGBMRegressor,"[-5.552886250226585, -5.671700068563222, -5.67...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.552886250226585, -5.552886250226585, -5.6...","[-5.4345869291739834, -5.4345869291739834, -5....","[0.12878170475806108, 0.12878170475806108, 0.1..."
2,XGBRegressor,"[-6.2152042, -6.3192315, -5.3157883, -5.444396...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.037485, -5.9965076, -6.291835, -5.2926407...","[-5.668134, -5.4515257, -5.833188, -5.35652, -...","[0.3809242, 0.31747642, 0.4418105, 0.12930405,..."
3,DecisionTreeRegressor,"[-6.22, -6.17, -5.1, -5.493665733, -7.69897000...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.25, -6.25, -6.4, -5.289036881, -5.1857524...","[-5.519, -5.4483378892, -6.0840000000000005, -...","[0.6690321367468082, 0.5980063521010841, 0.632..."
4,RandomForestRegressor,"[-5.880915490890009, -5.933675982524997, -5.63...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.845710398403337, -5.780429195543338, -5.6...","[-5.616266369216002, -5.507914621284003, -5.60...","[0.16590938559267723, 0.1434599055456683, 0.10..."
5,GradientBoostingRegressor,"[-6.100183029276827, -6.197972311450167, -5.73...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.630561325596315, -5.551326527085098, -5.4...","[-5.4040654040563885, -5.366108777094103, -5.6...","[0.25247067271474743, 0.12304483683230462, 0.1..."
6,AdaBoostRegressor,"[-5.793596479153847, -6.004285714285714, -5.85...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.745454545454546, -5.5861452474615385, -4....","[-5.547713805282128, -5.495651377234732, -5.28...","[0.1544685030759875, 0.09823586459230899, 0.22..."
7,SVR,"[-5.470219537785439, -5.599631184612745, -5.78...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.5766940061426675, -5.4828313097403685, -5...","[-5.475735129321194, -5.613630964511478, -5.33...","[0.1584975996565043, 0.06819459330282343, 0.06..."
8,LinearRegression,"[-4.819502825783286, -4.451005595778879, -5.92...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.0, -4.1877842983843205, -5.79267509523123...","[-4.287270887621954, -4.851696895252497, -6.28...","[0.5745417752439086, 0.5192652583283319, 0.399..."
9,KNeighborsRegressor,"[-5.1866666666666665, -5.433333333333334, -5.4...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.426642538, -5.0966666...","[-5.528666666666667, -5.289537972266666, -5.01...","[0.1545689346393881, 0.10181842386894503, 0.15..."


In [11]:
const_col

['Ala_tBu_',
 'Ala_indol_2_yl_',
 'dAla_indol_2_yl_',
 'Me_Ala_indol_2_yl_',
 'Ala_5_Tet_',
 'Me_dAbu',
 'Me_Abu_morpholino_',
 '2Abz',
 'Aib',
 'Aoc_2_',
 '5_Ava',
 'Bal',
 'Me_Bal',
 'HOCOCH2_Bal',
 'Cys_EtO2H__NH2',
 'dCha',
 'Me_Cha',
 'D',
 'meD',
 'Asp_piperidide',
 'Asp_OMe_',
 'Asp_Ph_2_NH2__',
 'dAsp_pyrrol_1_yl_',
 'E',
 'Glu_NH2',
 'Glu_3R_Me_',
 'Glu_OMe_',
 'dGlu_OMe_',
 'Phe_4_F_',
 'dPhe_4_F_',
 'Phe_4_CF3_',
 'Phe_4_NO2_',
 'Phe_CHF2_',
 'dPhe_3_4_diF_',
 'Et_Phe',
 'H2NEt_Phe',
 'Me_Phe_3_Cl_',
 'Me_Phe_4_Cl_',
 'Me_Phe_a_b_dehydro_',
 'G',
 'Bn_Gly',
 'Bn_4_Cl__Gly',
 'Bn_4_OH__Gly',
 'Bu_Gly',
 'EtOEt_Gly',
 'HOCOCH2_Gly_ol',
 'MeOEt_Gly',
 'NH2Bu_Gly',
 'PhEt_Gly',
 'PhPr_Gly',
 'isoamyl_Gly',
 'pentyl_Gly',
 '3_pyridylethyl_Gly',
 '2_pyridylmethyl_Gly',
 'd_N__O_Gly_allyl_',
 'GABA',
 'H',
 'Hph',
 'Me_Hph',
 'bHph',
 'Hph_2_Cl_',
 'Hph_3_Cl_',
 'Hph_4_Cl_',
 'Hse_Et_',
 'dHyp',
 'Hyp_Et_',
 'dI',
 'meI',
 'Me_dI',
 '_N__O_xiIle',
 'd_N__O_aIle',
 'K',
 'dK',
 'meK

In [12]:
result_df.to_csv('results/Monomeric/Monomer_comp_constRemoval_results_MDCK.csv')
prediction_df.to_csv('results/Monomeric/Monomer_comp_constRemoval_prediction_data_MDCK.csv')

In [13]:
#Low variance column removal
def remove_low_variance_columns(df, threshold=0.005):
    variances = df.var()
    
    low_variance_columns = variances[variances < threshold].index.tolist()
    
    df_cleaned = df.drop(columns=low_variance_columns)
    
    return df_cleaned, low_variance_columns

In [14]:
df_train = pd.read_csv('features/Monomeric/Train_mon_comp_MDCK.csv')
df_mc_train = clean_feature_names(df_train)
df_mc_train = df_mc_train.drop(['ID','SMILES','Permeability'],axis=1)
df_mc, const_col = remove_low_variance_columns(df_mc_train)
X_train = df_mc
y_train = df_train['Permeability']
print(X_train.shape)
print(y_train.shape)

df_mc_test = pd.read_csv('features/Monomeric/Test_mon_comp_MDCK.csv')
df_mc_test = clean_feature_names(df_mc_test)
X_test = df_mc_test.drop(['ID','SMILES','Permeability'], axis=1)
X_test = X_test.drop(const_col, axis=1)
y_test = df_mc_test['Permeability']
print(X_test.shape)
print(y_test.shape)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    xgb.XGBRegressor(random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    SVR(),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3), 
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

(51, 7)
(51,)
(13, 7)
(13,)
0.2268169642204767
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.071744 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 1
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warni

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
ExtraTreesRegressor,0.4669,0.5363,0.6833,0.1126,0.4959,0.2344,0.5529,0.5473,0.7436,0.2268,0.4765,0.4807
LGBMRegressor,0.5254,0.5894,0.7248,0.0015,0.1082,0.0861,0.6872,0.6663,0.8290,0.0390,0.2219,0.1983
XGBRegressor,0.5553,0.5597,0.7452,-0.0554,0.4513,0.2157,0.4818,0.4760,0.6941,0.3263,0.5759,0.7238
DecisionTreeRegressor,0.5402,0.5642,0.7350,-0.0267,0.4907,0.2901,0.5480,0.5042,0.7403,0.2337,0.5049,0.5242
RandomForestRegressor,0.4292,0.5346,0.6551,0.1844,0.4523,0.3015,0.4775,0.5311,0.6910,0.3323,0.6464,0.6713
GradientBoostingRegressor,0.4975,0.5525,0.7054,0.0544,0.4663,0.2562,0.4798,0.4918,0.6927,0.3290,0.5790,0.6492
AdaBoostRegressor,0.4413,0.5087,0.6643,0.1612,0.4837,0.2752,0.5042,0.5471,0.7101,0.2950,0.6228,0.6133
SVR,0.5718,0.5762,0.7562,-0.0867,0.1512,0.2114,0.4633,0.5630,0.6807,0.3521,0.7520,0.6740
LinearRegression,0.4972,0.5549,0.7051,0.0551,0.3047,0.2996,0.5446,0.5929,0.7379,0.2385,0.5262,0.3757
KNeighborsRegressor,0.6990,0.6568,0.8361,-0.3285,0.0831,0.1106,0.6035,0.6308,0.7769,0.1561,0.4199,0.4523


In [15]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,ExtraTreesRegressor,"[-6.220000000000016, -6.185999999999999, -5.47...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.220000000000016, -6.093953592675014, -5.6...","[-6.019999999999998, -5.907136996114456, -5.71...","[0.12649110640673977, 0.1275305124374541, 0.31..."
1,LGBMRegressor,"[-5.552886250226585, -5.671700068563222, -5.67...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.552886250226585, -5.552886250226585, -5.6...","[-5.4345869291739834, -5.4345869291739834, -5....","[0.12878170475806108, 0.12878170475806108, 0.1..."
2,XGBRegressor,"[-6.218225, -6.2831087, -5.99926, -5.399957, -...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.218225, -6.2179446, -5.4919643, -5.516775...","[-6.018996, -5.95082, -5.6083817, -5.4483557, ...","[0.12658179, 0.20637104, 0.43822455, 0.1363716..."
3,DecisionTreeRegressor,"[-6.22, -6.3, -6.05, -5.399830559166666, -7.69...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.22, -6.22, -5.61, -5.516882188, -5.399830...","[-6.02, -6.02, -5.610000000000001, -5.42550575...","[0.126491106406735, 0.126491106406735, 0.49963..."
4,RandomForestRegressor,"[-5.894087501324485, -5.965699999999998, -6.10...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.894087501324485, -5.901987728656715, -5.7...","[-5.843193696110073, -5.7295767432302736, -5.7...","[0.1341149389725133, 0.13713496888179186, 0.20..."
5,GradientBoostingRegressor,"[-6.046361208926163, -6.045069315898642, -6.11...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.046361208926163, -5.985666812857362, -5.7...","[-5.9120141589914805, -5.746668712529173, -5.8...","[0.10179287578583175, 0.16074547343932089, 0.3..."
6,AdaBoostRegressor,"[-5.641333333333334, -5.751320243111112, -6.05...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.641333333333334, -5.641333333333334, -5.6...","[-5.556604736618939, -5.578445055341666, -5.59...","[0.10420634095537598, 0.07448245701164503, 0.3..."
7,SVR,"[-5.7647500886207235, -6.121808170991155, -5.9...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.7647500886207235, -5.9206606343342685, -5...","[-5.72907866398883, -5.746360174706517, -5.686...","[0.017835797849759884, 0.1074061055237505, 0.1..."
8,LinearRegression,"[-5.197302493534164, -5.703136488479879, -5.84...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.197302493534164, -5.090276185319464, -5.6...","[-5.216765591263774, -5.147924838266224, -5.79...","[0.0552686864153892, 0.1281576338951431, 0.204..."
9,KNeighborsRegressor,"[-5.6026857393333325, -5.760000000000001, -6.1...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.6026857393333325, -5.6026857393333325, -5...","[-5.687246221466666, -5.687246221466666, -5.52...","[0.1207157359727746, 0.1207157359727746, 0.194..."


In [16]:
result_df.to_csv('results/Monomeric/Monomer_comp_LVR_results_MDCK.csv')
prediction_df.to_csv('results/Monomeric/Monomer_comp_LVR_prediction_data_MDCK.csv')

In [17]:
#AA composition
df_aac_train = pd.read_csv('features/Monomeric/Train_aac_MDCK.csv')
X_train = df_aac_train.drop(['ID','SMILES','Permeability'], axis=1)
y_train = df_aac_train['Permeability']
print(X_train.shape)
print(y_train.shape)
df_aac_test = pd.read_csv('features/Monomeric/Test_aac_MDCK.csv')
X_test = df_aac_test.drop(['ID','SMILES','Permeability'], axis=1)
y_test = df_aac_test['Permeability']
print(X_test.shape)
print(y_test.shape)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    xgb.XGBRegressor(random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    SVR(),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3), 
    MLPRegressor(random_state=101)
]
aac_comp,prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
aac_comp

(51, 21)
(51,)
(13, 21)
(13,)
0.2869051619662686
[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 0
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training b

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
ExtraTreesRegressor,0.6822,0.6573,0.8260,-0.2966,0.2124,0.2568,0.5100,0.5760,0.7141,0.2869,0.5835,0.3715
LGBMRegressor,0.4912,0.5710,0.7008,0.0665,0.2632,0.2534,0.6428,0.6748,0.8017,0.1011,0.4245,0.2624
XGBRegressor,0.6525,0.6562,0.8078,-0.2402,0.2333,0.2785,0.6064,0.6409,0.7787,0.1520,0.4245,0.3384
DecisionTreeRegressor,0.7501,0.7220,0.8661,-0.4255,0.1803,0.2221,0.6218,0.6422,0.7885,0.1305,0.3882,0.1644
RandomForestRegressor,0.6118,0.6388,0.7822,-0.1628,0.2356,0.2904,0.5286,0.5993,0.7270,0.2608,0.5520,0.3384
GradientBoostingRegressor,0.6781,0.6781,0.8234,-0.2887,0.2040,0.2479,0.5481,0.6033,0.7403,0.2336,0.5246,0.3715
AdaBoostRegressor,0.5522,0.6265,0.7431,-0.0495,0.3331,0.3374,0.5690,0.6149,0.7544,0.2043,0.4797,0.3881
SVR,0.5905,0.6390,0.7684,-0.1223,0.1089,0.0993,0.5163,0.5792,0.7186,0.2780,0.6294,0.3743
LinearRegression,0.5622,0.6091,0.7498,-0.0684,0.2674,0.2923,0.5957,0.6316,0.7718,0.1671,0.4744,0.3550
KNeighborsRegressor,0.6975,0.6959,0.8352,-0.3256,0.1993,0.2669,0.5634,0.5754,0.7506,0.2122,0.5638,0.4406


In [18]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,ExtraTreesRegressor,"[-5.2950000000000035, -5.7300000000000075, -5....",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.451550000000003, -6.25, -5.29500000000000...","[-5.242879229763606, -5.971531887076002, -5.47...","[0.4136185335773759, 0.5569362258479963, 0.147..."
1,LGBMRegressor,"[-5.6122931599617, -5.6122931599617, -5.612293...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.6122931599617, -5.6122931599617, -5.61229...","[-5.390906233628531, -5.553378645833561, -5.39...","[0.17169618719084973, 0.06934332374565914, 0.1..."
2,XGBRegressor,"[-5.2957315, -5.7299514, -5.7299514, -5.678163...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.6846294, -6.2477527, -5.2957315, -5.29089...","[-4.873517, -5.9578676, -5.470054, -5.4094343,...","[0.4134306, 0.5801909, 0.14739683, 0.23752233,..."
3,DecisionTreeRegressor,"[-5.295, -5.7299999999999995, -5.7299999999999...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.61, -6.25, -5.295, -5.289036881, -5.18575...","[-4.8384, -5.948, -5.470000000000001, -5.51122...","[0.37336609380070945, 0.604, 0.147614362444851..."
4,RandomForestRegressor,"[-5.346065714285714, -5.7221678282828305, -5.7...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.234702752356904, -6.009662235409787, -5.3...","[-5.2110772275872685, -5.760780379919408, -5.4...","[0.19695054069031223, 0.2909711874005021, 0.10..."
5,GradientBoostingRegressor,"[-5.311822710045263, -5.729236608787275, -5.72...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.744643672268496, -6.228460867087006, -5.3...","[-5.048289263821047, -5.927772534021331, -5.46...","[0.3631172144859594, 0.587338597669397, 0.1459..."
6,AdaBoostRegressor,"[-5.270657249625, -5.6650479899, -5.6650479899...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.61, -5.6650479899, -5.270657249625, -5.49...","[-4.9441, -5.697958532890256, -5.3480906933662...","[0.20945462515781316, 0.1912890063855193, 0.09..."
7,SVR,"[-5.231507171110114, -5.950056233474417, -5.95...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.391848157904755, -5.913971182082774, -5.2...","[-5.271556232183689, -5.77503463335029, -5.353...","[0.1642359916221465, 0.3069499581930055, 0.196..."
8,LinearRegression,"[-5.146044731085824, -5.729999999999999, -5.72...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.127439319444217, -5.458310087664266, -5.1...","[-4.829303007985404, -5.193654855434683, -5.29...","[0.32100949916125937, 0.26198808252195244, 0.1..."
9,KNeighborsRegressor,"[-5.183333333333334, -5.400000000000001, -5.40...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.319999999999999, -5.503333333333333, -5.1...","[-5.101333333333333, -5.472, -5.368, -5.605048...","[0.20126930549225178, 0.18465102220134053, 0.1..."


In [19]:
aac_comp.to_csv('results/Monomeric/AAC_comp_results_MDCK.csv')
prediction_df.to_csv('results/Monomeric/AAC_comp_prediction_data_MDCK.csv')

In [20]:
#Constant column removal
df_mc_train = pd.read_csv('features/Monomeric/Train_aac_MDCK.csv')
df_mc_train, const_col = remove_constant_columns(df_mc_train)
X_train = df_mc_train.drop(['ID','SMILES','Permeability'], axis=1)
y_train = df_mc_train['Permeability']
print(X_train.shape)
print(y_train.shape)
df_mc_test = pd.read_csv('features/Monomeric/Test_aac_MDCK.csv')
X_test = df_mc_test.drop(['ID','SMILES','Permeability'], axis=1)
X_test = X_test.drop(const_col, axis=1)
y_test = df_mc_test['Permeability']
print(X_test.shape)
print(y_test.shape)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models_mc = [
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    xgb.XGBRegressor(random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    SVR(),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3), 
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models_mc, X_train,y_train, X_test,  y_test)
result_df

(51, 11)
(51,)
(13, 11)
(13,)
0.2740853620143081
[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 0
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training b

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
ExtraTreesRegressor,0.6734,0.6526,0.8206,-0.2799,0.2218,0.2621,0.5191,0.5826,0.7205,0.2741,0.5672,0.3715
LGBMRegressor,0.4912,0.5710,0.7008,0.0665,0.2632,0.2534,0.6428,0.6748,0.8017,0.1011,0.4245,0.2624
XGBRegressor,0.6525,0.6562,0.8078,-0.2402,0.2333,0.2785,0.6064,0.6409,0.7787,0.1520,0.4245,0.3384
DecisionTreeRegressor,0.7285,0.7033,0.8535,-0.3845,0.1974,0.2498,0.6056,0.6332,0.7782,0.1532,0.4191,0.2307
RandomForestRegressor,0.6068,0.6347,0.7789,-0.1532,0.2368,0.2891,0.5327,0.6033,0.7299,0.2550,0.5450,0.3384
GradientBoostingRegressor,0.6764,0.6729,0.8224,-0.2854,0.2015,0.2498,0.5513,0.6038,0.7425,0.2291,0.5204,0.3715
AdaBoostRegressor,0.5921,0.6449,0.7695,-0.1253,0.2961,0.3077,0.5703,0.6129,0.7552,0.2026,0.4749,0.3439
SVR,0.5905,0.6390,0.7684,-0.1223,0.1089,0.0993,0.5163,0.5792,0.7185,0.2780,0.6294,0.3743
LinearRegression,0.5622,0.6091,0.7498,-0.0684,0.2674,0.2923,0.5957,0.6316,0.7718,0.1671,0.4744,0.3550
KNeighborsRegressor,0.6975,0.6959,0.8352,-0.3256,0.1993,0.2659,0.5634,0.5754,0.7506,0.2122,0.5638,0.4406


In [21]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,ExtraTreesRegressor,"[-5.2950000000000035, -5.7300000000000075, -5....",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.352300000000003, -6.25, -5.29500000000000...","[-5.201093477357206, -5.9707870956020015, -5.4...","[0.4117693369203956, 0.558425808795997, 0.1476..."
1,LGBMRegressor,"[-5.6122931599617, -5.6122931599617, -5.612293...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.6122931599617, -5.6122931599617, -5.61229...","[-5.390906233628531, -5.553378645833561, -5.39...","[0.17169618719084973, 0.06934332374565914, 0.1..."
2,XGBRegressor,"[-5.2957315, -5.7299514, -5.7299514, -5.678163...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.6846294, -6.2477527, -5.2957315, -5.29089...","[-4.873517, -5.9578676, -5.470054, -5.4094343,...","[0.4134306, 0.5801909, 0.14739683, 0.23752233,..."
3,DecisionTreeRegressor,"[-5.295, -5.73, -5.73, -5.9566379538, -5.95663...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.61, -6.25, -5.295, -5.289036881, -5.18575...","[-4.8384, -5.948, -5.470000000000001, -5.51122...","[0.37336609380070945, 0.604, 0.147614362444851..."
4,RandomForestRegressor,"[-5.356558154479663, -5.72216782828283, -5.722...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.207066507066176, -6.003736663387287, -5.3...","[-5.185623358950599, -5.774953204414381, -5.49...","[0.18892176185399787, 0.2803800897277758, 0.10..."
5,GradientBoostingRegressor,"[-5.311822710045263, -5.729236608787275, -5.72...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.557041164518372, -6.228460867087006, -5.3...","[-5.026258176939469, -5.926105236069808, -5.46...","[0.3839707541729165, 0.5906729970752512, 0.145..."
6,AdaBoostRegressor,"[-5.1384338401428575, -5.4871504808, -5.487150...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.94, -6.028296251714287, -5.13843384014285...","[-5.0020646110493505, -5.794346045019781, -5.2...","[0.14988254828377334, 0.30531950206547187, 0.1..."
7,SVR,"[-5.231503433603377, -5.9500565333142195, -5.9...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.3918497634159825, -5.9139745936124655, -5...","[-5.271571197550909, -5.775051014389474, -5.35...","[0.16422881001634865, 0.30694999599247785, 0.1..."
8,LinearRegression,"[-5.146044731085823, -5.73, -5.73, -5.61043025...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.127439319444216, -5.458310087664265, -5.1...","[-4.829303007985404, -5.1936548554346835, -5.2...","[0.3210094991612591, 0.2619880825219521, 0.112..."
9,KNeighborsRegressor,"[-5.183333333333334, -5.3999999999999995, -5.3...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.319999999999999, -5.503333333333333, -5.1...","[-5.101333333333334, -5.472, -5.368, -5.605048...","[0.20126930549225144, 0.18465102220134053, 0.1..."


In [22]:
result_df.to_csv('results/Monomeric/AAC_comp_const_rem_results_MDCK.csv')
prediction_df.to_csv('results/Monomeric/AAC_comp_const_rem_prediction_data_MDCK.csv')

In [23]:
#LVR column removal
df_mc_train = pd.read_csv('features/Monomeric/Train_aac_MDCK.csv')
X_train = df_mc_train.drop(['ID','SMILES','Permeability'], axis=1)
X_train, const_col = remove_low_variance_columns(X_train)

y_train = df_mc_train['Permeability']
print(X_train.shape)
print(y_train.shape)
df_mc_test = pd.read_csv('features/Monomeric/Test_aac_MDCK.csv')
X_test = df_mc_test.drop(['ID','SMILES','Permeability'], axis=1)
X_test = X_test.drop(const_col, axis=1)
y_test = df_mc_test['Permeability']
print(X_test.shape)
print(y_test.shape)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models_mc = [
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    xgb.XGBRegressor(random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    SVR(),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3), 
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models_mc, X_train,y_train, X_test,  y_test)
result_df

(51, 5)
(51,)
(13, 5)
(13,)
0.18688709900245326
[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 0
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training be

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
ExtraTreesRegressor,0.5984,0.6112,0.7735,-0.1372,0.2734,0.3020,0.5815,0.6387,0.7625,0.1869,0.4411,0.2279
LGBMRegressor,0.5075,0.5763,0.7124,0.0355,0.2117,0.1648,0.6806,0.7084,0.8250,0.0483,0.2399,0.1836
XGBRegressor,0.5845,0.6224,0.7645,-0.1109,0.2707,0.3028,0.6797,0.6943,0.8244,0.0496,0.3097,0.1423
DecisionTreeRegressor,0.6342,0.6518,0.7963,-0.2052,0.2120,0.2582,0.6903,0.6955,0.8308,0.0347,0.3048,0.0870
RandomForestRegressor,0.5460,0.6092,0.7389,-0.0377,0.2753,0.3066,0.5900,0.6421,0.7681,0.1750,0.4233,0.2445
GradientBoostingRegressor,0.5933,0.6276,0.7702,-0.1275,0.2544,0.2894,0.6306,0.6551,0.7941,0.1182,0.3778,0.1920
AdaBoostRegressor,0.5748,0.6259,0.7582,-0.0925,0.3113,0.3739,0.7246,0.7114,0.8512,-0.0132,0.2638,0.2030
SVR,0.5188,0.5555,0.7203,0.0140,0.2698,0.2971,0.5419,0.6206,0.7361,0.2423,0.5205,0.4185
LinearRegression,0.4891,0.5587,0.6993,0.0705,0.3212,0.2852,0.6698,0.6986,0.8184,0.0634,0.2916,0.2887
KNeighborsRegressor,0.6484,0.6508,0.8052,-0.2323,0.2620,0.3260,0.8058,0.7193,0.8977,-0.1268,0.2020,0.3025


In [24]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,ExtraTreesRegressor,"[-5.2950000000000035, -5.7300000000000075, -5....",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.435800013528336, -6.25, -5.29500000000000...","[-5.307213392579671, -5.986756268464001, -5.47...","[0.4225334789764783, 0.5264874630719969, 0.147..."
1,LGBMRegressor,"[-5.6122931599617, -5.6122931599617, -5.612293...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.6122931599617, -5.6122931599617, -5.61229...","[-5.432103562174467, -5.622943169621069, -5.43...","[0.1772622186145124, 0.10310894525966106, 0.17..."
2,XGBRegressor,"[-5.295006, -5.730156, -5.730156, -5.956259, -...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.685129, -6.248374, -5.295006, -5.2913527,...","[-4.9287972, -5.980279, -5.469829, -5.4437633,...","[0.4494258, 0.5355617, 0.14760725, 0.30150056,..."
3,DecisionTreeRegressor,"[-5.295, -5.7299999999999995, -5.7299999999999...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.61, -6.25, -5.295, -5.289036881, -5.15738...","[-4.854, -5.948, -5.470000000000001, -5.269229...","[0.3183457240171444, 0.604, 0.1476143624448516..."
4,RandomForestRegressor,"[-5.332146666666667, -5.722167828282828, -5.72...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.2681294312027, -5.958356049659788, -5.332...","[-5.226201787384748, -5.766624213829944, -5.47...","[0.16017345516292458, 0.26653928568730123, 0.1..."
5,GradientBoostingRegressor,"[-5.2745660148083475, -5.714824273906094, -5.7...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.626194579034391, -6.248032044852147, -5.2...","[-4.976802856365359, -5.926810320824062, -5.45...","[0.4208316617983213, 0.6332439079860911, 0.145..."
6,AdaBoostRegressor,"[-5.455575240399999, -5.668277246846154, -5.66...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.61, -6.206666666666667, -5.45557524039999...","[-4.7956, -5.776491882859276, -5.4323286467199...","[0.22522308940248573, 0.33306963784734706, 0.0..."
7,SVR,"[-5.040357053606516, -5.420307061164386, -5.42...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.535014738132937, -6.150042167454407, -5.0...","[-5.441846676838745, -5.989745028982486, -5.35...","[0.17614474907580635, 0.320447045032469, 0.227..."
8,LinearRegression,"[-5.144945432761743, -5.436367794942288, -5.43...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.148091839746878, -5.380339808136233, -5.1...","[-5.009940209024267, -5.270530973028047, -5.16...","[0.22748693023651623, 0.17768835576341685, 0.0..."
9,KNeighborsRegressor,"[-5.136666666666667, -5.400000000000001, -5.40...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.319999999999999, -5.5857249976666665, -5....","[-5.0920000000000005, -5.507208392133333, -5.3...","[0.19610201426808416, 0.19294486084682386, 0.1..."


In [26]:
result_df.to_csv('results/Monomeric/AAC_comp_LVR_results_MDCK.csv')
prediction_df.to_csv('results/Monomeric/AAC_comp_LVR_prediction_data_MDCK.csv')

In [27]:
#Atomic models
df_train = pd.read_csv('features/Atomic/Train_all_atomic_desc_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Atomic/Test_all_atomic_desc_MDCK.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models_degree = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models_degree, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 23)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 23)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.073120 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 1
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with 

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.4215,0.5058,0.6492,0.1989,0.4470,0.4767,0.6382,0.6856,0.7989,0.1076,0.3320,0.3961
DecisionTreeRegressor,0.5435,0.5530,0.7372,-0.0329,0.5085,0.5146,0.8266,0.6751,0.9092,-0.1558,0.2872,0.1492
RandomForestRegressor,0.4199,0.5115,0.6480,0.2020,0.5242,0.5191,0.7331,0.6830,0.8562,-0.0251,0.3239,0.1103
GradientBoostingRegressor,0.4850,0.5309,0.6964,0.0782,0.5269,0.5261,0.7959,0.6835,0.8921,-0.1129,0.3047,0.1517
AdaBoostRegressor,0.3889,0.5024,0.6236,0.2609,0.5805,0.5440,0.7587,0.6936,0.8710,-0.0609,0.2984,0.0828
XGBRegressor,0.5307,0.5531,0.7285,-0.0085,0.5045,0.5225,0.7957,0.6728,0.8920,-0.1126,0.3097,0.2234
ExtraTreesRegressor,0.4702,0.5109,0.6857,0.1064,0.5509,0.5535,0.8214,0.6926,0.9063,-0.1486,0.2784,0.1324
LinearRegression,0.6905,0.6423,0.8310,-0.3124,0.3412,0.4325,0.5973,0.6482,0.7728,0.1648,0.4163,0.3586
KNeighborsRegressor,0.4816,0.5260,0.6940,0.0847,0.4296,0.4894,0.6516,0.6574,0.8072,0.0889,0.3949,0.2928
SVR,0.4076,0.5048,0.6384,0.2253,0.4775,0.4462,0.4992,0.5833,0.7065,0.3019,0.6072,0.3779


In [28]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.599003550019117, -5.599003550019117, -5.59...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.599003550019117, -5.625582771608608, -5.6...","[-5.3546850035216, -5.272146292871387, -5.2721...","[0.13016524490335224, 0.19627843859881136, 0.1..."
1,DecisionTreeRegressor,"[-5.295, -4.95, -6.3, -7.698970004, -7.6989700...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.295, -6.25, -6.25, -5.289036881, -5.18575...","[-5.324000000000001, -6.094, -6.094, -5.615433...","[0.32719107567291644, 0.3119999999999994, 0.31..."
2,RandomForestRegressor,"[-5.37618, -5.023954310426663, -5.818648879526...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.357238333333332, -5.916911666666665, -5.8...","[-5.388192222112856, -5.847569150890189, -5.82...","[0.14419260784225363, 0.17579064844244066, 0.1..."
3,GradientBoostingRegressor,"[-5.292630685488414, -4.9653956079948145, -6.2...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.2799711853456275, -6.2254194320847, -6.20...","[-5.385605886776788, -6.076183884332962, -6.02...","[0.204955266914944, 0.32035227977870345, 0.312..."
4,AdaBoostRegressor,"[-5.295, -4.885, -5.986009220249999, -7.265980...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.295, -6.04368187375, -6.04368187375, -5.7...","[-5.3926504807999995, -5.83381161515, -5.86928...","[0.17469299083215853, 0.3089386014071394, 0.31..."
5,XGBRegressor,"[-5.295256, -4.95033, -6.298491, -7.695771, -7...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.294503, -6.24896, -6.2480273, -5.2900734,...","[-5.371639, -6.0928373, -6.061701, -5.557718, ...","[0.24268094, 0.31156412, 0.3542259, 0.5345353,..."
6,ExtraTreesRegressor,"[-5.2950000000000035, -4.949999999999992, -6.2...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.156297274950005, -6.25, -6.05235000000000...","[-5.279319454990007, -6.078760000000003, -5.88...","[0.13039990680767105, 0.34247999999999557, 0.3..."
7,LinearRegression,"[-5.031351321950627, -5.671902481253781, -5.74...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.98082901599769, -5.856993555004487, -5.57...","[-5.218264844120585, -5.510030275564914, -5.62...","[0.143344510275309, 0.2156458891072152, 0.1086..."
8,KNeighborsRegressor,"[-5.613333333333333, -5.3999999999999995, -5.3...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333333, -5.6...","[-5.6259999999999994, -5.666, -5.666, -5.94014...","[0.16096100286853474, 0.14081350945290877, 0.1..."
9,SVR,"[-5.4524366480689475, -5.8060725999753435, -5....",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.443619900238809, -5.50780915536032, -5.52...","[-5.580832354960098, -5.625786302409056, -5.63...","[0.10461814234350492, 0.1122050123653244, 0.11..."


In [29]:
result_df.to_csv('results/Atomic/Results_all_atomic_desc_MDCK.csv')
prediction_df.to_csv('results/Atomic/Prediction_data_all_atomic_desc_MDCK.csv')

In [30]:
#Atomic + monomeric_composition based features
df1 = pd.read_csv('features/Monomeric/Train_mon_comp_MDCK.csv')
df2 = pd.read_csv('features/Atomic/Train_all_atomic_desc_MDCK.csv')
df_train = pd.merge(df1, df2, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_train

,ID,SMILES,Permeability,A,dA,meA,Me_dA,Ala(tBu),Ala(indol-2-yl),dAla(indol-2-yl),...,Degree_Br,Single,Double,Triple,Aromatic,Conjugated,No-bond,Overall_Formal_Charge,Is_Aromatic,Is_In_Ring
0,1114,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-4.940000,0.0,0.0,0.200000,0.000000,0.0,0.0,0.0,...,0,64,10,0,12,0,0,102,1,1
1,1113,CC(C)C[C@@H]1NC(=O)[C@@H]2CCCN2C(=O)[C@@H](Cc2...,-5.820000,0.0,0.0,0.200000,0.000000,0.0,0.0,0.0,...,0,64,10,0,12,0,0,102,1,1
2,1117,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H]2CCCN...,-5.650000,0.0,0.0,0.200000,0.000000,0.0,0.0,0.0,...,0,64,10,0,12,0,0,102,1,1
3,1119,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-6.250000,0.0,0.0,0.200000,0.200000,0.0,0.0,0.0,...,0,60,10,0,12,0,0,102,1,1
4,2428,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N(C)[C@@H](C)C(=...,-5.510000,0.0,0.0,0.200000,0.000000,0.0,0.0,0.0,...,0,66,10,0,6,0,0,98,1,1
5,2446,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N2CCC[C@@H]2C(=O...,-6.400000,0.2,0.0,0.000000,0.000000,0.0,0.0,0.0,...,0,66,10,0,6,0,0,98,1,1
6,2445,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N2CCC[C@@H]2C(=O...,-4.820000,0.2,0.0,0.000000,0.000000,0.0,0.0,0.0,...,0,66,10,0,6,0,0,94,1,1
7,2427,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N(C)[C@@H](C)C(=...,-4.610000,0.0,0.0,0.200000,0.000000,0.0,0.0,0.0,...,0,66,10,0,6,0,0,94,1,1
8,8145,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.744727,0.0,0.0,0.222222,0.000000,0.0,0.0,0.0,...,0,65,10,0,0,0,0,95,0,1
9,1107,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](C)N...,-4.610000,0.0,0.0,0.200000,0.200000,0.0,0.0,0.0,...,0,64,10,0,0,0,0,90,0,1


In [31]:
df1 = pd.read_csv('features/Monomeric/Test_mon_comp_MDCK.csv')
df2 = pd.read_csv('features/Atomic/Test_all_atomic_desc_MDCK.csv')
df_test = pd.merge(df1, df2, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_test

,ID,SMILES,Permeability,A,dA,meA,Me_dA,Ala(tBu),Ala(indol-2-yl),dAla(indol-2-yl),...,Degree_Br,Single,Double,Triple,Aromatic,Conjugated,No-bond,Overall_Formal_Charge,Is_Aromatic,Is_In_Ring
0,1120,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-6.300000,0.0,0.0,0.200000,0.000000,0.0,0.0,0.0,...,0,66,10,0,12,0,0,102,1,1
1,1118,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](Cc2...,-5.350000,0.0,0.0,0.400000,0.000000,0.0,0.0,0.0,...,0,60,10,0,12,0,0,102,1,1
2,1121,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-6.200000,0.2,0.0,0.000000,0.000000,0.0,0.0,0.0,...,0,60,10,0,12,0,0,102,1,1
3,8133,CCC[C@@H]1NC(=O)CN(CC)C(=O)[C@H](CC(C)C)NC(=O)...,-5.355561,0.0,0.0,0.222222,0.000000,0.0,0.0,0.0,...,0,64,10,0,0,0,0,94,0,1
4,8143,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.406714,0.0,0.0,0.222222,0.000000,0.0,0.0,0.0,...,0,62,10,0,0,0,0,95,0,1
5,8119,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.073658,0.0,0.0,0.222222,0.000000,0.0,0.0,0.0,...,0,61,10,0,0,0,0,94,0,1
6,6496,CC(=O)N1CCC[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N[...,-4.860000,0.0,0.0,0.000000,0.100000,0.0,0.0,0.0,...,0,63,10,0,0,0,0,92,0,1
7,8168,CCOC(=O)[C@@H]1CSCC(=O)N[C@@H](CC)C(=O)N(C)[C@...,-6.031517,0.0,0.0,0.111111,0.222222,0.0,0.0,0.0,...,0,59,10,0,0,0,0,94,0,1
8,8345,CCOC(=O)[C@@H]1CSCC(=O)N[C@@H](CC)C(=O)N(CC(C)...,-7.534591,0.0,0.0,0.111111,0.222222,0.0,0.0,0.0,...,0,59,10,0,0,0,0,94,0,1
9,6423,CC(=O)N1CCC[C@@H]1C(=O)N(C)[C@@H](CC(C)C)C(=O)...,-6.300000,0.1,0.0,0.000000,0.000000,0.0,0.0,0.0,...,0,62,10,0,0,0,0,92,0,1


In [32]:
import re
def clean_feature_names(df):
    def clean_name(name):
        return re.sub(r'[^a-zA-Z0-9_]', '_', name)

    df.columns = [clean_name(col) for col in df.columns]
    return df

In [33]:
#Removal of constant columns
def remove_constant_columns(df):
    constant_columns = [col for col in df.columns if df[col].nunique() <= 1]
    
    df_cleaned = df.drop(columns=constant_columns)
    
    return df_cleaned, constant_columns

In [34]:
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = clean_feature_names(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = clean_feature_names(X_test)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 408)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 408)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.068940 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 16
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 2
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wit

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

-0.5403237938407552


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.4048,0.4939,0.6363,0.2306,0.4870,0.5121,0.6269,0.6731,0.7917,0.1234,0.3562,0.3393
DecisionTreeRegressor,0.5336,0.5191,0.7305,-0.0142,0.5337,0.4506,0.5689,0.5343,0.7543,0.2044,0.4971,0.2841
RandomForestRegressor,0.2960,0.4451,0.5440,0.4375,0.6624,0.5219,0.5240,0.5230,0.7239,0.2673,0.5272,0.3117
GradientBoostingRegressor,0.2978,0.4415,0.5457,0.4340,0.6866,0.5327,0.5295,0.5113,0.7277,0.2595,0.5255,0.3448
AdaBoostRegressor,0.3222,0.4513,0.5676,0.3877,0.6453,0.4600,0.5578,0.5583,0.7469,0.2200,0.4781,0.2731
XGBRegressor,0.3050,0.4367,0.5523,0.4202,0.6820,0.5258,0.5095,0.4831,0.7138,0.2876,0.5606,0.4083
ExtraTreesRegressor,0.3398,0.4736,0.5829,0.3542,0.6358,0.4786,0.5775,0.5436,0.7599,0.1925,0.4649,0.2097
LinearRegression,4.7132,1.4008,2.1710,-7.9577,0.0677,0.1719,2.8447,1.3274,1.6866,-2.9778,0.1105,0.0386
KNeighborsRegressor,0.4876,0.5663,0.6983,0.0734,0.4132,0.4104,0.4085,0.4814,0.6391,0.4288,0.7547,0.6547
SVR,0.3714,0.4939,0.6095,0.2941,0.5490,0.5046,0.4764,0.5186,0.6902,0.3339,0.7293,0.7117


In [35]:
result_df.to_csv('results/Atomic/Results_all_atomic_desc_and_mono_comp_MDCK.csv')
prediction_df.to_csv('results/Atomic/Prediction_data_all_atomic_desc_and_mono_comp_MDCK.csv')

In [36]:
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = clean_feature_names(X_train)
X_train, const_col = remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = clean_feature_names(X_test)
X_test = X_test.drop(const_col,axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 44)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 44)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.101692 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 16
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 2
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with 

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.4048,0.4939,0.6363,0.2306,0.4870,0.5121,0.6269,0.6731,0.7917,0.1234,0.3562,0.3393
DecisionTreeRegressor,0.3700,0.4604,0.6083,0.2968,0.6304,0.4876,0.5516,0.5045,0.7427,0.2286,0.5050,0.3283
RandomForestRegressor,0.3061,0.4532,0.5533,0.4182,0.6480,0.5164,0.5252,0.5240,0.7247,0.2656,0.5261,0.2345
GradientBoostingRegressor,0.3016,0.4413,0.5492,0.4268,0.6834,0.5411,0.5294,0.5093,0.7276,0.2597,0.5253,0.3669
AdaBoostRegressor,0.2848,0.4149,0.5337,0.4587,0.6861,0.5141,0.5223,0.5314,0.7227,0.2696,0.5283,0.3062
XGBRegressor,0.3050,0.4367,0.5523,0.4202,0.6820,0.5258,0.5095,0.4831,0.7138,0.2876,0.5606,0.4083
ExtraTreesRegressor,0.3321,0.4718,0.5763,0.3688,0.6436,0.4828,0.5728,0.5392,0.7568,0.1990,0.4728,0.2869
LinearRegression,4.7132,1.4008,2.1710,-7.9577,0.0677,0.1719,2.8447,1.3274,1.6866,-2.9778,0.1105,0.0386
KNeighborsRegressor,0.4913,0.5679,0.7009,0.0663,0.4093,0.4075,0.4019,0.4779,0.6339,0.4380,0.7598,0.6547
SVR,0.3714,0.4939,0.6095,0.2941,0.5490,0.5046,0.4764,0.5186,0.6902,0.3339,0.7293,0.7117


In [37]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.543514350566838, -5.658775107068941, -5.65...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.543514350566838, -5.5658112122703445, -5....","[-5.247070211565934, -5.205186404421357, -5.29...","[0.17440105884100882, 0.19733107184279022, 0.2..."
1,DecisionTreeRegressor,"[-5.295, -5.32, -6.05, -5.493665733, -7.698970...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.295, -6.25, -6.4, -5.289036881, -5.185752...","[-5.404000000000001, -5.853, -6.10399999999999...","[0.3014863180975218, 0.36575401569907606, 0.31..."
2,RandomForestRegressor,"[-5.206427524040002, -5.282499999999999, -5.71...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.407341465656666, -5.446153161143335, -5.5...","[-5.307218762001332, -5.561531780196001, -5.59...","[0.16456333952304583, 0.09668701768568387, 0.1..."
3,GradientBoostingRegressor,"[-5.198891850277154, -5.016556433686429, -6.21...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.2519193262266874, -5.675807676774869, -5....","[-5.442852171279833, -5.789248379071239, -5.81...","[0.3250591094156472, 0.12023658154352206, 0.15..."
4,AdaBoostRegressor,"[-5.209292067333334, -5.0810747886000005, -6.1...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.209292067333334, -5.63676768, -5.71492906...","[-5.307097009447059, -5.612844777813334, -5.70...","[0.1920477216428831, 0.1487458238204066, 0.175..."
5,XGBRegressor,"[-5.3992496, -5.5091763, -6.0130806, -5.471843...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.4004397, -5.6284685, -6.3387055, -5.28953...","[-5.285525, -5.693409, -5.976907, -5.3233843, ...","[0.29472634, 0.07591528, 0.30510923, 0.0675504..."
6,ExtraTreesRegressor,"[-4.845850000000008, -5.575499999999998, -5.72...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.396600000000003, -5.052107524040007, -4.9...","[-5.50390727495, -5.655828770926002, -5.297791...","[0.19947572801859462, 0.3189099157395863, 0.28..."
7,LinearRegression,"[-10.0, -5.287400933974368, -6.725738889269451...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.0, -10.0, -10.0, -5.289036880999106, -10....","[-5.2, -7.776654018468271, -10.0, -5.031229504...","[2.4, 1.227977652300011, 0.0, 0.51561475239992..."
8,KNeighborsRegressor,"[-5.613333333333333, -5.3999999999999995, -5.4...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333333, -5.3...","[-5.4126666666666665, -5.638, -5.4773333333333...","[0.21437661564017046, 0.16956938927111156, 0.1..."
9,SVR,"[-5.396448634901098, -5.6065374599802995, -5.8...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.519246413068403, -5.417026916602952, -5.3...","[-5.42954581494422, -5.604726122733707, -5.319...","[0.18552717049498602, 0.09533761586374052, 0.0..."


In [38]:
result_df.to_csv('results/Atomic/Results_all_atomic_desc_and_mono_comp_const_rem_MDCK.csv')
prediction_df.to_csv('results/Atomic/Prediction_data_all_atomic_desc_and_mono_comp_const_rem_MDCK.csv')

In [39]:
const_col

['Ala_tBu_',
 'Ala_indol_2_yl_',
 'dAla_indol_2_yl_',
 'Me_Ala_indol_2_yl_',
 'Ala_5_Tet_',
 'Me_dAbu',
 'Me_Abu_morpholino_',
 '2Abz',
 'Aib',
 'Aoc_2_',
 '5_Ava',
 'Bal',
 'Me_Bal',
 'HOCOCH2_Bal',
 'Cys_EtO2H__NH2',
 'dCha',
 'Me_Cha',
 'D',
 'meD',
 'Asp_piperidide',
 'Asp_OMe_',
 'Asp_Ph_2_NH2__',
 'dAsp_pyrrol_1_yl_',
 'E',
 'Glu_NH2',
 'Glu_3R_Me_',
 'Glu_OMe_',
 'dGlu_OMe_',
 'Phe_4_F_',
 'dPhe_4_F_',
 'Phe_4_CF3_',
 'Phe_4_NO2_',
 'Phe_CHF2_',
 'dPhe_3_4_diF_',
 'Et_Phe',
 'H2NEt_Phe',
 'Me_Phe_3_Cl_',
 'Me_Phe_4_Cl_',
 'Me_Phe_a_b_dehydro_',
 'G',
 'Bn_Gly',
 'Bn_4_Cl__Gly',
 'Bn_4_OH__Gly',
 'Bu_Gly',
 'EtOEt_Gly',
 'HOCOCH2_Gly_ol',
 'MeOEt_Gly',
 'NH2Bu_Gly',
 'PhEt_Gly',
 'PhPr_Gly',
 'isoamyl_Gly',
 'pentyl_Gly',
 '3_pyridylethyl_Gly',
 '2_pyridylmethyl_Gly',
 'd_N__O_Gly_allyl_',
 'GABA',
 'H',
 'Hph',
 'Me_Hph',
 'bHph',
 'Hph_2_Cl_',
 'Hph_3_Cl_',
 'Hph_4_Cl_',
 'Hse_Et_',
 'dHyp',
 'Hyp_Et_',
 'dI',
 'meI',
 'Me_dI',
 '_N__O_xiIle',
 'd_N__O_aIle',
 'K',
 'dK',
 'meK

In [40]:
#Fingerprints models
#All fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/All_fingerprints_train_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/All_fingerprints_test_MDCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 20188)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 20188)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.124545 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 665
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 177
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spl

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.5170,0.6134,0.7190,0.0175,0.2205,0.1887,0.5864,0.6610,0.7658,0.1800,0.4758,0.3641
DecisionTreeRegressor,0.7036,0.6307,0.8388,-0.3372,0.2221,0.2159,0.4235,0.5382,0.6508,0.4077,0.6496,0.5813
RandomForestRegressor,0.4031,0.5142,0.6349,0.2339,0.4884,0.4637,0.4045,0.5260,0.6360,0.4344,0.7136,0.5096
GradientBoostingRegressor,0.4570,0.5323,0.6760,0.1315,0.5039,0.4655,0.4098,0.5053,0.6402,0.4270,0.6625,0.5703
AdaBoostRegressor,0.4212,0.5061,0.6490,0.1996,0.4746,0.4370,0.4530,0.5446,0.6731,0.3665,0.6249,0.4711
XGBRegressor,0.3916,0.4991,0.6257,0.2558,0.5468,0.4643,0.3430,0.4530,0.5856,0.5204,0.7528,0.6171
ExtraTreesRegressor,0.6159,0.5893,0.7848,-0.1706,0.2962,0.2395,0.3779,0.4761,0.6147,0.4716,0.7024,0.6198
LinearRegression,0.6826,0.6340,0.8262,-0.2973,0.5295,0.4424,0.4087,0.5056,0.6393,0.4285,0.6620,0.5537
KNeighborsRegressor,0.5056,0.5726,0.7111,0.0390,0.3916,0.4870,0.4131,0.5262,0.6427,0.4224,0.7334,0.6143
SVR,0.4505,0.5331,0.6712,0.1438,0.3934,0.4438,0.4106,0.5012,0.6408,0.4259,0.7778,0.6804


In [41]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.518164737322221, -5.833013972428597, -5.69...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.518164737322221, -5.750468772944585, -5.5...","[-5.449001315406229, -5.459458807161469, -5.42...","[0.08364446080528505, 0.19903811079871922, 0.0..."
1,DecisionTreeRegressor,"[-4.94, -5.32, -5.984295768000001, -5.58502665...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.61, -6.25, -6.4, -5.744727495, -5.1857524...","[-5.266, -5.9239999999999995, -5.6560000000000...","[0.8034326356328824, 0.42178667593939007, 0.78..."
2,RandomForestRegressor,"[-5.219724999999996, -5.4656, -5.6510213968016...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.20781470004, -5.681189800610001, -5.73298...","[-5.255587431222834, -5.603898009939711, -5.70...","[0.15462529202970515, 0.19142051998968446, 0.0..."
3,GradientBoostingRegressor,"[-4.963575395378998, -5.339502265355207, -5.99...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.834556673535456, -6.1319879400628805, -6....","[-5.104139295277177, -5.905428634843926, -5.96...","[0.30725658587776045, 0.44832566532009693, 0.2..."
4,AdaBoostRegressor,"[-5.16, -5.16, -6.0586372639629635, -5.4166173...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.965, -6.0586372639629635, -5.336, -5.4166...","[-5.049402507145455, -5.717198263705557, -5.57...","[0.2054884154019605, 0.2501086250450278, 0.302..."
5,XGBRegressor,"[-5.3041806, -5.7082834, -5.6653643, -5.46308,...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.155613, -5.363766, -5.95833, -5.4886036, ...","[-5.3544903, -5.770476, -6.020243, -5.400271, ...","[0.31823218, 0.4511988, 0.26020205, 0.11068646..."
6,ExtraTreesRegressor,"[-4.9548, -5.5840000000000005, -5.882047274950...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.9165000000000045, -6.075799999999999, -5....","[-5.134939527180005, -5.666541031988002, -6.10...","[0.5463516004040704, 0.5849407794638487, 0.119..."
7,LinearRegression,"[-4.0, -5.582106237378324, -5.137509876741222,...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.785734020924389, -5.664405762806289, -4.8...","[-5.054401139958552, -5.6885827645550435, -6.1...","[1.1646059133392646, 0.3890847561769706, 1.032..."
8,KNeighborsRegressor,"[-5.613333333333333, -5.400000000000001, -5.31...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333334, -5.1...","[-5.6546666666666665, -5.666000000000001, -5.3...","[0.14230796026770795, 0.14081350945290821, 0.1..."
9,SVR,"[-5.2089341829391715, -5.654262786489938, -5.6...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.605516469005313, -5.629140596442484, -5.1...","[-5.6056763515772765, -5.6599641746333, -5.294...","[0.31937412152704514, 0.2810560522818997, 0.10..."


In [42]:
result_df.to_csv('results/Fingerprints/Results_All_fingerprints_fp_MDCK.csv')
prediction_df.to_csv('results/Fingerprints/Prediction_data_All_fingerprints_fp_MDCK.csv')

In [43]:
#Removal of constant columns
def remove_constant_columns(df):
    constant_columns = [col for col in df.columns if df[col].nunique() <= 1]
    
    df_cleaned = df.drop(columns=constant_columns)
    
    return df_cleaned, constant_columns

In [44]:
#Low variance column removal
def remove_low_variance_columns(df, threshold=0.005):
    df = df.drop(['ID','SMILES','Permeability'],axis=1)
    variances = df.var()
    
    low_variance_columns = variances[variances < threshold].index.tolist()
    
    df_cleaned = df.drop(columns=low_variance_columns)
    
    return df_cleaned, low_variance_columns

In [45]:
#All fingerprints constant removal
df_train = pd.read_csv('features/Fingerprints/Train/All_fingerprints_train_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train, const_col = remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/All_fingerprints_test_MDCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 2131)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 2131)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.108350 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 665
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 177
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further split

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.5170,0.6134,0.7190,0.0175,0.2205,0.1887,0.5864,0.6610,0.7658,0.1800,0.4758,0.3641
DecisionTreeRegressor,0.7072,0.6259,0.8410,-0.3441,0.2057,0.2053,0.4604,0.5488,0.6785,0.3563,0.5991,0.5427
RandomForestRegressor,0.3999,0.5129,0.6324,0.2400,0.4941,0.4778,0.4045,0.5247,0.6360,0.4344,0.7109,0.5152
GradientBoostingRegressor,0.4646,0.5375,0.6816,0.1171,0.4968,0.4500,0.4188,0.5141,0.6471,0.4144,0.6513,0.5703
AdaBoostRegressor,0.4208,0.5067,0.6487,0.2002,0.4731,0.4428,0.4450,0.5366,0.6671,0.3778,0.6315,0.5317
XGBRegressor,0.3916,0.4991,0.6257,0.2558,0.5468,0.4643,0.3430,0.4530,0.5856,0.5204,0.7528,0.6171
ExtraTreesRegressor,0.6153,0.5864,0.7844,-0.1695,0.2973,0.2449,0.3830,0.4778,0.6189,0.4644,0.6958,0.6198
LinearRegression,0.6826,0.6340,0.8262,-0.2973,0.5295,0.4424,0.4087,0.5056,0.6393,0.4285,0.6620,0.5537
KNeighborsRegressor,0.5056,0.5726,0.7111,0.0390,0.3916,0.4870,0.4131,0.5262,0.6427,0.4224,0.7334,0.6143
SVR,0.4505,0.5331,0.6712,0.1438,0.3935,0.4438,0.4106,0.5012,0.6408,0.4259,0.7778,0.6804


In [46]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.518164737322221, -5.833013972428597, -5.69...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.518164737322221, -5.750468772944585, -5.5...","[-5.449001315406229, -5.459458807161469, -5.42...","[0.08364446080528505, 0.19903811079871922, 0.0..."
1,DecisionTreeRegressor,"[-4.94, -5.32, -6.05, -5.585026652, -5.9842957...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.61, -6.25, -6.05, -5.752026734, -5.185752...","[-5.004, -5.918000000000001, -5.65600000000000...","[0.6359748422697236, 0.43185182644050474, 0.78..."
2,RandomForestRegressor,"[-5.244499999999997, -5.476200000000002, -5.60...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.25308970004, -5.648218653940002, -5.64966...","[-5.256702751145902, -5.611773613539102, -5.69...","[0.1493559576969119, 0.15652250053477793, 0.11..."
3,GradientBoostingRegressor,"[-4.963575395378998, -5.305155750841516, -6.03...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.833828265388861, -6.0654394666719, -6.001...","[-5.08600848109967, -5.900271467186961, -5.913...","[0.3019558130152305, 0.4065891769535996, 0.275..."
4,AdaBoostRegressor,"[-5.141585464777778, -5.32, -5.481485273500001...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.94, -6.05539705082353, -5.365694483388889...","[-5.033816057716395, -5.747073786782484, -5.71...","[0.21537639110946147, 0.25039312423974824, 0.1..."
5,XGBRegressor,"[-5.3041806, -5.7082834, -5.6653643, -5.46308,...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.155613, -5.363766, -5.95833, -5.4886036, ...","[-5.3544903, -5.770476, -6.020243, -5.400271, ...","[0.31823218, 0.4511988, 0.26020205, 0.11068646..."
6,ExtraTreesRegressor,"[-4.9451, -5.504099999999999, -5.9046543578550...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.891800000000004, -6.029996499132499, -5.8...","[-5.111899454990005, -5.660799543669835, -6.09...","[0.5374100573649526, 0.5744919750226793, 0.153..."
7,LinearRegression,"[-4.0, -5.582106237378432, -5.137509876741019,...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.785734020924667, -5.66440576280653, -4.86...","[-5.054401139958619, -5.688582764555088, -6.11...","[1.164605913339242, 0.38908475617696986, 1.032..."
8,KNeighborsRegressor,"[-5.613333333333333, -5.400000000000001, -5.31...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333334, -5.1...","[-5.6546666666666665, -5.666000000000001, -5.3...","[0.14230796026770795, 0.14081350945290821, 0.1..."
9,SVR,"[-5.209049765016605, -5.6542898787529525, -5.6...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.6057890216281825, -5.629354729475095, -5....","[-5.60579829303681, -5.660069568363115, -5.294...","[0.3192803526390272, 0.28096099299411037, 0.10..."


In [47]:
result_df.to_csv('results/Fingerprints/Results_All_const_rem_fingerprints_MDCK.csv')
prediction_df.to_csv('results/Fingerprints/Prediction_data_All_const_rem_fingerprints_MDCK.csv')

In [48]:
#Morgan fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/morgan_fp_train_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/morgan_fp_test_MDCK.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_morgan_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_morgan_fp

X_train shape:  (51, 2048)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 2048)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.130096 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 12
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 4
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits w

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.5373,0.6101,0.7330,-0.0212,0.1555,0.1191,0.6404,0.6745,0.8002,0.1045,0.3294,0.2800
DecisionTreeRegressor,0.4400,0.4692,0.6634,0.1637,0.5403,0.4971,0.5372,0.5903,0.7330,0.2488,0.5273,0.4517
RandomForestRegressor,0.3945,0.4999,0.6281,0.2502,0.5185,0.4753,0.3311,0.4691,0.5754,0.5371,0.8067,0.6869
GradientBoostingRegressor,0.3666,0.4722,0.6055,0.3033,0.5866,0.5343,0.2915,0.4225,0.5399,0.5924,0.8248,0.8469
AdaBoostRegressor,0.4277,0.4871,0.6540,0.1872,0.4898,0.4098,0.3545,0.4821,0.5954,0.5042,0.7909,0.6510
XGBRegressor,0.3841,0.4456,0.6198,0.2700,0.5747,0.5536,0.4375,0.5181,0.6615,0.3882,0.6244,0.5710
ExtraTreesRegressor,0.4566,0.4811,0.6757,0.1322,0.5086,0.4776,0.4080,0.5189,0.6388,0.4295,0.6679,0.5462
LinearRegression,0.3375,0.5049,0.5809,0.3586,0.7078,0.6140,0.4748,0.5652,0.6891,0.3361,0.5831,0.4855
KNeighborsRegressor,0.5508,0.6216,0.7422,-0.0468,0.3637,0.3890,0.5326,0.5579,0.7298,0.2552,0.5419,0.3641
SVR,0.4387,0.5401,0.6624,0.1662,0.4110,0.4569,0.4081,0.4983,0.6388,0.4293,0.7483,0.5959


In [49]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.411197694774015, -5.813388622975106, -5.81...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.411197694774015, -5.411197694774015, -5.4...","[-5.303177447057438, -5.303177447057438, -5.30...","[0.07550928741930664, 0.07550928741930664, 0.0..."
1,DecisionTreeRegressor,"[-6.25, -6.3, -4.95, -5.585026652, -7.69897000...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.61, -6.25, -6.4, -5.585026652, -5.22, -5....","[-5.18, -6.164, -6.33, -5.626028747000001, -5....","[0.7112242965478611, 0.17199999999999988, 0.14..."
2,RandomForestRegressor,"[-5.5914592005183374, -6.046033333333332, -5.1...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.581877335408337, -5.655264264183338, -5.8...","[-5.4087751288193076, -5.609435774238118, -5.8...","[0.21793494344207254, 0.11104260498705806, 0.0..."
3,GradientBoostingRegressor,"[-5.584477324532825, -6.217169064887989, -5.02...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.924826595999216, -5.822953385256624, -5.9...","[-5.6080431339564045, -5.771131168882211, -6.1...","[0.43272722478777503, 0.3325812730990856, 0.13..."
4,AdaBoostRegressor,"[-5.5332993193125, -6.234999999999999, -4.9757...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.376666666666668, -5.6736569932, -5.675502...","[-5.329285146606925, -5.594953960915328, -5.81...","[0.17320455180849045, 0.14297260777332269, 0.1..."
5,XGBRegressor,"[-6.1454077, -6.350171, -4.9486403, -5.4906363...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.84003, -6.240695, -6.1728196, -5.512159, ...","[-5.1077704, -6.061293, -6.1259904, -5.5106053...","[0.40684322, 0.28170016, 0.09697879, 0.1248964..."
6,ExtraTreesRegressor,"[-5.9419575240400055, -6.299999999999998, -4.9...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.542600000000002, -6.25, -6.39999999999998...","[-5.5709443327580015, -6.062373275144001, -6.3...","[0.24048307775915123, 0.20278894473440134, 0.1..."
7,LinearRegression,"[-5.507715559672882, -6.066768923407601, -4.43...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.051623078423915, -6.0227062062243535, -6....","[-5.548674950387568, -5.934427467328747, -5.77...","[0.5701947127140036, 0.7161254246221442, 0.370..."
8,KNeighborsRegressor,"[-5.183333333333334, -5.400000000000001, -5.31...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.503333333333333, -5.503333333333333, -4.8...","[-5.390666666666666, -5.483333333333333, -4.75...","[0.19764614845728687, 0.187936159373336, 0.181..."
9,SVR,"[-5.465162990922863, -5.962599824960984, -5.65...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.690303490274396, -5.629260333480628, -5.2...","[-5.536536155298061, -5.592760375306883, -5.23...","[0.3378645408735395, 0.22016279557490595, 0.04..."


In [50]:
df_morgan_fp.to_csv('results/Fingerprints/Results_Morgan_fp_MDCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_Morgan_fp_MDCK.csv')

In [51]:
#Morgan count fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/count_morgan_fp_train_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/count_morgan_fp_test_MDCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_morgan_count_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_morgan_count_fp

X_train shape:  (51, 2048)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 2048)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.126338 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 38
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 8
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits w

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.5780,0.6283,0.7603,-0.0985,0.0492,0.0023,0.5893,0.6456,0.7676,0.1760,0.4731,0.4550
DecisionTreeRegressor,0.6574,0.6098,0.8108,-0.2493,0.3133,0.2115,0.4149,0.5165,0.6441,0.4198,0.6534,0.5289
RandomForestRegressor,0.4052,0.5250,0.6366,0.2298,0.4929,0.4400,0.3563,0.4928,0.5969,0.5017,0.7731,0.6584
GradientBoostingRegressor,0.4224,0.5018,0.6499,0.1972,0.5232,0.4398,0.3547,0.4622,0.5956,0.5040,0.7363,0.6309
AdaBoostRegressor,0.4346,0.4905,0.6592,0.1741,0.4802,0.3926,0.3775,0.4938,0.6144,0.4721,0.7450,0.6336
XGBRegressor,0.4879,0.5242,0.6985,0.0728,0.4528,0.3564,0.3734,0.4707,0.6111,0.4778,0.7023,0.6143
ExtraTreesRegressor,0.4499,0.4909,0.6707,0.1450,0.4797,0.4405,0.3947,0.4962,0.6282,0.4481,0.6777,0.5923
LinearRegression,0.6361,0.6456,0.7976,-0.2089,0.5071,0.5586,0.6929,0.5999,0.8324,0.0311,0.4993,0.4545
KNeighborsRegressor,0.5311,0.6039,0.7288,-0.0095,0.3876,0.3656,0.5368,0.5742,0.7326,0.2494,0.5332,0.4601
SVR,0.4131,0.5254,0.6427,0.2149,0.4667,0.4966,0.4041,0.4987,0.6357,0.4349,0.7664,0.5895


In [52]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.530015958297521, -5.9575492946249895, -5.7...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.530015958297521, -5.530015958297521, -5.5...","[-5.408603310968301, -5.408603310968301, -5.34...","[0.10012589696761048, 0.10012589696761048, 0.1..."
1,DecisionTreeRegressor,"[-6.22, -6.3, -4.95, -5.289036881, -7.69897000...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.95, -6.25, -6.4, -5.289036881, -5.1857524...","[-5.328, -5.795999999999999, -6.308, -5.646357...","[0.7639476421849863, 0.6508947687606653, 0.112..."
2,RandomForestRegressor,"[-5.505938392523336, -5.805573333333334, -5.38...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.3401969234033375, -5.791117237885003, -5....","[-5.28961473111398, -5.6008828436612115, -5.82...","[0.17385278714303673, 0.2260887980387094, 0.05..."
3,GradientBoostingRegressor,"[-5.694995180648658, -6.072226785252764, -5.10...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.113393577072087, -6.1291095361338765, -5....","[-5.184047598812886, -5.7547053667339725, -6.0...","[0.5200794732690887, 0.4784479342322928, 0.270..."
4,AdaBoostRegressor,"[-5.676232798352941, -5.962663995222221, -5.2,...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.118047889260869, -5.959188769347826, -5.8...","[-5.241082806302063, -5.677086464804114, -5.86...","[0.18101756304968364, 0.2451163243217015, 0.09..."
5,XGBRegressor,"[-5.934829, -6.153833, -5.0689254, -5.5350575,...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.926636, -6.2475367, -6.1138887, -5.471974...","[-5.124515, -5.973868, -6.1555533, -5.510414, ...","[0.49387133, 0.4824748, 0.19628288, 0.1239549,..."
6,ExtraTreesRegressor,"[-5.804875284110007, -6.0493999999999994, -5.1...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.982795271800008, -6.14372549552, -5.94839...","[-5.2269781563966715, -5.858558176731334, -6.1...","[0.37187742512805566, 0.3653673298847714, 0.14..."
7,LinearRegression,"[-5.168206633491022, -5.59554708388105, -4.877...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.0, -4.561411339551818, -6.809027031557009...","[-4.0, -4.318683210821675, -6.64057158502622, ...","[0.0, 0.20493065940176342, 0.6096227200741149,..."
8,KNeighborsRegressor,"[-5.613333333333333, -5.3999999999999995, -5.3...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333333, -4.8...","[-5.4126666666666665, -5.666, -4.75, -5.532715...","[0.21437661564017046, 0.14081350945290877, 0.1..."
9,SVR,"[-5.482908578513038, -5.633732809718385, -5.73...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613517467228265, -5.644548215480978, -5.3...","[-5.483208350860414, -5.590678894686264, -5.39...","[0.24411648295851593, 0.15000519985700617, 0.0..."


In [53]:
df_morgan_count_fp.to_csv('results/Fingerprints/Results_Count_Morgan_fp_MDCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_Count_Morgan_fp_MDCK.csv')

In [54]:
#AtomPairs2d fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/AtomPairs2D_train_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/AtomPairs2D_test_MDCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_AtomPairs2D_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_AtomPairs2D_fp

X_train shape:  (51, 780)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 780)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.094214 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 3
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

0.09167243701481242


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.5383,0.6132,0.7337,-0.0230,0.0631,0.0921,0.6705,0.6871,0.8188,0.0624,0.2886,0.2688
DecisionTreeRegressor,0.4630,0.5582,0.6804,0.1201,0.3627,0.3979,0.6511,0.6834,0.8069,0.0896,0.3115,0.2815
RandomForestRegressor,0.4660,0.5620,0.6827,0.1143,0.3564,0.3728,0.6509,0.6842,0.8068,0.0898,0.3115,0.2815
GradientBoostingRegressor,0.4630,0.5582,0.6804,0.1201,0.3627,0.3979,0.6511,0.6834,0.8069,0.0896,0.3115,0.2815
AdaBoostRegressor,0.5398,0.6112,0.7347,-0.0260,0.2691,0.3280,0.6520,0.6864,0.8075,0.0882,0.3111,0.2815
XGBRegressor,0.4630,0.5583,0.6804,0.1201,0.3626,0.3979,0.6511,0.6834,0.8069,0.0896,0.3115,0.2815
ExtraTreesRegressor,0.4630,0.5582,0.6804,0.1201,0.3627,0.3979,0.6511,0.6834,0.8069,0.0896,0.3115,0.2815
LinearRegression,0.4649,0.5582,0.6818,0.1164,0.3609,0.3975,0.6511,0.6834,0.8069,0.0896,0.3115,0.2815
KNeighborsRegressor,0.6148,0.6355,0.7841,-0.1684,0.0280,0.0183,0.7692,0.7042,0.8770,-0.0756,-0.0541,-0.2246
SVR,0.5236,0.5951,0.7236,0.0049,0.2519,0.2123,0.6821,0.6807,0.8259,0.0462,0.3104,0.2815


In [55]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.411197694774015, -5.813388622975106, -5.81...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.411197694774015, -5.411197694774015, -5.4...","[-5.448700397817824, -5.448700397817824, -5.44...","[0.11989892378494135, 0.11989892378494135, 0.1..."
1,DecisionTreeRegressor,"[-5.210624999999999, -5.7299999999999995, -5.7...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.210624999999999, -5.210624999999999, -5.2...","[-5.204354131652662, -5.204354131652662, -5.20...","[0.02982355434907917, 0.02982355434907917, 0.0..."
2,RandomForestRegressor,"[-5.21114693327041, -5.7221678282828305, -5.72...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.21114693327041, -5.21114693327041, -5.211...","[-5.202839881974775, -5.202839881974775, -5.20...","[0.032308787966273775, 0.032308787966273775, 0..."
3,GradientBoostingRegressor,"[-5.210635668867855, -5.729996873541307, -5.72...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.210635668867855, -5.210635668867855, -5.2...","[-5.204363830411267, -5.204363830411267, -5.20...","[0.029823271689800892, 0.029823271689800892, 0..."
4,AdaBoostRegressor,"[-5.2558823529411764, -5.923333333333333, -5.9...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.2558823529411764, -5.2558823529411764, -5...","[-5.204557422969188, -5.204557422969188, -5.20...","[0.13173807277665758, 0.13173807277665758, 0.1..."
5,XGBRegressor,"[-5.2107687, -5.730269, -5.730269, -5.842719, ...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.2107687, -5.2107687, -5.2107687, -5.84271...","[-5.2045836, -5.2045836, -5.2045836, -5.843833...","[0.029835096, 0.029835096, 0.029835096, 0.0632..."
6,ExtraTreesRegressor,"[-5.210624999999996, -5.7300000000000075, -5.7...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.210624999999996, -5.210624999999996, -5.2...","[-5.204354131652659, -5.204354131652659, -5.20...","[0.029823554349075545, 0.029823554349075545, 0..."
7,LinearRegression,"[-5.210624999999999, -5.7299999999999995, -5.7...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.210624999999999, -5.210624999999999, -5.2...","[-5.204354131652661, -5.204354131652661, -5.20...","[0.02982355434907991, 0.02982355434907991, 0.0..."
8,KNeighborsRegressor,"[-5.613333333333333, -5.400000000000001, -5.40...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333333, -5.6...","[-5.6259999999999994, -5.6259999999999994, -5....","[0.16096100286853474, 0.16096100286853474, 0.1..."
9,SVR,"[-5.060262761083607, -5.95026271726338, -5.950...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.060262761083607, -5.060262761083607, -5.0...","[-5.112210003380566, -5.112210003380566, -5.11...","[0.08990841283326315, 0.08990841283326315, 0.0..."


In [56]:
df_AtomPairs2D_fp.to_csv('results/Fingerprints/Results_AtomPairs2D_fp_MDCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_AtomPairs2D_fp_MDCK.csv')

In [57]:
#AtomPairs2d Count fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/AtomPairs2DCount_train_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/AtomPairs2DCount_test_MDCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_AtomPairs2DCount_fp , pred_df= train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_AtomPairs2DCount_fp

X_train shape:  (51, 780)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 780)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.137163 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 81
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 12
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wi

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

0.4875395747514878


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.5031,0.5958,0.7093,0.0438,0.2262,0.2115,0.6334,0.6881,0.7959,0.1143,0.3749,0.3689
DecisionTreeRegressor,0.6010,0.5677,0.7752,-0.1422,0.4342,0.4518,0.5451,0.6278,0.7383,0.2377,0.4911,0.3554
RandomForestRegressor,0.4041,0.5033,0.6357,0.2320,0.4945,0.4877,0.4748,0.5965,0.6891,0.3360,0.5957,0.3912
GradientBoostingRegressor,0.4032,0.4833,0.6350,0.2337,0.5193,0.5327,0.4928,0.5669,0.7020,0.3109,0.5582,0.4408
AdaBoostRegressor,0.4299,0.5087,0.6556,0.1830,0.4735,0.4848,0.4635,0.5831,0.6808,0.3519,0.6193,0.4793
XGBRegressor,0.5803,0.5702,0.7618,-0.1029,0.3684,0.4381,0.5327,0.6155,0.7299,0.2551,0.5194,0.4601
ExtraTreesRegressor,0.4039,0.4748,0.6355,0.2324,0.5527,0.5553,0.4520,0.5500,0.6723,0.3679,0.6111,0.5592
LinearRegression,2.4471,1.1379,1.5643,-3.6508,0.0875,0.1310,0.4681,0.5615,0.6842,0.3454,0.6113,0.5317
KNeighborsRegressor,0.5153,0.5387,0.7179,0.0206,0.4145,0.5015,0.4060,0.5337,0.6371,0.4323,0.6732,0.5062
SVR,0.4593,0.5357,0.6777,0.1271,0.3805,0.3342,0.4505,0.5421,0.6712,0.3701,0.7143,0.5923


In [58]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.462604631468421, -5.875830560067116, -5.87...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.462604631468421, -5.462604631468421, -5.4...","[-5.461404620340332, -5.461404620340332, -5.46...","[0.11514935636214321, 0.11514935636214321, 0.1..."
1,DecisionTreeRegressor,"[-4.94, -4.58, -6.05, -5.585026652, -7.6989700...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.25, -6.25, -4.61, -5.289036881, -5.744727...","[-5.53, -5.964, -5.184, -5.7710235056000005, -...","[0.5547612098912468, 0.5719999999999998, 0.642..."
2,RandomForestRegressor,"[-5.231099999999997, -5.192299999999998, -5.99...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.449299999999997, -5.763600000000001, -5.4...","[-5.564299999999998, -5.645062288872001, -5.38...","[0.0942624209322031, 0.24667395645880003, 0.15..."
3,GradientBoostingRegressor,"[-4.955273061234525, -5.196137860848248, -6.12...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.127994249051142, -6.162881706229909, -5.3...","[-5.490194466730608, -5.9217106932653625, -5.3...","[0.32843424715043, 0.4210590611655802, 0.11941..."
4,AdaBoostRegressor,"[-5.2276149432727275, -5.245, -6.1683333333333...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.5425, -6.094925680454545, -6.094925680454...","[-5.543198762441577, -5.852894227, -5.74748422...","[0.18511227041890546, 0.3235628500232159, 0.21..."
5,XGBRegressor,"[-4.940943, -5.2996035, -6.07435, -5.353994, -...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.6933637, -6.2492557, -5.0523844, -5.37449...","[-5.67056, -6.006079, -5.276924, -5.5303483, -...","[0.3255424, 0.48640558, 0.2751475, 0.4387639, ..."
6,ExtraTreesRegressor,"[-4.972799999999998, -5.147399999999996, -6.13...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.424199999999997, -6.074900000000001, -5.3...","[-5.543059999999999, -5.90096, -5.321087274950...","[0.254651696244106, 0.41631373554087775, 0.145..."
7,LinearRegression,"[-5.827306067300834, -4.9549471631370094, -5.4...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.0, -5.43669508169484, -5.628362005238163,...","[-5.317871239446795, -6.524857711420536, -5.85...","[1.7841396994125907, 2.0628785558283873, 2.198..."
8,KNeighborsRegressor,"[-5.613333333333333, -5.3999999999999995, -5.8...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333334, -5.2...","[-5.6259999999999994, -5.666000000000001, -5.4...","[0.16096100286853474, 0.14081350945290821, 0.2..."
9,SVR,"[-5.087311009907667, -5.5975051793848065, -5.7...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.36043129015533, -5.099586403535032, -5.20...","[-5.597084613409837, -5.3386864071656746, -5.3...","[0.23202320440253998, 0.2737972979989138, 0.25..."


In [59]:
df_AtomPairs2DCount_fp.to_csv('results/Fingerprints/Results_AtomPairs2D_Count_fp_MDCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_df_AtomPairs2D_Count_fp_MDCK.csv')

In [60]:
#EState fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/EState_train_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/EState_test_MDCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_estate_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_estate_fp

X_train shape:  (51, 79)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 79)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 0
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there 

/tmp/ipykernel_2380537/3970550512.py:47: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearson_test, _ = pearsonr(y_test, predictions_test_mean)
/tmp/ipykernel_2380537/3970550512.py:48: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  spearman_test, _ = spearmanr(y_test, predictions_test_mean)


0.046575992293117374
0.04647296563733849
0.046580463635808256
0.05894909933689274
0.04669588987072737
0.04657599229311604
0.05825606841635356
-0.05827733826981829
-0.01859992290858714
0.04586365153225325


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.5359,0.6047,0.7320,-0.0185,-0.1795,-0.2310,0.7151,0.6961,0.8457,-0.0000,nan,nan
DecisionTreeRegressor,0.4972,0.5766,0.7051,0.0550,0.3083,0.2644,0.6818,0.7081,0.8257,0.0466,0.2810,0.2588
RandomForestRegressor,0.4853,0.5704,0.6966,0.0777,0.3243,0.2751,0.6819,0.7104,0.8258,0.0465,0.2800,0.2588
GradientBoostingRegressor,0.4972,0.5766,0.7051,0.0550,0.3083,0.2644,0.6818,0.7081,0.8257,0.0466,0.2810,0.2588
AdaBoostRegressor,0.5766,0.6314,0.7593,-0.0958,0.2103,0.2156,0.6730,0.7063,0.8203,0.0589,0.2928,0.2588
XGBRegressor,0.4970,0.5765,0.7050,0.0555,0.3085,0.2644,0.6817,0.7080,0.8257,0.0467,0.2810,0.2588
ExtraTreesRegressor,0.4972,0.5766,0.7051,0.0550,0.3083,0.2644,0.6818,0.7081,0.8257,0.0466,0.2810,0.2588
LinearRegression,0.4815,0.5618,0.6939,0.0849,0.3279,0.2567,0.6735,0.6957,0.8207,0.0583,0.2851,0.2588
KNeighborsRegressor,0.7386,0.6971,0.8594,-0.4037,-0.1651,-0.2097,0.7568,0.6849,0.8699,-0.0583,0.0545,-0.0833
SVR,0.5302,0.5713,0.7282,-0.0077,0.2642,0.2377,0.7284,0.6975,0.8535,-0.0186,0.2637,0.2032


In [61]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.6122931599617, -5.6122931599617, -5.612293...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.6122931599617, -5.6122931599617, -5.61229...","[-5.569499010748979, -5.569499010748979, -5.56...","[0.03286865439674929, 0.03286865439674929, 0.0..."
1,DecisionTreeRegressor,"[-5.029999999999999, -5.938571428571428, -5.93...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.029999999999999, -5.029999999999999, -5.0...","[-5.049516666666666, -5.049516666666666, -5.04...","[0.013276587580315352, 0.013276587580315352, 0..."
2,RandomForestRegressor,"[-5.031438413669662, -5.916997450327452, -5.91...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.031438413669662, -5.031438413669662, -5.0...","[-5.052640252753782, -5.052640252753782, -5.05...","[0.013431344611293873, 0.013431344611293873, 0..."
3,GradientBoostingRegressor,"[-5.0300154665205286, -5.938562762163825, -5.9...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.0300154665205286, -5.0300154665205286, -5...","[-5.04953184633621, -5.04953184633621, -5.0495...","[0.013276066595356024, 0.013276066595356024, 0..."
4,AdaBoostRegressor,"[-5.048181818181819, -5.923333333333335, -5.92...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.048181818181819, -5.048181818181819, -5.0...","[-5.084822222222223, -5.084822222222223, -5.08...","[0.02682969571550364, 0.02682969571550364, 0.0..."
5,XGBRegressor,"[-5.03038, -5.938377, -5.938377, -5.8428693, -...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.03038, -5.03038, -5.03038, -5.8428693, -5...","[-5.049924, -5.049924, -5.049924, -5.8439054, ...","[0.013233862, 0.013233862, 0.013233862, 0.0631..."
6,ExtraTreesRegressor,"[-5.029999999999988, -5.938571428571433, -5.93...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.029999999999988, -5.029999999999988, -5.0...","[-5.049516666666663, -5.049516666666663, -5.04...","[0.0132765875803172, 0.0132765875803172, 0.013..."
7,LinearRegression,"[-5.069924242424243, -5.995606060606061, -5.99...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.069924242424243, -5.069924242424243, -5.0...","[-5.1018749085884005, -5.1018749085884005, -5....","[0.030603563290909887, 0.030603563290909887, 0..."
8,KNeighborsRegressor,"[-5.613333333333333, -5.400000000000001, -5.40...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333333, -5.6...","[-5.6259999999999994, -5.6259999999999994, -5....","[0.16096100286853474, 0.16096100286853474, 0.1..."
9,SVR,"[-4.920051940436305, -6.199789500930947, -6.19...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.920051940436305, -4.920051940436305, -4.9...","[-4.920196314445822, -4.920196314445822, -4.92...","[0.00010751813541060764, 0.0001075181354106076..."


In [62]:
df_estate_fp.to_csv('results/Fingerprints/Results_EState_fp_MDCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_EState_fp_MDCK.csv')

In [63]:
#Extended fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/Extended_train_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/Extended_test_MDCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_extended_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_extended_fp

X_train shape:  (51, 1024)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 1024)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.110109 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 96
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 32
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.5344,0.6164,0.7310,-0.0156,0.1225,0.1646,0.6679,0.6975,0.8173,0.0660,0.2631,0.1887
DecisionTreeRegressor,0.7014,0.5999,0.8375,-0.3330,0.2892,0.4328,0.7010,0.7267,0.8372,0.0198,0.3631,0.1354
RandomForestRegressor,0.6081,0.6260,0.7798,-0.1557,0.2027,0.3121,0.5651,0.6501,0.7517,0.2098,0.4708,0.2800
GradientBoostingRegressor,0.6733,0.6014,0.8205,-0.2796,0.2413,0.3714,0.5659,0.6532,0.7522,0.2087,0.4706,0.2800
AdaBoostRegressor,0.6040,0.6136,0.7772,-0.1479,0.2239,0.3227,0.6620,0.7279,0.8137,0.0743,0.3310,0.2735
XGBRegressor,0.7152,0.6155,0.8457,-0.3593,0.2367,0.3735,0.6556,0.7078,0.8097,0.0832,0.3682,0.2579
ExtraTreesRegressor,0.6775,0.5916,0.8231,-0.2876,0.2989,0.4713,0.7124,0.7377,0.8441,0.0038,0.3428,0.1338
LinearRegression,0.7243,0.6747,0.8511,-0.3766,0.2301,0.2836,0.4084,0.5415,0.6391,0.4289,0.7003,0.5255
KNeighborsRegressor,0.6407,0.6610,0.8004,-0.2177,0.0801,0.1545,0.5799,0.6235,0.7615,0.1891,0.4802,0.4724
SVR,0.5253,0.5500,0.7248,0.0016,0.2637,0.2931,0.5049,0.5508,0.7105,0.2940,0.6539,0.4952


In [64]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.411197694774015, -5.813388622975106, -5.81...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.411197694774015, -5.411197694774015, -5.4...","[-5.367116342821914, -5.367116342821914, -5.36...","[0.14020492769216447, 0.14020492769216447, 0.1..."
1,DecisionTreeRegressor,"[-5.295, -5.7299999999999995, -5.7299999999999...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.6850000000000005, -4.6850000000000005, -5...","[-4.67, -4.67, -5.470000000000001, -5.77344462...","[0.03000000000000007, 0.03000000000000007, 0.1..."
2,RandomForestRegressor,"[-5.285679999999998, -5.72216782828283, -5.722...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.064519047619049, -5.105019537545716, -5.2...","[-5.089022154270573, -5.134963853635384, -5.44...","[0.05723216695206617, 0.0476257314625307, 0.11..."
3,GradientBoostingRegressor,"[-5.273853242397939, -5.732079657434891, -5.73...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.911502847134629, -4.911502847134629, -5.2...","[-5.160727002110415, -5.211709470970288, -5.41...","[0.3899283320527973, 0.33979098034069616, 0.15..."
4,AdaBoostRegressor,"[-5.090751332599999, -5.923333333333333, -5.92...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.96, -4.96, -5.090751332599999, -5.5462965...","[-5.024622807017544, -5.024622807017544, -5.22...","[0.14513865503684154, 0.14513865503684154, 0.1..."
5,XGBRegressor,"[-5.2942195, -5.729858, -5.729858, -5.6679044,...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.6218443, -4.6218443, -5.2942195, -5.72185...","[-5.00004, -5.2402315, -5.469335, -6.040741, -...","[0.6244969, 0.66916704, 0.14734796, 0.1623527,..."
6,ExtraTreesRegressor,"[-5.2950000000000035, -5.7300000000000075, -5....",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.664750000000003, -4.661750000000004, -5.2...","[-4.676810000000005, -4.749640000000005, -5.47...","[0.014066250388786006, 0.15982345071985005, 0...."
7,LinearRegression,"[-5.294999999999998, -5.73, -5.73, -5.66504993...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.8289854240373735, -5.847009411585862, -5....","[-5.741972831818893, -5.698926057867327, -5.47...","[0.5995995268232479, 0.5502066756212369, 0.147..."
8,KNeighborsRegressor,"[-5.183333333333334, -5.400000000000001, -5.40...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.206666666666667, -5.206666666666667, -5.1...","[-5.259333333333333, -5.259333333333333, -5.36...","[0.1053333333333331, 0.1053333333333331, 0.128..."
9,SVR,"[-4.9290299627997385, -5.9501115648643506, -5....",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.491241304940216, -5.50206375368502, -4.92...","[-5.431094686722692, -5.44026899280352, -5.055...","[0.2481608274452243, 0.24406948206894497, 0.06..."


In [65]:
df_extended_fp.to_csv('results/Fingerprints/Results_Extended_fp_MDCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_Extended_fp_MDCK.csv')

In [66]:
#Fingerprinter fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/Fingerprinter_train_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/Fingerprinter_test_MDCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_fingerprinter_fp , pred_df= train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_fingerprinter_fp

X_train shape:  (51, 1024)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 1024)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.107594 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 84
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 28
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.4950,0.5856,0.7036,0.0592,0.2597,0.2643,0.6421,0.6827,0.8013,0.1022,0.3264,0.3211
DecisionTreeRegressor,0.5744,0.5959,0.7579,-0.0916,0.2794,0.3199,0.5395,0.5942,0.7345,0.2456,0.5137,0.3831
RandomForestRegressor,0.5534,0.6142,0.7439,-0.0518,0.2229,0.2374,0.5632,0.6301,0.7505,0.2124,0.4737,0.3167
GradientBoostingRegressor,0.5492,0.5879,0.7411,-0.0438,0.2641,0.2828,0.5338,0.6113,0.7306,0.2535,0.5066,0.3499
AdaBoostRegressor,0.6645,0.6839,0.8151,-0.2628,0.2072,0.2294,0.6304,0.6712,0.7940,0.1185,0.3615,0.3167
XGBRegressor,0.5100,0.5694,0.7141,0.0308,0.3214,0.3657,0.5938,0.6691,0.7706,0.1696,0.4206,0.2061
ExtraTreesRegressor,0.5811,0.6078,0.7623,-0.1044,0.2606,0.2876,0.5449,0.6154,0.7382,0.2381,0.5000,0.3389
LinearRegression,0.6282,0.6210,0.7926,-0.1940,0.2239,0.2629,0.5225,0.5977,0.7228,0.2694,0.5210,0.4744
KNeighborsRegressor,0.5833,0.6498,0.7638,-0.1086,0.2203,0.2550,0.6650,0.6713,0.8155,0.0700,0.3184,0.3906
SVR,0.5289,0.5584,0.7272,-0.0052,0.2509,0.2703,0.5310,0.5648,0.7287,0.2574,0.6034,0.4633


In [67]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.411197694774015, -5.813388622975106, -5.81...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.411197694774015, -5.411197694774015, -5.4...","[-5.333929146201267, -5.333929146201267, -5.33...","[0.11335706042633634, 0.11335706042633634, 0.1..."
1,DecisionTreeRegressor,"[-4.99, -5.73, -5.73, -5.728279051285713, -5.7...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.25, -6.25, -4.99, -5.516882188, -5.728279...","[-5.4952, -5.469200000000001, -5.1716999999999...","[0.6510638678347922, 0.6223652946622263, 0.169..."
2,RandomForestRegressor,"[-5.022796309523809, -5.7221678282828305, -5.7...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.502129285714287, -5.569183684422383, -5.0...","[-5.444813305005538, -5.485868712222982, -5.19...","[0.17613877356539331, 0.1273865639031989, 0.16..."
3,GradientBoostingRegressor,"[-4.986968703344718, -5.7287837467276, -5.7287...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.030966662988224, -6.23956627067099, -4.98...","[-5.821395645266517, -5.957845668868661, -5.16...","[0.5419098338439216, 0.5207629147101, 0.165698..."
4,AdaBoostRegressor,"[-4.91, -5.923333333333333, -5.923333333333333...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.43, -6.12, -4.91, -5.516882188, -6.302579...","[-5.347792460525926, -5.485792460525927, -5.12...","[0.4410702482872722, 0.5416717131427857, 0.183..."
5,XGBRegressor,"[-4.9898105, -5.7299194, -5.7299194, -5.728260...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.248285, -6.1813273, -4.9898105, -5.516908...","[-5.8726034, -6.046617, -5.171443, -5.582341, ...","[0.53955495, 0.14968191, 0.16908976, 0.1310523..."
6,ExtraTreesRegressor,"[-4.990000000000006, -5.7300000000000075, -5.7...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.25, -6.2476, -4.990000000000006, -5.51688...","[-5.502080000000002, -5.601180000000001, -5.17...","[0.6417890195383502, 0.5137145604321517, 0.169..."
7,LinearRegression,"[-4.989999999999998, -5.730000000000001, -5.73...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.289999999999997, -6.1957907427697325, -4....","[-6.085851411132666, -5.951917447120097, -5.17...","[0.5375503899981857, 0.4475563684058064, 0.169..."
8,KNeighborsRegressor,"[-5.183333333333334, -5.400000000000001, -5.40...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333333, -5.1...","[-5.666, -5.666, -5.368, -5.562923063266666, -...","[0.14081350945290877, 0.14081350945290877, 0.1..."
9,SVR,"[-4.922302979754659, -5.949935388787168, -5.94...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.451038241526298, -5.464780671118978, -4.9...","[-5.394862370150271, -5.4055861731014785, -5.0...","[0.19894572096165108, 0.1964424601252415, 0.06..."


In [68]:
df_fingerprinter_fp.to_csv('results/Fingerprints/Results_Fingerprinter_fp_MDCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_Fingerprinter_fp_MDCK.csv')

In [69]:
#GraphOnly fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/Graphonly_train_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/Graphonly_test_MDCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_graph_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_graph_fp

X_train shape:  (51, 1024)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 1024)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.160706 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 54
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 18
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.5188,0.6027,0.7203,0.0140,0.1861,0.1918,0.6481,0.6802,0.8050,0.0937,0.3214,0.2372
DecisionTreeRegressor,0.6105,0.6017,0.7813,-0.1603,0.2276,0.2867,0.5169,0.6447,0.7190,0.2772,0.5281,0.5090
RandomForestRegressor,0.5882,0.6231,0.7670,-0.1179,0.1967,0.2690,0.4895,0.6212,0.6996,0.3155,0.5900,0.4758
GradientBoostingRegressor,0.6170,0.6205,0.7855,-0.1726,0.2033,0.2571,0.4641,0.5900,0.6813,0.3510,0.6078,0.5560
AdaBoostRegressor,0.6317,0.6370,0.7948,-0.2006,0.1566,0.2399,0.5047,0.6305,0.7104,0.2942,0.5704,0.4979
XGBRegressor,0.6375,0.6214,0.7984,-0.2115,0.1942,0.2580,0.4725,0.6074,0.6874,0.3392,0.5940,0.5560
ExtraTreesRegressor,0.6242,0.6082,0.7901,-0.1863,0.2122,0.2792,0.4754,0.5990,0.6895,0.3352,0.5867,0.5671
LinearRegression,0.6137,0.6232,0.7834,-0.1664,0.2109,0.2722,0.4930,0.6063,0.7021,0.3106,0.5615,0.5477
KNeighborsRegressor,0.6603,0.6991,0.8126,-0.2549,0.1045,0.0665,0.5798,0.6330,0.7614,0.1893,0.4599,0.4592
SVR,0.5369,0.5482,0.7327,-0.0203,0.2539,0.3128,0.5240,0.5542,0.7239,0.2673,0.6187,0.5007


In [70]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.411197694774015, -5.813388622975106, -5.81...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.411197694774015, -5.411197694774015, -5.4...","[-5.341228542168531, -5.341228542168531, -5.34...","[0.14834679958972471, 0.14834679958972471, 0.1..."
1,DecisionTreeRegressor,"[-4.898333333333333, -5.7299999999999995, -5.7...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.25, -6.25, -4.898333333333333, -5.5168821...","[-6.095000000000001, -6.095000000000001, -5.04...","[0.31000000000000016, 0.31000000000000016, 0.1..."
2,RandomForestRegressor,"[-4.9093557972582955, -5.722167828282828, -5.7...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.72323013708514, -5.72323013708514, -4.909...","[-5.604518421163562, -5.604518421163562, -5.07...","[0.2735867922085509, 0.2735867922085509, 0.159..."
3,GradientBoostingRegressor,"[-4.898080557860812, -5.7292525962616, -5.7292...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.244274659335176, -6.244274659335176, -4.8...","[-5.934869757994774, -5.934869757994774, -5.04...","[0.6207744605296873, 0.6207744605296873, 0.143..."
4,AdaBoostRegressor,"[-4.91, -5.923333333333333, -5.923333333333333...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.43, -5.43, -4.91, -5.577807376200001, -5....","[-5.642999999999999, -5.642999999999999, -5.04...","[0.48122344082556906, 0.48122344082556906, 0.0..."
5,XGBRegressor,"[-4.8983436, -5.7299175, -5.7299175, -5.669708...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.2481008, -6.2481008, -4.8983436, -5.51259...","[-5.9209914, -5.9209914, -5.0484705, -5.752773...","[0.65471137, 0.65471137, 0.1446589, 0.21338573..."
6,ExtraTreesRegressor,"[-4.898333333333325, -5.7300000000000075, -5.7...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.25, -6.25, -4.898333333333325, -5.5168821...","[-5.935170000000002, -5.935170000000002, -5.04...","[0.6296599999999963, 0.6296599999999963, 0.144..."
7,LinearRegression,"[-4.898333333333332, -5.7299999999999995, -5.7...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.249999999999999, -6.249999999999999, -4.8...","[-5.938223960775576, -5.938223960775576, -5.04...","[0.6235520784488474, 0.6235520784488474, 0.144..."
8,KNeighborsRegressor,"[-5.136666666666667, -5.3999999999999995, -5.3...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.503333333333333, -5.503333333333333, -5.1...","[-5.472, -5.472, -5.34, -5.669417907400001, -5...","[0.18465102220134053, 0.18465102220134053, 0.1..."
9,SVR,"[-4.913372558968358, -5.949551021325661, -5.94...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.428406630070809, -5.428406630070809, -4.9...","[-5.354849474703567, -5.354849474703567, -4.97...","[0.31364235762039616, 0.31364235762039616, 0.0..."


In [71]:
df_graph_fp.to_csv('results/Fingerprints/Results_Graphonly_fp_MDCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_Graphonly_fp_MDCK.csv')

In [72]:
#KlekotaRoth fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/KlekotaRoth_train_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/KlekotaRoth_test_MDCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_KlekotaRoth_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_KlekotaRoth_fp

X_train shape:  (51, 4860)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 4860)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.097750 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 75
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 25
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positi

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


0.5479028451889164


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.5240,0.6033,0.7239,0.0042,0.1801,0.1713,0.6728,0.6958,0.8202,0.0592,0.2444,0.1183
DecisionTreeRegressor,0.5224,0.5298,0.7228,0.0072,0.5009,0.2917,0.3289,0.4058,0.5735,0.5401,0.7859,0.8147
RandomForestRegressor,0.3227,0.4663,0.5680,0.3867,0.6231,0.4825,0.3582,0.4384,0.5985,0.4991,0.8022,0.8398
GradientBoostingRegressor,0.3692,0.4870,0.6077,0.2982,0.5936,0.4673,0.3221,0.4020,0.5675,0.5496,0.8036,0.7873
AdaBoostRegressor,0.3052,0.4379,0.5525,0.4199,0.6519,0.5245,0.3557,0.4413,0.5964,0.5026,0.8350,0.8564
XGBRegressor,0.3855,0.4866,0.6209,0.2673,0.5938,0.4151,0.3341,0.3904,0.5780,0.5328,0.7714,0.7873
ExtraTreesRegressor,0.5194,0.5274,0.7207,0.0129,0.5100,0.3307,0.3154,0.3953,0.5616,0.5590,0.7927,0.8036
LinearRegression,0.7740,0.7353,0.8798,-0.4710,0.4560,0.2891,0.5352,0.5900,0.7316,0.2516,0.5444,0.6077
KNeighborsRegressor,0.5990,0.6328,0.7739,-0.1384,0.3408,0.4537,0.4869,0.5022,0.6978,0.3191,0.6373,0.6238
SVR,0.4573,0.5320,0.6763,0.1308,0.3790,0.4335,0.4116,0.4873,0.6416,0.4244,0.7947,0.6961


In [73]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.411197694774015, -5.813388622975106, -5.81...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.411197694774015, -5.411197694774015, -5.4...","[-5.318604437320289, -5.318604437320289, -5.31...","[0.055709001717999765, 0.055709001717999765, 0..."
1,DecisionTreeRegressor,"[-5.295, -6.3, -4.95, -5.493665733, -7.6989700...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.25, -6.25, -6.17, -5.289036881, -5.185752...","[-5.922, -5.922, -5.6245, -5.1532295048000005,...","[0.6559999999999998, 0.6559999999999998, 0.691..."
2,RandomForestRegressor,"[-5.363520111860001, -6.167891666666667, -5.10...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.878315757608955, -5.883906298576733, -5.5...","[-5.754627999698757, -5.7472616005198365, -5.4...","[0.27960223350973956, 0.27520214525991105, 0.0..."
3,GradientBoostingRegressor,"[-5.318617702669948, -6.25849812418385, -4.960...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.154430343201205, -6.225278114250181, -5.7...","[-5.884409744769452, -5.897654680732079, -5.75...","[0.5248162784708812, 0.5308347157466331, 0.215..."
4,AdaBoostRegressor,"[-5.422, -6.152547488363635, -5.129011186, -5....",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.744545454545454, -5.754444444444445, -5.9...","[-5.665913986769091, -5.683546052146465, -5.80...","[0.21640285878426496, 0.21999804700792117, 0.1..."
5,XGBRegressor,"[-5.29549, -6.2984986, -4.9500446, -5.260342, ...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.2480006, -6.2480006, -5.2081156, -5.29161...","[-6.015321, -6.020434, -5.570546, -5.3288913, ...","[0.46513164, 0.40456662, 0.441572, 0.07685929,..."
6,ExtraTreesRegressor,"[-5.2950000000000035, -6.299999999999998, -4.9...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.25, -6.25, -6.175250000000001, -5.2890368...","[-5.922000000000002, -5.922000000000002, -5.80...","[0.6559999999999959, 0.6559999999999959, 0.552..."
7,LinearRegression,"[-5.295, -6.300000000000001, -4.63744578983082...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.2500000000000036, -6.58783560705526, -7.6...","[-5.987452934586721, -6.599359688697703, -6.84...","[0.5250941308265619, 0.9405919351037454, 0.871..."
8,KNeighborsRegressor,"[-5.183333333333334, -6.25, -5.316666666666666...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.503333333333333, -5.503333333333333, -4.8...","[-5.472, -5.472, -4.761333333333334, -5.532715...","[0.18465102220134053, 0.18465102220134053, 0.2..."
9,SVR,"[-5.237500146566388, -6.057421449144197, -5.50...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.647711438463738, -5.647927552696668, -5.3...","[-5.554932114205404, -5.5506254997076585, -5.2...","[0.41778604144334136, 0.40966212532464735, 0.0..."


In [74]:
df_KlekotaRoth_fp.to_csv('results/Fingerprints/Results_KlekotaRoth_fp_MDCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_KlekotaRoth_fp_MDCK.csv')

In [75]:
#KlekotaRoth Count fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/KlekotaRothCount_train_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/KlekotaRothCount_test_MDCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_KlekotaRothCount_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_KlekotaRothCount_fp

X_train shape:  (51, 4860)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 4860)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.109762 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 177
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 38
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


0.4934862848710614


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.5511,0.6240,0.7424,-0.0474,0.1319,0.0968,0.5899,0.6490,0.7681,0.1751,0.5174,0.3729
DecisionTreeRegressor,0.4659,0.5070,0.6825,0.1146,0.5280,0.3667,0.5537,0.5264,0.7441,0.2258,0.4871,0.4386
RandomForestRegressor,0.3135,0.4662,0.5599,0.4041,0.6364,0.5168,0.4913,0.5335,0.7009,0.3130,0.5731,0.6097
GradientBoostingRegressor,0.3194,0.4321,0.5652,0.3929,0.6655,0.5319,0.5463,0.5443,0.7391,0.2361,0.4956,0.4772
AdaBoostRegressor,0.2404,0.3958,0.4903,0.5431,0.7403,0.6474,0.5059,0.5373,0.7113,0.2926,0.5465,0.5986
XGBRegressor,0.3266,0.4457,0.5715,0.3792,0.6521,0.5028,0.5670,0.5546,0.7530,0.2071,0.4731,0.4276
ExtraTreesRegressor,0.3363,0.4677,0.5799,0.3609,0.6269,0.4847,0.5009,0.5231,0.7078,0.2995,0.5525,0.4552
LinearRegression,0.6360,0.6106,0.7975,-0.2087,0.4737,0.4757,0.8824,0.6366,0.9393,-0.2339,0.4902,0.4690
KNeighborsRegressor,0.5518,0.6060,0.7429,-0.0488,0.3761,0.4516,0.4439,0.5027,0.6663,0.3793,0.6669,0.5318
SVR,0.4055,0.5035,0.6368,0.2293,0.4877,0.5346,0.4507,0.5224,0.6713,0.3698,0.6539,0.5545


In [76]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.534860730823712, -5.73792077160033, -5.769...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.534860730823712, -5.819825885165192, -5.5...","[-5.488903110576687, -5.482782808974274, -5.41...","[0.08342313843729457, 0.2382574215119878, 0.08..."
1,DecisionTreeRegressor,"[-5.295, -4.95, -6.3, -5.493665733, -7.6989700...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.61, -6.25, -5.295, -5.289036881, -5.18575...","[-4.732, -5.922, -6.079000000000001, -5.340918...","[0.24399999999999977, 0.6559999999999998, 0.41..."
2,RandomForestRegressor,"[-5.37568, -5.430269400080001, -5.588852270461...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.035020954665007, -5.993038190358326, -5.8...","[-5.096043848953675, -5.778669420950129, -5.78...","[0.21400535417257613, 0.2874638192041783, 0.10..."
3,GradientBoostingRegressor,"[-5.296644472128322, -5.073764453752048, -6.03...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.676859190235919, -6.239872030565247, -5.7...","[-4.874941455614576, -6.002584604733476, -6.03...","[0.39457515755792033, 0.3582199087732535, 0.25..."
4,AdaBoostRegressor,"[-5.215, -5.2023464332, -5.055714285714286, -5...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.923917555222221, -6.1049999999999995, -6....","[-4.9755058401111105, -5.792145114114675, -5.8...","[0.1772715575027063, 0.2298231449484629, 0.164..."
5,XGBRegressor,"[-5.295013, -5.5919776, -5.6884923, -5.438094,...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.6145477, -6.2375073, -5.4409127, -5.28960...","[-4.795109, -6.021085, -5.950311, -5.33987, -5...","[0.27423543, 0.41939843, 0.3612201, 0.10087944..."
6,ExtraTreesRegressor,"[-5.2950000000000035, -5.922000000000001, -5.2...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.637750000000009, -6.227250040585002, -5.4...","[-4.907146417788009, -5.894160620649574, -5.85...","[0.5176803023922139, 0.6110649218080428, 0.422..."
7,LinearRegression,"[-5.6341570785675685, -5.803560401638584, -4.7...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.538903121199564, -5.99386614502865, -7.24...","[-4.535058330956325, -5.22407815931387, -8.513...","[0.2816649545563694, 0.6588536293952653, 0.884..."
8,KNeighborsRegressor,"[-5.613333333333333, -5.3999999999999995, -5.3...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.503333333333333, -5.503333333333333, -5.6...","[-5.472, -5.472, -5.482666666666667, -5.532715...","[0.18465102220134053, 0.18465102220134053, 0.0..."
9,SVR,"[-5.489601100003094, -5.590301249736795, -5.70...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.4008228989665374, -5.933735596172749, -5....","[-5.378608416500381, -5.7840094855281015, -5.5...","[0.3154776760044484, 0.2929934843096387, 0.065..."


In [77]:
df_KlekotaRothCount_fp.to_csv('results/Fingerprints/Results_KlekotaRoth_Count_fp_MDCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_KlekotaRoth_Count_fp_MDCK.csv')

In [78]:
#MACCS fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/MACCS_train_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/MACCS_test_MDCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_MACCS_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_MACCS_fp

X_train shape:  (51, 166)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 166)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.203793 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 1
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

0.36649930592762026


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.5240,0.6032,0.7239,0.0042,0.1827,0.1599,0.6725,0.6962,0.8201,0.0596,0.2456,0.1183
DecisionTreeRegressor,0.7412,0.6634,0.8609,-0.4086,0.2510,0.3030,0.5186,0.5567,0.7202,0.2748,0.5415,0.5560
RandomForestRegressor,0.6064,0.6357,0.7787,-0.1526,0.2790,0.3270,0.4959,0.5402,0.7042,0.3066,0.5995,0.6036
GradientBoostingRegressor,0.5876,0.6177,0.7666,-0.1168,0.3315,0.3634,0.4947,0.5150,0.7034,0.3082,0.5725,0.5870
AdaBoostRegressor,0.5454,0.6071,0.7385,-0.0365,0.3645,0.3563,0.5059,0.5406,0.7113,0.2925,0.5762,0.5373
XGBRegressor,0.6441,0.6180,0.8026,-0.2242,0.3145,0.4005,0.4913,0.5267,0.7010,0.3129,0.5855,0.6395
ExtraTreesRegressor,0.7162,0.6537,0.8463,-0.3611,0.2646,0.3174,0.5209,0.5638,0.7217,0.2716,0.5362,0.5704
LinearRegression,0.6155,0.6279,0.7845,-0.1697,0.3914,0.4266,0.5988,0.5804,0.7738,0.1626,0.4946,0.5594
KNeighborsRegressor,0.6237,0.6643,0.7898,-0.1854,0.3134,0.3817,0.5580,0.5569,0.7470,0.2197,0.5240,0.5698
SVR,0.5450,0.5761,0.7382,-0.0357,0.2250,0.2582,0.4854,0.5321,0.6967,0.3212,0.6247,0.6588


In [79]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.411197694774015, -5.813388622975106, -5.81...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.411197694774015, -5.411197694774015, -5.4...","[-5.313430455830277, -5.313430455830277, -5.31...","[0.05587638162241636, 0.05587638162241636, 0.0..."
1,DecisionTreeRegressor,"[-5.295, -6.25, -6.25, -5.157381794999999, -6....",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.25, -6.25, -4.6850000000000005, -5.516882...","[-5.922, -5.922, -4.8875, -5.5138436396, -5.15...","[0.6559999999999998, 0.6559999999999998, 0.404..."
2,RandomForestRegressor,"[-5.2912799999999995, -6.211125000000002, -6.2...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.815753679429285, -5.750158054012897, -4.8...","[-5.630356327323841, -5.592293229084855, -5.04...","[0.3358561754516854, 0.3023223311519646, 0.194..."
3,GradientBoostingRegressor,"[-5.272641783676473, -6.221362630972792, -6.22...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.210183950188246, -6.366465236097036, -4.6...","[-5.966388702028814, -5.9374359201064575, -4.8...","[0.5235713877759749, 0.5170966597235204, 0.338..."
4,AdaBoostRegressor,"[-5.4316717685625004, -6.225, -6.225, -5.32294...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.4316717685625004, -5.741961592318182, -4....","[-5.629401020379167, -5.808679587990559, -4.96...","[0.335788645215997, 0.29507856127486987, 0.241..."
5,XGBRegressor,"[-5.295401, -6.2498174, -6.2498174, -5.171926,...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.247373, -6.1311946, -4.6857786, -5.516731...","[-5.9808908, -5.740996, -4.9264793, -5.5149784...","[0.5346146, 0.45513728, 0.31402513, 0.00590633..."
6,ExtraTreesRegressor,"[-5.2950000000000035, -6.25, -6.25, -5.1324959...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.25, -6.25, -4.685000000000001, -5.5168821...","[-5.925900000000001, -5.924600000000002, -4.88...","[0.6481999999999963, 0.6475548934260287, 0.404..."
7,LinearRegression,"[-5.279727422811152, -6.249999999999999, -6.24...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.2805451543776964, -6.550913895382942, -5....","[-6.391308048174271, -6.830715544675373, -6.03...","[0.13989090773171584, 0.1651972351852128, 0.39..."
8,KNeighborsRegressor,"[-5.136666666666667, -6.25, -6.25, -5.35316369...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.503333333333333, -5.503333333333333, -4.7...","[-5.472, -5.472, -4.7, -5.461041660666667, -5....","[0.18465102220134053, 0.18465102220134053, 0.0..."
9,SVR,"[-5.058701423553831, -5.949924297331553, -5.94...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.1112721140968205, -6.07365924696715, -4.9...","[-5.845452520851954, -5.823356660768288, -4.98...","[0.40975442385666594, 0.38025197608916966, 0.0..."


In [80]:
df_MACCS_fp.to_csv('results/Fingerprints/Results_MACCS_fp_MDCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_MACCS_fp_MDCK.csv')

In [81]:
#PubChem fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/PubChem_train_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/PubChem_test_MDCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_PubChem_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_PubChem_fp

X_train shape:  (51, 881)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 881)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.097511 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 18
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 6
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wit

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.5383,0.6132,0.7337,-0.0230,0.0631,0.0921,0.6705,0.6871,0.8188,0.0624,0.2886,0.2688
DecisionTreeRegressor,0.5683,0.5924,0.7538,-0.0800,0.3021,0.2999,0.4838,0.5822,0.6956,0.3235,0.5740,0.4598
RandomForestRegressor,0.5309,0.5955,0.7287,-0.0091,0.2914,0.2892,0.4910,0.5805,0.7007,0.3134,0.5808,0.4488
GradientBoostingRegressor,0.5231,0.5789,0.7233,0.0058,0.3398,0.3138,0.4854,0.5811,0.6967,0.3213,0.5726,0.4598
AdaBoostRegressor,0.5507,0.6272,0.7421,-0.0466,0.3198,0.3438,0.6125,0.6527,0.7826,0.1435,0.3956,0.2825
XGBRegressor,0.5330,0.5633,0.7301,-0.0130,0.3406,0.3399,0.4814,0.5756,0.6938,0.3269,0.5778,0.4626
ExtraTreesRegressor,0.5445,0.5803,0.7379,-0.0349,0.3270,0.3186,0.4850,0.5831,0.6964,0.3218,0.5715,0.4598
LinearRegression,0.5633,0.5898,0.7505,-0.0706,0.3068,0.3042,0.4701,0.5570,0.6856,0.3426,0.6094,0.4737
KNeighborsRegressor,0.5958,0.6490,0.7719,-0.1324,0.2360,0.2574,0.6277,0.6456,0.7922,0.1223,0.4015,0.4792
SVR,0.5197,0.5569,0.7209,0.0123,0.2659,0.2879,0.4816,0.5507,0.6940,0.3265,0.6811,0.5235


In [82]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.411197694774015, -5.813388622975106, -5.81...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.411197694774015, -5.411197694774015, -5.4...","[-5.448700397817824, -5.448700397817824, -5.44...","[0.11989892378494135, 0.11989892378494135, 0.1..."
1,DecisionTreeRegressor,"[-5.295, -5.73, -5.73, -5.728279051285715, -5....",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.25, -6.25, -5.295, -5.516882188, -5.72827...","[-5.922, -5.922, -5.470000000000001, -5.626305...","[0.6559999999999998, 0.6559999999999998, 0.147..."
2,RandomForestRegressor,"[-5.295354999999998, -5.722167828282828, -5.72...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.847605455306953, -5.847605455306953, -5.2...","[-5.678588744192277, -5.678588744192277, -5.46...","[0.3184531402789525, 0.3184531402789525, 0.141..."
3,GradientBoostingRegressor,"[-5.294867046734345, -5.741321707132634, -5.74...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.188426155808186, -6.188426155808186, -5.2...","[-5.925279856439175, -5.925279856439175, -5.46...","[0.565724776878558, 0.565724776878558, 0.14742..."
4,AdaBoostRegressor,"[-4.96, -5.923333333333335, -5.923333333333335...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.43, -5.43, -4.96, -5.957294062666666, -6....","[-5.290363025210084, -5.290363025210084, -5.24...","[0.31922678025699264, 0.31922678025699264, 0.1..."
5,XGBRegressor,"[-5.294531, -5.7299685, -5.7299685, -5.7283697...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.247788, -6.247788, -5.294531, -5.517509, ...","[-5.9311533, -5.9311533, -5.4696136, -5.519079...","[0.6329988, 0.6329988, 0.14770378, 0.003786926..."
6,ExtraTreesRegressor,"[-5.2950000000000035, -5.7300000000000075, -5....",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.25, -6.25, -5.2950000000000035, -5.516882...","[-5.944800000000002, -5.944800000000002, -5.47...","[0.6103999999999971, 0.6103999999999971, 0.147..."
7,LinearRegression,"[-5.601176470588236, -5.73, -5.73, -5.72827905...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.637647058823529, -5.637647058823529, -5.6...","[-5.619219251336898, -5.619219251336898, -5.63...","[0.491050511010694, 0.491050511010694, 0.18071..."
8,KNeighborsRegressor,"[-5.136666666666667, -5.400000000000001, -5.40...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333333, -5.1...","[-5.666, -5.666, -5.34, -5.562923063266666, -6...","[0.14081350945290877, 0.14081350945290877, 0.1..."
9,SVR,"[-5.0400480462613695, -5.949655291844474, -5.9...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.617478641846199, -5.617478641846199, -5.0...","[-5.544508486613557, -5.544508486613557, -5.24...","[0.23816211796861347, 0.23816211796861347, 0.2..."


In [83]:
df_PubChem_fp.to_csv('results/Fingerprints/Results_PubChem_fp_MDCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_PubChem_fp_MDCK.csv')

In [84]:
#Substructure fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/Substructure_train_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/Substructure_test_MDCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_Substructure_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_Substructure_fp

X_train shape:  (51, 307)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 307)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 0
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because ther

/tmp/ipykernel_2380537/3970550512.py:47: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  pearson_test, _ = pearsonr(y_test, predictions_test_mean)
/tmp/ipykernel_2380537/3970550512.py:48: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  spearman_test, _ = spearmanr(y_test, predictions_test_mean)


0.055784696139162815
0.05684017009789977
0.05377714242020315
0.038727033221868434
0.05586352355412394
0.05578469613916148
0.048558119996843785
0.006649010818622458
-0.01725334199514217


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

0.029092747465239377


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.5359,0.6047,0.7320,-0.0185,-0.1795,-0.2310,0.7151,0.6961,0.8457,-0.0000,nan,nan
DecisionTreeRegressor,0.4264,0.5016,0.6530,0.1897,0.4553,0.4756,0.6752,0.6946,0.8217,0.0558,0.3039,0.2588
RandomForestRegressor,0.4306,0.5228,0.6562,0.1817,0.4349,0.4585,0.6745,0.6981,0.8213,0.0568,0.3022,0.2588
GradientBoostingRegressor,0.4318,0.5125,0.6571,0.1794,0.4416,0.4651,0.6767,0.6993,0.8226,0.0538,0.3002,0.2588
AdaBoostRegressor,0.4955,0.5720,0.7039,0.0582,0.3358,0.3481,0.6874,0.7279,0.8291,0.0387,0.2698,0.2325
XGBRegressor,0.4449,0.5197,0.6670,0.1545,0.4241,0.4449,0.6752,0.6946,0.8217,0.0559,0.3038,0.2588
ExtraTreesRegressor,0.4285,0.5072,0.6546,0.1856,0.4501,0.4599,0.6752,0.6946,0.8217,0.0558,0.3039,0.2588
LinearRegression,0.4249,0.5076,0.6518,0.1925,0.4528,0.4647,0.6804,0.6997,0.8249,0.0486,0.3000,0.2588
KNeighborsRegressor,0.6139,0.6275,0.7835,-0.1668,0.0923,0.1122,0.7104,0.6649,0.8428,0.0066,0.1874,0.1771
SVR,0.4662,0.5293,0.6828,0.1139,0.3955,0.3323,0.7275,0.6937,0.8529,-0.0173,0.2964,0.2588


In [85]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.6122931599617, -5.6122931599617, -5.612293...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.6122931599617, -5.6122931599617, -5.61229...","[-5.569499010748979, -5.569499010748979, -5.56...","[0.03286865439674929, 0.03286865439674929, 0.0..."
1,DecisionTreeRegressor,"[-5.03, -5.73, -5.73, -5.8427817234666675, -5....",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.03, -5.03, -5.03, -5.8427817234666675, -5...","[-5.049516666666667, -5.049516666666667, -5.04...","[0.013276587580315233, 0.013276587580315233, 0..."
2,RandomForestRegressor,"[-5.031438413669662, -5.7221678282828305, -5.7...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.031438413669662, -5.031438413669662, -5.0...","[-5.052640252753782, -5.052640252753782, -5.05...","[0.013431344611293904, 0.013431344611293904, 0..."
3,GradientBoostingRegressor,"[-5.0230845608555335, -5.717598197757037, -5.7...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.0230845608555335, -5.0230845608555335, -5...","[-5.045746844158459, -5.045746844158459, -5.04...","[0.013891785175811671, 0.013891785175811671, 0..."
4,AdaBoostRegressor,"[-5.048181818181819, -5.649166666666666, -5.64...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.048181818181819, -5.048181818181819, -5.0...","[-5.0637452991453, -5.0637452991453, -5.063745...","[0.04309266304542953, 0.04309266304542953, 0.0..."
5,XGBRegressor,"[-5.0301476, -5.7298985, -5.7298985, -5.842714...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.0301476, -5.0301476, -5.0301476, -5.84271...","[-5.0497975, -5.0497975, -5.0497975, -5.843862...","[0.013277847, 0.013277847, 0.013277847, 0.0632..."
6,ExtraTreesRegressor,"[-5.029999999999988, -5.7300000000000075, -5.7...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.029999999999988, -5.029999999999988, -5.0...","[-5.049516666666663, -5.049516666666663, -5.04...","[0.0132765875803172, 0.0132765875803172, 0.013..."
7,LinearRegression,"[-5.004814814814815, -5.73, -5.73, -5.84278172...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.004814814814815, -5.004814814814815, -5.0...","[-5.02760605307572, -5.02760605307572, -5.0276...","[0.02346777200512083, 0.02346777200512083, 0.0..."
8,KNeighborsRegressor,"[-5.613333333333333, -5.400000000000001, -5.40...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333333, -5.6...","[-5.6259999999999994, -5.6259999999999994, -5....","[0.16096100286853474, 0.16096100286853474, 0.1..."
9,SVR,"[-4.919854487446512, -5.949878525498174, -5.94...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.919854487446512, -4.919854487446512, -4.9...","[-4.920021439245367, -4.920021439245367, -4.92...","[8.677075376552815e-05, 8.677075376552815e-05,..."


In [86]:
df_Substructure_fp.to_csv('results/Fingerprints/Results_Substructure_fp_MDCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_Substructure_fp_MDCK.csv')

In [87]:
#Substructure Count fingerprints
df_train = pd.read_csv('features/Fingerprints/Train/SubstructureCount_train_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Fingerprints/Test/SubstructureCount_test_MDCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
df_SubstructureCount_fp, pred_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
df_SubstructureCount_fp

X_train shape:  (51, 307)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 307)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.133283 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 18
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 2
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

0.4314064719754537


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.5108,0.5777,0.7147,0.0292,0.2117,0.2092,0.6401,0.6734,0.8001,0.1049,0.4057,0.4777
DecisionTreeRegressor,0.5328,0.5148,0.7299,-0.0126,0.5051,0.4450,0.6049,0.6105,0.7777,0.1542,0.4315,0.3062
RandomForestRegressor,0.3552,0.4728,0.5960,0.3249,0.5870,0.5019,0.5347,0.5597,0.7312,0.2523,0.5024,0.3697
GradientBoostingRegressor,0.5025,0.5168,0.7088,0.0451,0.5359,0.4930,0.5570,0.5388,0.7463,0.2211,0.4881,0.4276
AdaBoostRegressor,0.3054,0.4443,0.5526,0.4195,0.6564,0.4976,0.5718,0.6090,0.7562,0.2005,0.4506,0.4303
XGBRegressor,0.5309,0.5285,0.7287,-0.0091,0.5221,0.4706,0.5142,0.5479,0.7170,0.2810,0.5314,0.4276
ExtraTreesRegressor,0.3140,0.4191,0.5604,0.4032,0.6793,0.5691,0.5641,0.5595,0.7511,0.2112,0.4775,0.3172
LinearRegression,0.5859,0.5863,0.7654,-0.1134,0.4236,0.4914,0.4948,0.4881,0.7034,0.3081,0.5999,0.5848
KNeighborsRegressor,0.3213,0.4471,0.5668,0.3893,0.6377,0.5837,0.4217,0.4728,0.6494,0.4103,0.6810,0.6067
SVR,0.4285,0.5132,0.6546,0.1857,0.4438,0.4636,0.4527,0.5381,0.6728,0.3670,0.6956,0.5793


In [88]:
pred_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.599003550019117, -5.599003550019117, -5.59...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.599003550019117, -5.625582771608608, -5.5...","[-5.491429773814767, -5.440334738647157, -5.41...","[0.15003012002299942, 0.2011587290304711, 0.09..."
1,DecisionTreeRegressor,"[-5.295, -4.95, -6.3, -5.493665733, -7.6989700...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.61, -6.25, -6.4, -5.289036881, -5.1857524...","[-5.082000000000001, -6.093999999999999, -5.77...","[0.34787354024127787, 0.3120000000000001, 0.56..."
2,RandomForestRegressor,"[-5.327255, -5.064316666666663, -6.09973333333...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.2117480060631705, -6.1322033671066665, -5...","[-5.185826839992252, -6.18155565501752, -5.790...","[0.08868213669680895, 0.11301871074661447, 0.1..."
3,GradientBoostingRegressor,"[-5.28858798254462, -4.996691501572245, -6.227...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.959163202914668, -6.25265259387373, -5.82...","[-5.042423912002789, -6.297024362993231, -6.17...","[0.2370749711370446, 0.08442699095533293, 0.24..."
4,AdaBoostRegressor,"[-5.277647058823528, -5.12625, -6.034097504818...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.995251938454544, -6.0982810118333335, -5....","[-5.0618787115361465, -6.087098150100001, -5.6...","[0.2830532530581922, 0.1663224546398239, 0.116..."
5,XGBRegressor,"[-5.295148, -4.951231, -6.2983737, -5.4252114,...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.822855, -6.249848, -5.3624716, -5.2904925...","[-5.3945694, -6.0763216, -5.814916, -5.1703424...","[0.36359614, 0.34777468, 0.347572, 0.2399047, ..."
6,ExtraTreesRegressor,"[-5.2950000000000035, -4.949999999999992, -6.2...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.066300000000008, -6.25, -5.81356666666666...","[-5.1409244060760075, -6.2317021449601455, -6....","[0.11929513927234087, 0.03659571007970968, 0.1..."
7,LinearRegression,"[-5.252912970184574, -5.161764705882351, -5.63...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.169372094785525, -5.562052953509072, -7.1...","[-4.946880839571998, -5.3083680303174585, -7.0...","[0.4448646677493685, 0.4529054955506591, 0.173..."
8,KNeighborsRegressor,"[-5.613333333333333, -5.3999999999999995, -5.3...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333333, -5.6...","[-5.666, -5.666, -5.6259999999999994, -5.41891...","[0.14081350945290877, 0.14081350945290877, 0.1..."
9,SVR,"[-5.520274391918902, -5.928446422804116, -5.93...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.607358782186577, -6.011235324646328, -5.5...","[-5.6185754733062545, -5.9067575837563195, -5....","[0.18260829407234908, 0.18761334461891915, 0.1..."


In [89]:
df_SubstructureCount_fp.to_csv('results/Fingerprints/Results_Substructure_Count_fp_MDCK.csv')
pred_df.to_csv('results/Fingerprints/Prediction_data_Substructure_Count_fp_MDCK.csv')

In [90]:
#Descriptors models
#2d RDKit descriptors
df_train = pd.read_csv('features/Descriptors/Train_2d_RDKit_des_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_RDKit_des_MDCK.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models_2drdkit = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models_2drdkit, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 217)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 217)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.079398 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 335
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 30
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits w

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


-0.8013169678785474


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.4257,0.5454,0.6524,0.1910,0.4398,0.3893,0.5405,0.6220,0.7352,0.2442,0.5818,0.5269
DecisionTreeRegressor,0.5794,0.5673,0.7612,-0.1013,0.3326,0.4084,0.5772,0.5917,0.7597,0.1929,0.4996,0.4160
RandomForestRegressor,0.3926,0.5160,0.6266,0.2539,0.5149,0.5363,0.4839,0.5994,0.6957,0.3233,0.5902,0.4821
GradientBoostingRegressor,0.4506,0.5101,0.6713,0.1436,0.4645,0.5062,0.4464,0.5131,0.6682,0.3757,0.6253,0.4490
AdaBoostRegressor,0.4021,0.4903,0.6341,0.2359,0.5441,0.4896,0.4590,0.5305,0.6775,0.3581,0.6076,0.4490
XGBRegressor,0.4450,0.5353,0.6671,0.1542,0.4872,0.5157,0.4429,0.5244,0.6655,0.3806,0.6300,0.5096
ExtraTreesRegressor,0.3478,0.4762,0.5898,0.3389,0.6321,0.5128,0.4259,0.5168,0.6526,0.4044,0.6500,0.5179
LinearRegression,2.4637,1.0022,1.5696,-3.6824,0.2721,0.3539,1.9701,1.0254,1.4036,-1.7549,0.2981,0.4160
KNeighborsRegressor,0.4006,0.5178,0.6329,0.2386,0.5071,0.4931,0.4219,0.4874,0.6496,0.4100,0.6734,0.5421
SVR,0.4159,0.4985,0.6449,0.2095,0.4634,0.5206,0.4562,0.4984,0.6754,0.3620,0.6258,0.5289


In [91]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.307959962769381, -5.818827806306796, -5.94...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.791917339848148, -6.021988150243992, -5.3...","[-5.578755891162001, -5.566878414112298, -5.35...","[0.30228197658420997, 0.3270853870402451, 0.05..."
1,DecisionTreeRegressor,"[-5.65, -4.58, -6.05, -5.402304814, -5.4023048...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.65, -6.25, -6.52, -5.289036881, -5.185752...","[-5.218, -5.718000000000001, -6.6139258908, -5...","[0.40236302016959763, 0.71859306982464, 0.9518..."
2,RandomForestRegressor,"[-5.246136974989996, -4.926406108258567, -5.87...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.51913061782, -5.667086067920001, -5.91947...","[-5.441167758035667, -5.607370364577568, -5.86...","[0.15384383640249608, 0.198930269835132, 0.052..."
3,GradientBoostingRegressor,"[-5.176637238136932, -4.707441190963438, -6.19...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.672811264987768, -5.785202480872903, -6.3...","[-5.415753094877516, -5.6134165249118535, -6.1...","[0.20980982962624609, 0.5663348554599393, 0.51..."
4,AdaBoostRegressor,"[-5.114000000000001, -4.68, -6.178945499000001...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.65, -5.91790088, -6.2, -5.537003026333333...","[-5.383702699491343, -5.6571747343412095, -5.9...","[0.2577023120846628, 0.3635523592727579, 0.201..."
5,XGBRegressor,"[-4.9624715, -4.5821066, -6.296578, -6.001588,...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.067129, -5.61362, -5.9536376, -5.316198, ...","[-5.3146544, -5.6518216, -6.224606, -5.544085,...","[0.30879638, 0.47676566, 0.35698715, 0.3138110..."
6,ExtraTreesRegressor,"[-5.1920999999999955, -4.662899999999996, -6.2...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.445547274949998, -5.6529401118600004, -5....","[-5.544904028472835, -5.777751602004, -5.74432...","[0.15659897635846842, 0.42103781386816097, 0.1..."
7,LinearRegression,"[-4.0, -6.051164539166766, -5.564946461832513,...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.0, -9.82693768801926, -4.0, -5.1443258943...","[-8.19324912727839, -8.395685182777601, -4.0, ...","[2.4034109910004733, 2.249210085561705, 0.0, 0..."
8,KNeighborsRegressor,"[-5.366666666666667, -5.3999999999999995, -5.8...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333334, -5.6...","[-5.6259999999999994, -5.4126666666666665, -5....","[0.16096100286853474, 0.2143766156401708, 0.14..."
9,SVR,"[-5.479855366534142, -5.30688600261383, -5.802...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.640093577096563, -5.989400082355616, -5.7...","[-5.586382696918635, -5.7947055295000585, -5.6...","[0.07429611086592079, 0.2936410118158758, 0.09..."


In [92]:
result_df.to_csv('results/Descriptors/Results_2d_RDKit_desc_MDCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2d_RDKit_desc_MDCK.csv')

In [93]:
#2d Mordred descriptors
df_train = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc_MDCK.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = X_test.select_dtypes(include=['number'])
X_test = X_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models_2dM = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df , prediction_df= train_and_test_predict(models_2dM, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 1436)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 1436)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.142039 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4817
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 358
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spli

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3817,0.4985,0.6178,0.2746,0.5256,0.4980,0.4398,0.5388,0.6632,0.3850,0.6853,0.5344
DecisionTreeRegressor,0.5930,0.5453,0.7701,-0.1270,0.3936,0.3285,0.4594,0.4983,0.6778,0.3576,0.6001,0.5427
RandomForestRegressor,0.3190,0.4662,0.5648,0.3937,0.6323,0.5914,0.4238,0.5491,0.6510,0.4074,0.6698,0.3719
GradientBoostingRegressor,0.3516,0.4760,0.5929,0.3318,0.6027,0.5374,0.4506,0.5399,0.6713,0.3699,0.6149,0.5399
AdaBoostRegressor,0.2577,0.4007,0.5076,0.5103,0.7244,0.5764,0.4712,0.5509,0.6864,0.3412,0.5881,0.4380
XGBRegressor,0.3711,0.4853,0.6092,0.2947,0.5676,0.4486,0.4724,0.5485,0.6873,0.3394,0.5845,0.4766
ExtraTreesRegressor,0.2886,0.4302,0.5372,0.4515,0.6869,0.6068,0.4316,0.5261,0.6570,0.3964,0.6394,0.4711
LinearRegression,0.7382,0.6873,0.8592,-0.4029,0.5518,0.4772,0.5374,0.4869,0.7331,0.2485,0.5768,0.6667
KNeighborsRegressor,0.3760,0.4833,0.6131,0.2855,0.5561,0.5480,0.4782,0.5533,0.6915,0.3313,0.5909,0.3931
SVR,0.4131,0.5110,0.6427,0.2149,0.4704,0.4998,0.4438,0.5036,0.6662,0.3794,0.6465,0.5179


In [94]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.17886263740318, -5.324280272622234, -5.959...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.324280272622234, -5.565454977675508, -5.2...","[-5.463514987313761, -5.606022037089763, -5.62...","[0.17120230998091476, 0.209969041213214, 0.316..."
1,DecisionTreeRegressor,"[-5.65, -5.129011186, -6.05, -4.61, -7.6989700...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.94, -5.65, -4.73, -5.289036881, -5.289036...","[-5.3938298042, -5.937531791850001, -6.0037940...","[0.6476576221128268, 0.3474331287763371, 1.115..."
2,RandomForestRegressor,"[-5.466767864559994, -5.1515292490791635, -5.8...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.304215838076669, -5.672317194110001, -5.5...","[-5.455416572508567, -5.74477886685623, -5.722...","[0.1278672795856363, 0.13327937859291494, 0.12..."
3,GradientBoostingRegressor,"[-5.484052593747885, -5.108962580773248, -6.08...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.791490425941876, -5.715685405014332, -4.9...","[-5.444669586650762, -5.966016707127882, -5.67...","[0.4991493770762742, 0.18346008273777784, 0.49..."
4,AdaBoostRegressor,"[-5.442709469227274, -5.171502284857143, -6.16...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.94, -6.153408949333334, -5.57929238299999...","[-5.277345786152654, -5.847704197873334, -5.73...","[0.22632431585282567, 0.2899696615445823, 0.09..."
5,XGBRegressor,"[-5.6543465, -5.235243, -6.179201, -4.879936, ...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.9547057, -5.59727, -4.7449546, -5.274257,...","[-5.354905, -5.923278, -5.5835414, -5.5346403,...","[0.41577792, 0.23689225, 0.59295946, 0.2686962..."
6,ExtraTreesRegressor,"[-5.341697274949999, -5.045222769579996, -6.18...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.368390368809998, -6.07283745671, -5.74988...","[-5.4035675977765, -5.941354272586001, -5.8087...","[0.22366292972901994, 0.19614980250508335, 0.0..."
7,LinearRegression,"[-5.019448812414413, -4.324279281608511, -5.82...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.167643053307332, -6.888798643840394, -5.1...","[-5.85171079331073, -7.230166173004347, -6.278...","[0.8051183145404481, 0.32762335285088207, 0.84..."
8,KNeighborsRegressor,"[-5.066666666666666, -5.3999999999999995, -5.6...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333333, -5.2...","[-5.713333333333333, -5.713333333333333, -5.28...","[0.1707890186425604, 0.1707890186425604, 0.189..."
9,SVR,"[-5.23894164475384, -5.432387101021765, -5.746...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.399496564755412, -5.809743753490667, -5.6...","[-5.514092711467532, -5.8170951520194, -5.6967...","[0.21446299126490823, 0.22890930661758324, 0.1..."


In [95]:
result_df.to_csv('results/Descriptors/Results_2d_Mordred_desc_MDCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2d_Mordred_desc_MDCK.csv')

In [96]:
#Removal of constant columns
def remove_constant_columns(df):
    constant_columns = [col for col in df.columns if df[col].nunique() <= 1]
    
    df_cleaned = df.drop(columns=constant_columns)
    
    return df_cleaned, constant_columns

In [97]:
#Low variance column removal
def remove_low_variance_columns(df, threshold=0.005):
    variances = df.var()
    
    low_variance_columns = variances[variances < threshold].index.tolist()
    
    df_cleaned = df.drop(columns=low_variance_columns)
    
    return df_cleaned, low_variance_columns

In [98]:
#2d RDKit descriptors const removal
df_train = pd.read_csv('features/Descriptors/Train_2d_RDKit_des_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train, const_col = remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_RDKit_des_MDCK.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = X_test.drop(const_col,axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 134)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 134)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.115921 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 335
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 30
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits w

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

0.20191327562611572


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.4257,0.5454,0.6524,0.1910,0.4398,0.3893,0.5405,0.6220,0.7352,0.2442,0.5818,0.5269
DecisionTreeRegressor,0.5559,0.5400,0.7456,-0.0566,0.3964,0.4916,0.4805,0.5397,0.6932,0.3281,0.5834,0.4904
RandomForestRegressor,0.3816,0.5080,0.6177,0.2748,0.5308,0.5449,0.4726,0.5908,0.6875,0.3391,0.6058,0.4821
GradientBoostingRegressor,0.4368,0.5039,0.6609,0.1699,0.4827,0.5162,0.4437,0.5191,0.6661,0.3795,0.6315,0.4876
AdaBoostRegressor,0.3503,0.4746,0.5919,0.3342,0.6122,0.5445,0.4362,0.5270,0.6605,0.3900,0.6409,0.4050
XGBRegressor,0.4450,0.5353,0.6671,0.1542,0.4872,0.5157,0.4429,0.5244,0.6655,0.3806,0.6300,0.5096
ExtraTreesRegressor,0.3291,0.4638,0.5737,0.3746,0.6539,0.5209,0.4115,0.5027,0.6415,0.4246,0.6695,0.5179
LinearRegression,2.4637,1.0022,1.5696,-3.6824,0.2721,0.3539,1.9701,1.0254,1.4036,-1.7549,0.2981,0.4160
KNeighborsRegressor,0.4006,0.5178,0.6329,0.2386,0.5071,0.4931,0.4219,0.4874,0.6496,0.4100,0.6734,0.5421
SVR,0.4159,0.4985,0.6449,0.2096,0.4634,0.5206,0.4562,0.4984,0.6754,0.3621,0.6258,0.5289


In [99]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.307959962769381, -5.818827806306796, -5.94...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.791917339848148, -6.021988150243992, -5.3...","[-5.578755891162001, -5.566878414112298, -5.35...","[0.30228197658420997, 0.3270853870402451, 0.05..."
1,DecisionTreeRegressor,"[-6.25, -4.58, -6.05, -5.402304814, -5.4023048...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.65, -6.25, -6.52, -5.402304814, -5.185752...","[-5.218, -5.644, -6.473485317800001, -5.513883...","[0.40236302016959763, 0.7422021288031985, 0.63..."
2,RandomForestRegressor,"[-5.211447274949995, -4.889380480669995, -5.88...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.530740376142498, -5.666993545977142, -5.9...","[-5.485253108458834, -5.620825601750191, -5.86...","[0.1476351661624793, 0.22226559190448222, 0.04..."
3,GradientBoostingRegressor,"[-5.2306491920792855, -4.690910911111456, -6.1...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.752723652549667, -5.816676422111794, -6.3...","[-5.425299004497271, -5.6191188418783815, -6.2...","[0.2228417760219292, 0.5620773448586596, 0.418..."
4,AdaBoostRegressor,"[-4.970000000000001, -4.726153846153846, -6.21...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.650000000000001, -5.99896485152381, -5.81...","[-5.49229294264504, -5.684832192438249, -5.801...","[0.17269423272410173, 0.3448377966725527, 0.14..."
5,XGBRegressor,"[-4.9624715, -4.5821066, -6.296578, -6.001588,...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.067129, -5.61362, -5.9536376, -5.316198, ...","[-5.3146544, -5.6518216, -6.224606, -5.544085,...","[0.30879638, 0.47676566, 0.35698715, 0.3138110..."
6,ExtraTreesRegressor,"[-5.1958999999999955, -4.708199999999995, -6.2...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.443199999999997, -5.71290479899, -5.68905...","[-5.546349271825334, -5.765792742915001, -5.75...","[0.22332279737318722, 0.40134138461115876, 0.1..."
7,LinearRegression,"[-4.0, -6.0511645391666775, -5.564946461832529...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.0, -9.826937688019246, -4.0, -5.144325894...","[-8.193249127278452, -8.395685182777594, -4.0,...","[2.4034109910004413, 2.249210085561706, 0.0, 0..."
8,KNeighborsRegressor,"[-5.366666666666667, -5.3999999999999995, -5.8...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333334, -5.6...","[-5.6259999999999994, -5.4126666666666665, -5....","[0.16096100286853474, 0.2143766156401708, 0.14..."
9,SVR,"[-5.479902271774093, -5.306846299287493, -5.80...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.640109164641454, -5.989485941808115, -5.7...","[-5.58639310215753, -5.794767258143841, -5.657...","[0.0743194634285623, 0.29360012358975035, 0.09..."


In [100]:
result_df.to_csv('results/Descriptors/Results_2d_rdkit_const_rem_MDCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2d_rdkit_const_rem_MDCK.csv')

In [101]:
#2d Mordred descriptors const removal
df_train = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train, const_col = remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc_MDCK.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = X_test.select_dtypes(include=['number'])
X_test = X_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models_2dM = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models_2dM, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 1148)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 1148)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.141294 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4817
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 358
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spli

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3817,0.4985,0.6178,0.2746,0.5256,0.4980,0.4398,0.5388,0.6632,0.3850,0.6853,0.5344
DecisionTreeRegressor,0.5060,0.5315,0.7114,0.0383,0.4671,0.4396,0.4587,0.5102,0.6773,0.3586,0.6016,0.4711
RandomForestRegressor,0.3332,0.4791,0.5772,0.3668,0.6122,0.5728,0.4266,0.5518,0.6532,0.4034,0.6662,0.4105
GradientBoostingRegressor,0.3624,0.4839,0.6020,0.3112,0.5860,0.5260,0.4569,0.5408,0.6760,0.3611,0.6059,0.5620
AdaBoostRegressor,0.2681,0.4101,0.5178,0.4904,0.7115,0.5712,0.4358,0.5311,0.6602,0.3905,0.6382,0.4766
XGBRegressor,0.3711,0.4853,0.6092,0.2947,0.5676,0.4486,0.4724,0.5485,0.6873,0.3394,0.5845,0.4766
ExtraTreesRegressor,0.2739,0.4214,0.5233,0.4795,0.7038,0.6176,0.4474,0.5297,0.6689,0.3744,0.6180,0.4683
LinearRegression,0.7382,0.6873,0.8592,-0.4029,0.5518,0.4772,0.5374,0.4869,0.7331,0.2485,0.5768,0.6667
KNeighborsRegressor,0.3760,0.4833,0.6131,0.2855,0.5561,0.5480,0.4782,0.5533,0.6915,0.3313,0.5909,0.3931
SVR,0.4131,0.5110,0.6427,0.2150,0.4705,0.4998,0.4438,0.5036,0.6662,0.3794,0.6465,0.5179


In [102]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.17886263740318, -5.324280272622234, -5.959...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.324280272622234, -5.565454977675508, -5.2...","[-5.463514987313761, -5.606022037089763, -5.62...","[0.17120230998091476, 0.209969041213214, 0.316..."
1,DecisionTreeRegressor,"[-5.65, -5.32, -6.05, -4.82, -7.698970004, -5....",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.1, -5.22, -4.73, -5.289036881, -5.2890368...","[-5.3340000000000005, -5.76507082905, -5.87362...","[0.550221773469571, 0.5543757151643797, 1.1324..."
2,RandomForestRegressor,"[-5.474887910279993, -5.138096703869998, -5.80...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.406737791392502, -5.6968631189200005, -5....","[-5.425112286706917, -5.720537606116164, -5.72...","[0.09695153868413703, 0.12495415878796373, 0.1..."
3,GradientBoostingRegressor,"[-5.362081074446435, -5.117146932417445, -6.14...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.779903439380446, -5.754939425753073, -4.9...","[-5.445897498140928, -5.95917586922461, -5.624...","[0.4534723999021456, 0.20005537714788524, 0.44..."
4,AdaBoostRegressor,"[-5.506666666666667, -5.2260266828, -6.2555807...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.632357051333333, -6.167581861444446, -5.6...","[-5.495881584291746, -5.8692575035246035, -5.7...","[0.14032708964291252, 0.23396063677169335, 0.0..."
5,XGBRegressor,"[-5.6543465, -5.235243, -6.179201, -4.879936, ...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.9547057, -5.59727, -4.7449546, -5.274257,...","[-5.354905, -5.923278, -5.5835414, -5.5346403,...","[0.41577792, 0.23689225, 0.59295946, 0.2686962..."
6,ExtraTreesRegressor,"[-5.470941824849999, -5.030160384079995, -6.16...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.272497892850001, -6.039608220880001, -5.7...","[-5.3934379848226675, -5.924529274493336, -5.7...","[0.20766864782376823, 0.21048895810605306, 0.1..."
7,LinearRegression,"[-5.019448812414435, -4.324279281608504, -5.82...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.16764305330736, -6.888798643840396, -5.15...","[-5.851710793310732, -7.230166173004347, -6.27...","[0.8051183145404477, 0.3276233528508872, 0.849..."
8,KNeighborsRegressor,"[-5.066666666666666, -5.3999999999999995, -5.6...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333333, -5.2...","[-5.713333333333333, -5.713333333333333, -5.28...","[0.1707890186425604, 0.1707890186425604, 0.189..."
9,SVR,"[-5.238965434995149, -5.432388144098546, -5.74...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.399535581206924, -5.809791329400151, -5.6...","[-5.514118224923472, -5.8171420173771775, -5.6...","[0.21446247550635486, 0.2289374938891217, 0.14..."


In [103]:
result_df.to_csv('results/Descriptors/Results_2d_Mordred_const_rem_MDCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_df_2d_Mordred_const_rem_MDCK.csv')

In [104]:
#2d RDKit descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_2d_RDKit_des_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train, const_col = remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_RDKit_des_MDCK.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = X_test.drop(const_col,axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models_LVR_rdkit = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models_LVR_rdkit, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 123)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 123)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.238375 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 284
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 25
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits w

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

0.13923232505404715


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.4313,0.5454,0.6567,0.1804,0.4263,0.3701,0.5485,0.6238,0.7406,0.2330,0.5655,0.4910
DecisionTreeRegressor,0.5578,0.5587,0.7469,-0.0602,0.4193,0.3700,0.4829,0.5614,0.6949,0.3247,0.6208,0.5234
RandomForestRegressor,0.3615,0.4979,0.6013,0.3129,0.5667,0.5596,0.4569,0.5781,0.6759,0.3611,0.6295,0.3857
GradientBoostingRegressor,0.3966,0.4862,0.6298,0.2463,0.5499,0.5252,0.4352,0.5251,0.6597,0.3914,0.6387,0.4683
AdaBoostRegressor,0.3355,0.4643,0.5792,0.3623,0.6463,0.5064,0.4528,0.5446,0.6729,0.3668,0.6215,0.5069
XGBRegressor,0.3992,0.5073,0.6318,0.2413,0.5661,0.5414,0.4570,0.5212,0.6760,0.3610,0.6096,0.4435
ExtraTreesRegressor,0.3476,0.4755,0.5896,0.3394,0.6363,0.5146,0.4256,0.5181,0.6524,0.4049,0.6492,0.5234
LinearRegression,2.7997,1.2105,1.6732,-4.3210,0.2958,0.3111,0.9374,0.7506,0.9682,-0.3108,0.4543,0.4077
KNeighborsRegressor,0.3834,0.5028,0.6192,0.2714,0.5324,0.5248,0.4318,0.5140,0.6571,0.3962,0.6578,0.4897
SVR,0.4097,0.4963,0.6401,0.2214,0.4756,0.5347,0.4573,0.4982,0.6762,0.3605,0.6237,0.5289


In [105]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.307959962769381, -5.818827806306796, -5.94...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.791917339848148, -6.021988150243992, -5.3...","[-5.549460480945126, -5.635215090891424, -5.31...","[0.31478498655707077, 0.3139781164956814, 0.03..."
1,DecisionTreeRegressor,"[-4.94, -4.58, -6.05, -5.289036881, -5.2890368...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.25, -6.25, -6.4, -5.289036881, -5.1857524...","[-5.868, -5.922, -6.8922739882, -5.3116904676,...","[0.5189373757978893, 0.6559999999999998, 0.604..."
2,RandomForestRegressor,"[-5.180999999999996, -4.936690368809996, -5.87...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.689014453925, -5.737497002036666, -5.8331...","[-5.601091263048099, -5.619375217532669, -5.76...","[0.1612565448870488, 0.2523125767343236, 0.055..."
3,GradientBoostingRegressor,"[-5.1085271697564405, -4.742046466248045, -6.2...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.525753567735792, -5.785115113642161, -5.7...","[-5.62775042944767, -5.850028525626195, -5.696...","[0.37161333277994624, 0.4888724901033585, 0.17..."
4,AdaBoostRegressor,"[-5.1145184405, -4.71375, -6.204205999133333, ...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.8500000000000005, -6.2125, -5.615, -5.373...","[-5.664442610562223, -5.852818727219048, -5.65...","[0.2233835086022917, 0.4558529482830521, 0.044..."
5,XGBRegressor,"[-4.9275684, -4.587584, -6.2463517, -6.1196485...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.364961, -5.915705, -5.794659, -5.3418407,...","[-5.639713, -5.639601, -5.8067174, -5.3585157,...","[0.35327363, 0.49095893, 0.24687527, 0.0594262..."
6,ExtraTreesRegressor,"[-5.042949999999996, -4.689299999999998, -6.24...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.429397274949997, -5.9742972749500005, -5....","[-5.538005432433334, -5.859848909980002, -5.71...","[0.23498097355453915, 0.45629620812770516, 0.1..."
7,LinearRegression,"[-6.265337269228257, -5.115136785362917, -5.55...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-10.0, -7.224147192637322, -4.0, -4.0, -4.0,...","[-6.4, -7.364384520452465, -5.138319881149852,...","[2.939387691339814, 2.3812851728464883, 2.2766..."
8,KNeighborsRegressor,"[-5.366666666666667, -5.3999999999999995, -5.8...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333334, -5.6...","[-5.6259999999999994, -5.6546666666666665, -5....","[0.16096100286853474, 0.14230796026770784, 0.1..."
9,SVR,"[-5.4800906630949315, -5.304889709493807, -5.7...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.634665544856663, -6.011635524672326, -5.7...","[-5.585547844628776, -5.812760499176273, -5.66...","[0.07845705728508723, 0.29554695836343137, 0.0..."


In [106]:
result_df.to_csv('results/Descriptors/Results_2d_rdkit_LVR_MDCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2d_rdkit_LVR_MDCK.csv')

In [107]:
#2d Mordred descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train, const_col = remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc_MDCK.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = X_test.select_dtypes(include=['number'])
X_test = X_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
results_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
results_df

X_train shape:  (51, 769)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 769)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.155882 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3144
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 234
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3812,0.5080,0.6174,0.2755,0.5292,0.4528,0.4465,0.5559,0.6682,0.3756,0.6943,0.5289
DecisionTreeRegressor,0.5592,0.5375,0.7478,-0.0628,0.4222,0.3651,0.4604,0.5294,0.6785,0.3562,0.6113,0.5069
RandomForestRegressor,0.3462,0.5012,0.5884,0.3421,0.5973,0.5765,0.4480,0.5723,0.6693,0.3736,0.6464,0.4711
GradientBoostingRegressor,0.3641,0.4756,0.6034,0.3080,0.6140,0.4804,0.4180,0.5070,0.6465,0.4156,0.6688,0.6309
AdaBoostRegressor,0.2815,0.4132,0.5306,0.4650,0.6852,0.6150,0.4486,0.5308,0.6698,0.3727,0.6269,0.5041
XGBRegressor,0.3266,0.4587,0.5715,0.3793,0.6311,0.5505,0.4142,0.5109,0.6436,0.4208,0.6645,0.6253
ExtraTreesRegressor,0.2794,0.4255,0.5286,0.4690,0.7002,0.5894,0.4407,0.5224,0.6638,0.3838,0.6287,0.5041
LinearRegression,0.7700,0.6871,0.8775,-0.4635,0.4850,0.4387,0.6326,0.5449,0.7954,0.1154,0.5190,0.6309
KNeighborsRegressor,0.3662,0.4760,0.6052,0.3040,0.5683,0.5653,0.4787,0.5565,0.6919,0.3306,0.5897,0.4041
SVR,0.4355,0.5235,0.6599,0.1724,0.4258,0.4623,0.4358,0.4972,0.6601,0.3906,0.6629,0.4959


In [108]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.193297965477244, -5.535778042677406, -5.62...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.280975908332179, -5.606666342554522, -6.3...","[-5.528472818793358, -5.680090552649182, -5.97...","[0.28028616079275587, 0.27291249589694644, 0.2..."
1,DecisionTreeRegressor,"[-5.65, -4.58, -6.05, -5.585026652, -7.6989700...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.25, -6.25, -4.94, -5.289036881, -5.744727...","[-5.570664237600001, -6.14, -6.133862362066667...","[0.4970918463688163, 0.21999999999999995, 0.95..."
2,RandomForestRegressor,"[-5.325624190706663, -5.274248180756664, -5.70...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.36376814138, -5.772133109936667, -5.85114...","[-5.508542976337802, -5.82187066650373, -5.984...","[0.19850355298353264, 0.10605269965327546, 0.1..."
3,GradientBoostingRegressor,"[-5.426738791175946, -5.191776552007337, -6.12...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.750836818567971, -6.141153567811248, -5.1...","[-5.665671380757834, -6.064971690968206, -6.11...","[0.41722694128157994, 0.2205254814248703, 0.52..."
4,AdaBoostRegressor,"[-5.478479742857142, -5.176937723, -6.05, -5.5...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.43, -6.167171840466667, -6.16666666666666...","[-5.5433636120698715, -5.9175647781937775, -5....","[0.15755476273489138, 0.2568651008868674, 0.18..."
5,XGBRegressor,"[-5.6326013, -4.892328, -6.080672, -5.5425777,...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.6588135, -6.2037945, -5.676158, -5.292030...","[-5.678637, -6.0517535, -6.0734735, -5.391955,...","[0.25548628, 0.12108629, 0.2763313, 0.08657554..."
6,ExtraTreesRegressor,"[-5.250247274949998, -5.1194398066124975, -6.1...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.233, -6.09700298081, -5.771580165509997, ...","[-5.343058529292, -5.9623128754705, -6.0044270...","[0.2104420631925955, 0.24478552004043644, 0.14..."
7,LinearRegression,"[-4.668036376462836, -4.404872747507602, -5.51...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.041673810051752, -7.12433672498913, -4.91...","[-6.118782598449753, -7.508590138998676, -5.58...","[1.4713478875919963, 0.5883045669646025, 0.985..."
8,KNeighborsRegressor,"[-5.066666666666666, -5.3999999999999995, -5.8...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333333, -5.2...","[-5.713333333333333, -5.713333333333333, -5.28...","[0.1707890186425604, 0.1707890186425604, 0.189..."
9,SVR,"[-5.246082162705119, -5.441158260586537, -5.75...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.466069202459343, -5.813382637340638, -5.6...","[-5.561158120264895, -5.795121970448821, -5.72...","[0.1875412301617725, 0.24211710644440199, 0.16..."


In [109]:
results_df.to_csv('results/Descriptors/Results_2d_Mordred_LVR_MDCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2d_Mordred_LVR_MDCK.csv')

In [110]:
#2d Padel descriptors
df_train = pd.read_csv('features/Descriptors/Train_2d_padel_MDCK.csv')
df_train['ID'] = df_train['Name'].str.extract(r'_(\d+)$')
df_train['ID'] = df_train['ID'].astype(int)
df_train = df_train.drop('Name',axis=1)
df_train = df_train.fillna(0)
df_train

,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,nAtom,nHeavyAtom,nH,...,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb,ID
0,0,-6.7922,46.133981,200.8102,123.721166,0,0,118,56,62,...,107.350908,1.916981,55.593344,24.924836,30.668508,13190.0,104.0,-0.452,276.0,1109
1,0,-1.2947,1.676248,186.2062,119.101580,0,0,111,51,60,...,100.950171,1.979415,36.645624,17.922134,18.723489,9286.0,80.0,6.411,254.0,1017
2,0,-0.9878,0.975749,203.2963,128.382338,0,0,120,54,66,...,106.401998,1.970407,37.167232,17.862301,19.304931,10372.0,92.0,5.550,272.0,1018
3,0,-2.1290,4.532641,277.7356,179.405714,0,0,172,74,98,...,142.514887,1.925877,56.524332,25.255908,31.268424,25829.0,128.0,8.158,360.0,1107
4,0,-8.4202,70.899768,189.3712,121.053994,0,0,114,56,58,...,109.785075,1.960448,56.482950,25.157078,31.325873,13235.0,98.0,-1.338,284.0,1112
5,0,-0.7316,0.535239,184.6007,118.299580,0,0,110,50,60,...,99.092833,1.981857,34.121681,15.400697,18.720983,8739.0,78.0,7.547,248.0,1840
6,0,-5.3114,28.210970,240.6548,158.177026,0,0,150,68,82,...,133.232246,1.959298,57.127492,25.386644,31.740847,21211.0,114.0,4.402,340.0,1111
7,0,-0.7316,0.535239,184.6007,118.299580,0,0,110,50,60,...,99.092833,1.981857,34.121681,15.400697,18.720983,8739.0,78.0,7.547,248.0,1844
8,0,-3.7570,14.115049,266.2966,176.738542,0,0,168,74,94,...,144.956003,1.958865,57.417652,25.489873,31.927779,25892.0,122.0,7.272,368.0,1110
9,0,-0.2899,0.084042,200.4822,122.245994,0,0,111,53,58,...,106.152092,2.002870,34.184233,15.422362,18.761870,10243.0,83.0,7.361,266.0,1845


In [111]:
df = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc_MDCK.csv')
df 


,ID,SMILES,Permeability,ABCIndex,ABCGGIndex,AcidicGroupCount,BasicGroupCount,AdjacencyMatrix,AdjacencyMatrix.1,AdjacencyMatrix.2,...,WalkCount.19,WalkCount.20,Weight,Weight.1,WienerIndex,WienerIndex.1,ZagrebIndex,ZagrebIndex.1,ZagrebIndex.2,ZagrebIndex.3
0,1114,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-4.940000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,101.372024,2.447751,4.856077,...,11.210644,138.066274,1138.715439,6.469974,33296,140,416.0,486.0,34.055556,18.111111
1,1113,CC(C)C[C@@H]1NC(=O)[C@@H]2CCCN2C(=O)[C@@H](Cc2...,-5.820000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,101.370874,2.447978,4.860529,...,11.210644,138.066274,1138.715439,6.469974,33296,140,416.0,486.0,34.055556,18.111111
2,1117,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H]2CCCN...,-5.650000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,101.371868,2.443590,4.850728,...,11.210644,138.066274,1138.715439,6.469974,33296,140,416.0,486.0,34.055556,18.111111
3,1119,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-6.250000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,97.295643,2.408298,4.816595,...,11.141644,120.163455,1114.715439,6.406411,31648,138,396.0,458.0,36.555556,17.777778
4,2428,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N(C)[C@@H](C)C(=...,-5.510000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,96.774478,2.436865,4.825853,...,11.126159,133.738666,1090.679054,6.492137,29794,130,394.0,458.0,31.222222,17.333333
5,2446,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N2CCC[C@@H]2C(=O...,-6.400000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,96.685868,2.442093,4.836451,...,11.129393,133.743086,1090.679054,6.492137,29704,130,394.0,458.0,31.222222,17.333333
6,2445,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N2CCC[C@@H]2C(=O...,-4.820000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,96.685868,2.442093,4.836451,...,11.129393,133.743086,1088.699789,6.404116,29704,130,394.0,458.0,31.222222,17.333333
7,2427,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N(C)[C@@H](C)C(=...,-4.610000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,96.774478,2.436865,4.825853,...,11.126159,133.738666,1088.699789,6.404116,29794,130,394.0,458.0,31.222222,17.333333
8,8145,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.744727,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,89.136837,2.437073,4.831019,...,11.025019,126.223556,1047.640226,6.466915,25439,124,358.0,415.0,31.750000,16.611111
9,1107,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](C)N...,-4.610000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,86.880063,2.403063,4.806126,...,11.046579,113.682027,1046.746739,6.085737,25829,128,360.0,414.0,38.055556,16.444444


In [112]:
merged_df = df_train.merge(df[['ID', 'SMILES', 'Permeability']], on='ID', how='left')
merged_df = merged_df[['ID', 'SMILES', 'Permeability'] + [col for col in merged_df.columns if col not in ['ID', 'SMILES', 'Permeability']]]
merged_df

,ID,SMILES,Permeability,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,...,AMW,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb
0,1109,C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](C)N(C)C(...,-6.120000,0,-6.7922,46.133981,200.8102,123.721166,0,0,...,6.732755,107.350908,1.916981,55.593344,24.924836,30.668508,13190.0,104.0,-0.452,276.0
1,1017,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)NC(=O)[C...,-5.700000,0,-1.2947,1.676248,186.2062,119.101580,0,0,...,6.418490,100.950171,1.979415,36.645624,17.922134,18.723489,9286.0,80.0,6.411,254.0
2,1018,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...,-4.960000,0,-0.9878,0.975749,203.2963,128.382338,0,0,...,6.287494,106.401998,1.970407,37.167232,17.862301,19.304931,10372.0,92.0,5.550,272.0
3,1107,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](C)N...,-4.610000,0,-2.1290,4.532641,277.7356,179.405714,0,0,...,6.085737,142.514887,1.925877,56.524332,25.255908,31.268424,25829.0,128.0,8.158,360.0
4,1112,C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](C)NC(=O)[...,-6.170000,0,-8.4202,70.899768,189.3712,121.053994,0,0,...,6.933629,109.785075,1.960448,56.482950,25.157078,31.325873,13235.0,98.0,-1.338,284.0
5,1840,CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](C...,-4.480000,0,-0.7316,0.535239,184.6007,118.299580,0,0,...,6.331431,99.092833,1.981857,34.121681,15.400697,18.720983,8739.0,78.0,7.547,248.0
6,1111,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C...,-6.220000,0,-5.3114,28.210970,240.6548,158.177026,0,0,...,6.390810,133.232246,1.959298,57.127492,25.386644,31.740847,21211.0,114.0,4.402,340.0
7,1844,CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](C...,-5.100000,0,-0.7316,0.535239,184.6007,118.299580,0,0,...,6.331431,99.092833,1.981857,34.121681,15.400697,18.720983,8739.0,78.0,7.547,248.0
8,1110,CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[C@H]...,-4.730000,0,-3.7570,14.115049,266.2966,176.738542,0,0,...,6.206640,144.956003,1.958865,57.417652,25.489873,31.927779,25892.0,122.0,7.272,368.0
9,1845,CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](C...,-4.960000,0,-0.2899,0.084042,200.4822,122.245994,0,0,...,6.580557,106.152092,2.002870,34.184233,15.422362,18.761870,10243.0,83.0,7.361,266.0


In [113]:
df_ordered = merged_df.merge(df[['ID']], on='ID', how='right')
df_ordered = df_ordered.reindex(df.index)
df_ordered

,ID,SMILES,Permeability,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,...,AMW,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb
0,1114,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-4.940000,0,-2.6816,7.190979,307.3786,190.818542,0,0,...,6.469974,162.717774,1.984363,57.439206,25.502169,31.937037,33296.0,140.0,7.952,416.0
1,1113,CC(C)C[C@@H]1NC(=O)[C@@H]2CCCN2C(=O)[C@@H](Cc2...,-5.820000,0,-2.6816,7.190979,307.3786,190.818542,0,0,...,6.469974,162.720191,1.984393,57.435556,25.501133,31.934422,33296.0,140.0,7.952,416.0
2,1117,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H]2CCCN...,-5.650000,0,-2.6816,7.190979,307.3786,190.818542,0,0,...,6.469974,162.717400,1.984359,57.440346,25.502649,31.937697,33296.0,140.0,7.952,416.0
3,1119,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-6.250000,0,-1.2456,1.551519,309.4986,187.298542,0,0,...,6.406411,156.626458,1.957831,56.630904,25.293944,31.336960,31648.0,138.0,7.786,396.0
4,2428,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N(C)[C@@H](C)C(=...,-5.510000,0,-5.7230,32.752729,279.5151,180.153370,0,0,...,6.492137,155.242276,1.990286,60.716736,28.525011,32.191725,29794.0,130.0,5.187,394.0
5,2446,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N2CCC[C@@H]2C(=O...,-6.400000,0,-5.7230,32.752729,279.5151,180.153370,0,0,...,6.492137,155.232056,1.990155,60.685463,28.512533,32.172930,29704.0,130.0,5.187,394.0
6,2445,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N2CCC[C@@H]2C(=O...,-4.820000,0,-5.3689,28.825087,280.2868,182.444956,0,0,...,6.404116,155.232056,1.990155,57.684402,25.511472,32.172930,29704.0,130.0,7.084,394.0
7,2427,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N(C)[C@@H](C)C(=...,-4.610000,0,-5.3689,28.825087,280.2868,182.444956,0,0,...,6.404116,155.242276,1.990286,57.714858,25.523133,32.191725,29794.0,130.0,7.084,394.0
8,8145,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.744727,0,-6.0254,36.305445,258.2126,172.486577,0,0,...,6.466915,143.466261,1.965291,59.644932,28.221894,28.405281,25439.0,124.0,7.222,358.0
9,1107,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](C)N...,-4.610000,0,-2.1290,4.532641,277.7356,179.405714,0,0,...,6.085737,142.514887,1.925877,56.524332,25.255908,31.268424,25829.0,128.0,8.158,360.0


In [114]:
df_ordered.to_csv('features/Descriptors/Train_2d_padel_curated_MDCK.csv', index=False)

In [115]:
#2d test padel descriptors
df_test = pd.read_csv('features/Descriptors/Test_2d_padel_MDCK.csv')
df_test['ID'] = df_test['Name'].str.extract(r'_(\d+)$')
df_test['ID'] = df_test['ID'].astype(int)
df_test = df_test.drop('Name',axis=1)
df_test = df_test.fillna(0)
df_test

,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,nAtom,nHeavyAtom,nH,...,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb,ID
0,0,-3.1134,9.693260,255.8998,162.779405,0,0,154,69,85,...,133.378413,1.933020,59.341751,28.115454,28.210938,22236.0,115.0,5.397,330.0,8168
1,0,-3.6834,13.567436,252.0938,160.844198,0,0,154,68,86,...,130.791399,1.923403,56.234345,25.152766,31.081579,21154.0,120.0,5.288,332.0,1108
2,0,-1.2217,1.492551,199.1082,122.245994,0,0,111,53,58,...,106.143859,2.002714,34.176054,15.419528,18.756526,10195.0,85.0,7.150,266.0,1843
3,0,-1.6634,2.766900,183.2267,118.299580,0,0,110,50,60,...,99.084602,1.981692,34.113502,15.397863,18.715639,8694.0,80.0,7.336,248.0,1842
4,0,-5.0816,25.822659,254.9508,167.632991,0,0,158,71,87,...,138.417801,1.949546,59.630333,28.216709,28.395866,23619.0,120.0,6.424,344.0,8143
5,0,-4.9290,24.295041,252.2939,164.539405,0,0,155,70,85,...,136.601049,1.951444,59.784728,28.235280,28.531690,22971.0,116.0,5.752,338.0,8119
6,0,-4.0401,16.322408,244.1556,163.399405,0,0,155,70,85,...,137.310770,1.961582,56.609941,28.393162,28.216778,22212.0,114.0,7.691,348.0,6423
7,0,-3.9378,15.506269,249.8523,166.492991,0,0,158,71,87,...,139.131996,1.959606,56.808725,28.380436,28.428289,22927.0,118.0,7.404,354.0,6496
8,0,-3.1134,9.693260,255.8998,162.779405,0,0,154,69,85,...,133.381424,1.933064,59.345483,28.116662,28.212683,22278.0,115.0,5.397,330.0,8345
9,0,-5.5830,31.169889,254.9353,169.392991,0,0,159,72,87,...,141.646739,1.967316,59.761039,28.238710,28.504056,24767.0,120.0,6.761,352.0,8133


In [116]:
df = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc_MDCK.csv')
df

,ID,SMILES,Permeability,ABCIndex,ABCGGIndex,AcidicGroupCount,BasicGroupCount,AdjacencyMatrix,AdjacencyMatrix.1,AdjacencyMatrix.2,...,WalkCount.19,WalkCount.20,Weight,Weight.1,WienerIndex,WienerIndex.1,ZagrebIndex,ZagrebIndex.1,ZagrebIndex.2,ZagrebIndex.3
0,1120,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-6.300000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,103.838268,2.420071,4.840143,...,11.200910,126.480968,1198.809340,6.243799,37307,146,424.0,488.0,39.277778,19.111111
1,1118,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](Cc2...,-5.350000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,97.302139,2.412394,4.824787,...,11.141934,120.163745,1114.715439,6.406411,31648,138,396.0,458.0,36.555556,17.777778
2,1121,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-6.200000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,95.621135,2.421653,4.790984,...,11.069213,133.614531,1082.652839,6.601542,30176,124,392.0,450.0,30.611111,17.222222
3,8133,CCC[C@@H]1NC(=O)CN(CC)C(=O)[C@H](CC(C)C)NC(=O)...,-5.355561,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,88.159241,2.431154,4.817857,...,10.985954,125.096326,1033.624576,6.500783,24767,120,352.0,406.0,30.888889,16.388889
4,8143,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.406714,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,85.381854,2.437654,4.830440,...,10.978917,124.007460,1021.624576,6.465978,23619,120,344.0,397.0,32.750000,16.194444
5,8119,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.073658,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,83.965708,2.427767,4.806183,...,10.937792,122.874218,1007.608926,6.500703,22971,116,338.0,388.0,31.888889,15.972222
6,6496,CC(=O)N1CCC[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N[...,-4.860000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,84.630944,2.429295,4.822808,...,11.018416,126.196276,1001.652505,6.339573,22927,118,354.0,409.0,32.472222,15.666667
7,8168,CCOC(=O)[C@@H]1CSCC(=O)N[C@@H](CC)C(=O)N(C)[C@...,-6.031517,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,81.585614,2.381030,4.762061,...,10.902997,108.140710,995.608926,6.464993,22236,115,330.0,376.0,33.750000,15.638889
8,8345,CCOC(=O)[C@@H]1CSCC(=O)N[C@@H](CC)C(=O)N(CC(C)...,-7.534591,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,81.585744,2.384457,4.768915,...,10.902997,108.140710,995.608926,6.464993,22278,115,330.0,376.0,33.750000,15.638889
9,6423,CC(=O)N1CCC[C@@H]1C(=O)N(C)[C@@H](CC(C)C)C(=O)...,-6.300000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,83.165492,2.427826,4.818625,...,10.978746,125.067539,987.636855,6.371851,22212,114,348.0,400.0,31.611111,15.444444


In [117]:
merged_df = df_test.merge(df[['ID', 'SMILES', 'Permeability']], on='ID', how='left')
merged_df = merged_df[['ID', 'SMILES', 'Permeability'] + [col for col in merged_df.columns if col not in ['ID', 'SMILES', 'Permeability']]]
merged_df

,ID,SMILES,Permeability,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,...,AMW,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb
0,8168,CCOC(=O)[C@@H]1CSCC(=O)N[C@@H](CC)C(=O)N(C)[C@...,-6.031517,0,-3.1134,9.693260,255.8998,162.779405,0,0,...,6.464993,133.378413,1.933020,59.341751,28.115454,28.210938,22236.0,115.0,5.397,330.0
1,1108,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](C)N...,-5.180000,0,-3.6834,13.567436,252.0938,160.844198,0,0,...,6.250992,130.791399,1.923403,56.234345,25.152766,31.081579,21154.0,120.0,5.288,332.0
2,1843,CC[C@H](C)[C@@H]1NC(=O)[C@@H](CC(C)C)NC(=O)[C@...,-4.400000,0,-1.2217,1.492551,199.1082,122.245994,0,0,...,6.580557,106.143859,2.002714,34.176054,15.419528,18.756526,10195.0,85.0,7.150,266.0
3,1842,CC[C@H](C)[C@@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H...,-4.400000,0,-1.6634,2.766900,183.2267,118.299580,0,0,...,6.331431,99.084602,1.981692,34.113502,15.397863,18.715639,8694.0,80.0,7.336,248.0
4,8143,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.406714,0,-5.0816,25.822659,254.9508,167.632991,0,0,...,6.465978,138.417801,1.949546,59.630333,28.216709,28.395866,23619.0,120.0,6.424,344.0
5,8119,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.073658,0,-4.9290,24.295041,252.2939,164.539405,0,0,...,6.500703,136.601049,1.951444,59.784728,28.235280,28.531690,22971.0,116.0,5.752,338.0
6,6423,CC(=O)N1CCC[C@@H]1C(=O)N(C)[C@@H](CC(C)C)C(=O)...,-6.300000,0,-4.0401,16.322408,244.1556,163.399405,0,0,...,6.371851,137.310770,1.961582,56.609941,28.393162,28.216778,22212.0,114.0,7.691,348.0
7,6496,CC(=O)N1CCC[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N[...,-4.860000,0,-3.9378,15.506269,249.8523,166.492991,0,0,...,6.339573,139.131996,1.959606,56.808725,28.380436,28.428289,22927.0,118.0,7.404,354.0
8,8345,CCOC(=O)[C@@H]1CSCC(=O)N[C@@H](CC)C(=O)N(CC(C)...,-7.534591,0,-3.1134,9.693260,255.8998,162.779405,0,0,...,6.464993,133.381424,1.933064,59.345483,28.116662,28.212683,22278.0,115.0,5.397,330.0
9,8133,CCC[C@@H]1NC(=O)CN(CC)C(=O)[C@H](CC(C)C)NC(=O)...,-5.355561,0,-5.5830,31.169889,254.9353,169.392991,0,0,...,6.500783,141.646739,1.967316,59.761039,28.238710,28.504056,24767.0,120.0,6.761,352.0


In [118]:
df_ordered = merged_df.merge(df[['ID']], on='ID', how='right')
df_ordered = df_ordered.reindex(df.index)
df_ordered

,ID,SMILES,Permeability,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,...,AMW,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb
0,1120,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-6.300000,0,0.3088,0.095357,335.1404,205.860058,0,0,...,6.243799,168.348068,1.957536,56.909620,25.395733,31.513887,37307.0,146.0,10.656,424.0
1,1118,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](Cc2...,-5.350000,0,-1.2456,1.551519,309.4986,187.298542,0,0,...,6.406411,156.622606,1.957783,56.617512,25.292701,31.324812,31648.0,138.0,7.786,396.0
2,1121,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-6.200000,0,-3.0908,9.553045,284.5918,178.444198,0,0,...,6.601542,155.443185,1.992861,56.706886,25.570597,31.136289,30176.0,124.0,9.100,392.0
3,8133,CCC[C@@H]1NC(=O)CN(CC)C(=O)[C@H](CC(C)C)NC(=O)...,-5.355561,0,-5.5830,31.169889,254.9353,169.392991,0,0,...,6.500783,141.646739,1.967316,59.761039,28.238710,28.504056,24767.0,120.0,6.761,352.0
4,8143,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.406714,0,-5.0816,25.822659,254.9508,167.632991,0,0,...,6.465978,138.417801,1.949546,59.630333,28.216709,28.395866,23619.0,120.0,6.424,344.0
5,8119,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.073658,0,-4.9290,24.295041,252.2939,164.539405,0,0,...,6.500703,136.601049,1.951444,59.784728,28.235280,28.531690,22971.0,116.0,5.752,338.0
6,6496,CC(=O)N1CCC[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N[...,-4.860000,0,-3.9378,15.506269,249.8523,166.492991,0,0,...,6.339573,139.131996,1.959606,56.808725,28.380436,28.428289,22927.0,118.0,7.404,354.0
7,8168,CCOC(=O)[C@@H]1CSCC(=O)N[C@@H](CC)C(=O)N(C)[C@...,-6.031517,0,-3.1134,9.693260,255.8998,162.779405,0,0,...,6.464993,133.378413,1.933020,59.341751,28.115454,28.210938,22236.0,115.0,5.397,330.0
8,8345,CCOC(=O)[C@@H]1CSCC(=O)N[C@@H](CC)C(=O)N(CC(C)...,-7.534591,0,-3.1134,9.693260,255.8998,162.779405,0,0,...,6.464993,133.381424,1.933064,59.345483,28.116662,28.212683,22278.0,115.0,5.397,330.0
9,6423,CC(=O)N1CCC[C@@H]1C(=O)N(C)[C@@H](CC(C)C)C(=O)...,-6.300000,0,-4.0401,16.322408,244.1556,163.399405,0,0,...,6.371851,137.310770,1.961582,56.609941,28.393162,28.216778,22212.0,114.0,7.691,348.0


In [119]:
df_ordered.to_csv('features/Descriptors/Test_2d_padel_curated_MDCK.csv', index=False)

In [120]:
#3d Train descriptors
df_train = pd.read_csv('features/Descriptors/Train_3d_padel_MDCK.csv')
df_train['ID'] = df_train['Name'].str.extract(r'_(\d+)$')
df_train['ID'] = df_train['ID'].astype(int)
df_train = df_train.drop('Name',axis=1)
df_train = df_train.fillna(0)
df_train

,TDB1u,TDB2u,TDB3u,TDB4u,TDB5u,TDB6u,TDB7u,TDB8u,TDB9u,TDB10u,...,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds,ID
0,1.257250,2.173560,3.040857,3.762207,4.479365,5.433091,6.324319,7.200341,8.017723,8.643154,...,0.379139,0.592853,0.588220,0.430843,37.539457,381.904349,1133.732531,0.404079,1.611916,1109
1,1.260416,2.176623,3.003512,3.724728,4.507123,5.364740,6.220417,6.957786,7.734671,8.646247,...,0.391473,0.605810,0.590096,0.443591,47.918648,635.293391,2339.415363,0.392580,1.639498,1111
2,1.258714,2.172720,2.998489,3.711930,4.575024,5.375895,6.148678,6.878328,7.654367,8.254441,...,0.361689,0.610498,0.476809,0.376802,46.741939,623.273439,2679.579442,0.348027,1.464109,1110
3,1.261300,2.176514,3.009104,3.750504,4.571767,5.406561,6.133185,6.938460,7.923559,8.891421,...,0.439382,0.599645,0.638739,0.385888,49.813316,679.522993,2224.208450,0.418444,1.624272,1115
4,1.253900,2.161926,3.006329,3.709741,4.490345,5.282280,5.979835,6.725081,7.522983,8.222529,...,0.305158,0.532685,0.549858,0.399249,48.688812,594.973414,2015.324533,0.450032,1.481792,1107
5,1.263424,2.192318,3.020211,3.760605,4.548130,5.363604,6.163289,6.940029,7.774619,8.584832,...,0.446412,0.502821,0.521474,0.367154,50.115915,697.880017,2474.945996,0.406107,1.391449,1114
6,1.262869,2.186784,3.024443,3.789388,4.554731,5.366070,6.164856,6.869123,7.593491,8.331515,...,0.387331,0.523372,0.543333,0.399556,48.105086,653.233321,2640.699755,0.372071,1.466261,1117
7,1.263840,2.204846,3.018464,3.779057,4.631373,5.386248,6.134860,6.769321,7.409481,8.126564,...,0.425476,0.454783,0.481387,0.383724,31.625480,282.045374,819.038655,0.387194,1.319895,1841
8,1.259134,2.173018,2.998297,3.721453,4.557337,5.305866,6.140953,6.875826,7.608722,8.275246,...,0.352479,0.529147,0.506830,0.434491,41.763791,509.002409,2183.417131,0.318955,1.470468,1116
9,1.262685,2.189631,3.022202,3.771761,4.572508,5.395520,6.177178,6.851905,7.539678,8.227930,...,0.398556,0.543790,0.527114,0.372276,47.613373,628.467138,2289.825818,0.393991,1.443180,1113


In [121]:
df = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc_MDCK.csv')
df 

,ID,SMILES,Permeability,ABCIndex,ABCGGIndex,AcidicGroupCount,BasicGroupCount,AdjacencyMatrix,AdjacencyMatrix.1,AdjacencyMatrix.2,...,WalkCount.19,WalkCount.20,Weight,Weight.1,WienerIndex,WienerIndex.1,ZagrebIndex,ZagrebIndex.1,ZagrebIndex.2,ZagrebIndex.3
0,1114,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-4.940000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,101.372024,2.447751,4.856077,...,11.210644,138.066274,1138.715439,6.469974,33296,140,416.0,486.0,34.055556,18.111111
1,1113,CC(C)C[C@@H]1NC(=O)[C@@H]2CCCN2C(=O)[C@@H](Cc2...,-5.820000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,101.370874,2.447978,4.860529,...,11.210644,138.066274,1138.715439,6.469974,33296,140,416.0,486.0,34.055556,18.111111
2,1117,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H]2CCCN...,-5.650000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,101.371868,2.443590,4.850728,...,11.210644,138.066274,1138.715439,6.469974,33296,140,416.0,486.0,34.055556,18.111111
3,1119,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-6.250000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,97.295643,2.408298,4.816595,...,11.141644,120.163455,1114.715439,6.406411,31648,138,396.0,458.0,36.555556,17.777778
4,2428,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N(C)[C@@H](C)C(=...,-5.510000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,96.774478,2.436865,4.825853,...,11.126159,133.738666,1090.679054,6.492137,29794,130,394.0,458.0,31.222222,17.333333
5,2446,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N2CCC[C@@H]2C(=O...,-6.400000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,96.685868,2.442093,4.836451,...,11.129393,133.743086,1090.679054,6.492137,29704,130,394.0,458.0,31.222222,17.333333
6,2445,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N2CCC[C@@H]2C(=O...,-4.820000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,96.685868,2.442093,4.836451,...,11.129393,133.743086,1088.699789,6.404116,29704,130,394.0,458.0,31.222222,17.333333
7,2427,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N(C)[C@@H](C)C(=...,-4.610000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,96.774478,2.436865,4.825853,...,11.126159,133.738666,1088.699789,6.404116,29794,130,394.0,458.0,31.222222,17.333333
8,8145,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.744727,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,89.136837,2.437073,4.831019,...,11.025019,126.223556,1047.640226,6.466915,25439,124,358.0,415.0,31.750000,16.611111
9,1107,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](C)N...,-4.610000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,86.880063,2.403063,4.806126,...,11.046579,113.682027,1046.746739,6.085737,25829,128,360.0,414.0,38.055556,16.444444


In [122]:
merged_df = df_train.merge(df[['ID', 'SMILES', 'Permeability']], on='ID', how='left')
merged_df = merged_df[['ID', 'SMILES', 'Permeability'] + [col for col in merged_df.columns if col not in ['ID', 'SMILES', 'Permeability']]]
merged_df

,ID,SMILES,Permeability,TDB1u,TDB2u,TDB3u,TDB4u,TDB5u,TDB6u,TDB7u,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,1109,C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](C)N(C)C(...,-6.120000,1.257250,2.173560,3.040857,3.762207,4.479365,5.433091,6.324319,...,0.556914,0.379139,0.592853,0.588220,0.430843,37.539457,381.904349,1133.732531,0.404079,1.611916
1,1111,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C...,-6.220000,1.260416,2.176623,3.003512,3.724728,4.507123,5.364740,6.220417,...,0.536914,0.391473,0.605810,0.590096,0.443591,47.918648,635.293391,2339.415363,0.392580,1.639498
2,1110,CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[C@H]...,-4.730000,1.258714,2.172720,2.998489,3.711930,4.575024,5.375895,6.148678,...,0.536996,0.361689,0.610498,0.476809,0.376802,46.741939,623.273439,2679.579442,0.348027,1.464109
3,1115,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H]2CCCN...,-5.190000,1.261300,2.176514,3.009104,3.750504,4.571767,5.406561,6.133185,...,0.506248,0.439382,0.599645,0.638739,0.385888,49.813316,679.522993,2224.208450,0.418444,1.624272
4,1107,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](C)N...,-4.610000,1.253900,2.161926,3.006329,3.709741,4.490345,5.282280,5.979835,...,0.633354,0.305158,0.532685,0.549858,0.399249,48.688812,594.973414,2015.324533,0.450032,1.481792
5,1114,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-4.940000,1.263424,2.192318,3.020211,3.760605,4.548130,5.363604,6.163289,...,0.490993,0.446412,0.502821,0.521474,0.367154,50.115915,697.880017,2474.945996,0.406107,1.391449
6,1117,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H]2CCCN...,-5.650000,1.262869,2.186784,3.024443,3.789388,4.554731,5.366070,6.164856,...,0.527383,0.387331,0.523372,0.543333,0.399556,48.105086,653.233321,2640.699755,0.372071,1.466261
7,1841,CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](C...,-4.410000,1.263840,2.204846,3.018464,3.779057,4.631373,5.386248,6.134860,...,0.499319,0.425476,0.454783,0.481387,0.383724,31.625480,282.045374,819.038655,0.387194,1.319895
8,1116,CC(C)C[C@@H]1NC(=O)[C@H](C(C)C)N(C)C(=O)[C@H]2...,-5.220000,1.259134,2.173018,2.998297,3.721453,4.557337,5.305866,6.140953,...,0.526825,0.352479,0.529147,0.506830,0.434491,41.763791,509.002409,2183.417131,0.318955,1.470468
9,1113,CC(C)C[C@@H]1NC(=O)[C@@H]2CCCN2C(=O)[C@@H](Cc2...,-5.820000,1.262685,2.189631,3.022202,3.771761,4.572508,5.395520,6.177178,...,0.530772,0.398556,0.543790,0.527114,0.372276,47.613373,628.467138,2289.825818,0.393991,1.443180


In [123]:
df_ordered = merged_df.merge(df[['ID']], on='ID', how='right')
df_ordered = df_ordered.reindex(df.index)
df_ordered

,ID,SMILES,Permeability,TDB1u,TDB2u,TDB3u,TDB4u,TDB5u,TDB6u,TDB7u,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,1114,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-4.940000,1.263424,2.192318,3.020211,3.760605,4.548130,5.363604,6.163289,...,0.490993,0.446412,0.502821,0.521474,0.367154,50.115915,697.880017,2474.945996,0.406107,1.391449
1,1113,CC(C)C[C@@H]1NC(=O)[C@@H]2CCCN2C(=O)[C@@H](Cc2...,-5.820000,1.262685,2.189631,3.022202,3.771761,4.572508,5.395520,6.177178,...,0.530772,0.398556,0.543790,0.527114,0.372276,47.613373,628.467138,2289.825818,0.393991,1.443180
2,1117,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H]2CCCN...,-5.650000,1.262869,2.186784,3.024443,3.789388,4.554731,5.366070,6.164856,...,0.527383,0.387331,0.523372,0.543333,0.399556,48.105086,653.233321,2640.699755,0.372071,1.466261
3,1119,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-6.250000,1.257678,2.183255,3.025337,3.733658,4.519376,5.374286,6.204598,...,0.528976,0.349388,0.516241,0.483660,0.323140,49.516594,715.115788,3493.983049,0.317545,1.323041
4,2428,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N(C)[C@@H](C)C(=...,-5.510000,1.264328,2.185662,3.003434,3.757888,4.608846,5.372392,6.150767,...,0.506249,0.395804,0.498185,0.483571,0.313967,46.999494,637.788214,2722.371988,0.353079,1.295723
5,2446,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N2CCC[C@@H]2C(=O...,-6.400000,1.264706,2.186683,3.011032,3.768187,4.555367,5.295765,6.051415,...,0.465091,0.397955,0.535696,0.559095,0.348772,46.003992,641.858241,3155.799517,0.294568,1.443562
6,2445,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N2CCC[C@@H]2C(=O...,-4.820000,1.263513,2.186487,3.002543,3.755821,4.583602,5.306940,6.051866,...,0.493305,0.428676,0.535662,0.498539,0.389916,53.087085,798.688967,3320.153449,0.382972,1.424116
7,2427,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N(C)[C@@H](C)C(=...,-4.610000,1.263546,2.188196,2.998217,3.747038,4.586668,5.393488,6.110721,...,0.483498,0.392982,0.564432,0.505432,0.322603,48.069815,689.212021,3344.170973,0.314720,1.392467
8,8145,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.744727,1.266162,2.183183,3.002416,3.760293,4.571845,5.408361,6.163837,...,0.565467,0.377819,0.494304,0.444191,0.445117,47.071607,591.915175,1902.720960,0.414930,1.383611
9,1107,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](C)N...,-4.610000,1.253900,2.161926,3.006329,3.709741,4.490345,5.282280,5.979835,...,0.633354,0.305158,0.532685,0.549858,0.399249,48.688812,594.973414,2015.324533,0.450032,1.481792


In [124]:
df_ordered.to_csv('features/Descriptors/Train_3d_padel_curated_MDCK.csv', index=False)

In [125]:
#3d test padel descriptors
df_test = pd.read_csv('features/Descriptors/Test_3d_padel_MDCK.csv')
df_test['ID'] = df_test['Name'].str.extract(r'_(\d+)$')
df_test['ID'] = df_test['ID'].astype(int)
df_test = df_test.drop('Name',axis=1)
df_test = df_test.fillna(0)
df_test

,TDB1u,TDB2u,TDB3u,TDB4u,TDB5u,TDB6u,TDB7u,TDB8u,TDB9u,TDB10u,...,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds,ID
0,1.262962,2.203757,3.010472,3.762148,4.605706,5.382284,6.090129,6.714410,7.318055,7.958420,...,0.403049,0.448745,0.502249,0.335460,28.929964,250.102493,847.276168,0.315790,1.286454,1843
1,1.254356,2.164453,3.007603,3.708399,4.515305,5.373686,6.085998,6.903745,7.805573,8.600115,...,0.369142,0.519954,0.531563,0.333557,46.528470,572.262997,1788.618525,0.418108,1.385074,1108
2,1.260366,2.172649,3.013872,3.733320,4.586889,5.399569,6.176406,7.035502,7.901117,8.810896,...,0.400623,0.475788,0.516266,0.353488,52.551241,713.790090,1832.795103,0.451477,1.345542,8168
3,1.265772,2.186831,2.995030,3.733071,4.548541,5.362531,6.157497,6.964772,7.862036,8.754818,...,0.391394,0.465474,0.444791,0.411105,54.943002,776.072156,2056.663536,0.450816,1.321370,8133
4,1.262152,2.177810,2.998624,3.747792,4.570792,5.345531,6.076177,6.749802,7.647241,8.600880,...,0.342655,0.553639,0.636460,0.287638,45.060837,582.506417,2560.016441,0.330056,1.477737,8119
5,1.263467,2.177773,3.010879,3.756409,4.537040,5.305276,6.013423,6.684024,7.506885,8.340009,...,0.411553,0.528651,0.525749,0.441578,46.170332,590.730936,2052.174504,0.399489,1.495978,8143
6,1.262625,2.178379,3.001984,3.751093,4.587410,5.377711,6.116195,6.829437,7.457753,8.147195,...,0.296155,0.470177,0.475961,0.342756,49.058557,593.936407,1983.399773,0.466531,1.288893,6496
7,1.262958,2.182461,2.999257,3.741833,4.604033,5.454285,6.214735,6.970312,7.669033,8.378282,...,0.311878,0.435148,0.417772,0.254081,49.869778,644.135872,2411.114189,0.424098,1.107002,6423
8,1.260968,2.174923,3.012771,3.726420,4.548957,5.352239,6.159423,6.898790,7.649232,8.374630,...,0.379510,0.478903,0.524793,0.340577,45.177438,575.947872,2255.562804,0.368514,1.344273,8345
9,1.263934,2.202480,3.003131,3.755747,4.669339,5.455309,6.206667,6.958425,7.732563,8.624447,...,0.319304,0.527342,0.614802,0.396168,50.379698,712.333697,3347.440319,0.354317,1.538312,1121


In [126]:
df = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc_MDCK.csv')
df

,ID,SMILES,Permeability,ABCIndex,ABCGGIndex,AcidicGroupCount,BasicGroupCount,AdjacencyMatrix,AdjacencyMatrix.1,AdjacencyMatrix.2,...,WalkCount.19,WalkCount.20,Weight,Weight.1,WienerIndex,WienerIndex.1,ZagrebIndex,ZagrebIndex.1,ZagrebIndex.2,ZagrebIndex.3
0,1120,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-6.300000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,103.838268,2.420071,4.840143,...,11.200910,126.480968,1198.809340,6.243799,37307,146,424.0,488.0,39.277778,19.111111
1,1118,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](Cc2...,-5.350000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,97.302139,2.412394,4.824787,...,11.141934,120.163745,1114.715439,6.406411,31648,138,396.0,458.0,36.555556,17.777778
2,1121,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-6.200000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,95.621135,2.421653,4.790984,...,11.069213,133.614531,1082.652839,6.601542,30176,124,392.0,450.0,30.611111,17.222222
3,8133,CCC[C@@H]1NC(=O)CN(CC)C(=O)[C@H](CC(C)C)NC(=O)...,-5.355561,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,88.159241,2.431154,4.817857,...,10.985954,125.096326,1033.624576,6.500783,24767,120,352.0,406.0,30.888889,16.388889
4,8143,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.406714,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,85.381854,2.437654,4.830440,...,10.978917,124.007460,1021.624576,6.465978,23619,120,344.0,397.0,32.750000,16.194444
5,8119,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.073658,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,83.965708,2.427767,4.806183,...,10.937792,122.874218,1007.608926,6.500703,22971,116,338.0,388.0,31.888889,15.972222
6,6496,CC(=O)N1CCC[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N[...,-4.860000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,84.630944,2.429295,4.822808,...,11.018416,126.196276,1001.652505,6.339573,22927,118,354.0,409.0,32.472222,15.666667
7,8168,CCOC(=O)[C@@H]1CSCC(=O)N[C@@H](CC)C(=O)N(C)[C@...,-6.031517,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,81.585614,2.381030,4.762061,...,10.902997,108.140710,995.608926,6.464993,22236,115,330.0,376.0,33.750000,15.638889
8,8345,CCOC(=O)[C@@H]1CSCC(=O)N[C@@H](CC)C(=O)N(CC(C)...,-7.534591,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,81.585744,2.384457,4.768915,...,10.902997,108.140710,995.608926,6.464993,22278,115,330.0,376.0,33.750000,15.638889
9,6423,CC(=O)N1CCC[C@@H]1C(=O)N(C)[C@@H](CC(C)C)C(=O)...,-6.300000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,83.165492,2.427826,4.818625,...,10.978746,125.067539,987.636855,6.371851,22212,114,348.0,400.0,31.611111,15.444444


In [127]:
merged_df = df_test.merge(df[['ID', 'SMILES', 'Permeability']], on='ID', how='left')
merged_df = merged_df[['ID', 'SMILES', 'Permeability'] + [col for col in merged_df.columns if col not in ['ID', 'SMILES', 'Permeability']]]
merged_df

,ID,SMILES,Permeability,TDB1u,TDB2u,TDB3u,TDB4u,TDB5u,TDB6u,TDB7u,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,1843,CC[C@H](C)[C@@H]1NC(=O)[C@@H](CC(C)C)NC(=O)[C@...,-4.400000,1.262962,2.203757,3.010472,3.762148,4.605706,5.382284,6.090129,...,0.474145,0.403049,0.448745,0.502249,0.335460,28.929964,250.102493,847.276168,0.315790,1.286454
1,1108,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](C)N...,-5.180000,1.254356,2.164453,3.007603,3.708399,4.515305,5.373686,6.085998,...,0.576263,0.369142,0.519954,0.531563,0.333557,46.528470,572.262997,1788.618525,0.418108,1.385074
2,8168,CCOC(=O)[C@@H]1CSCC(=O)N[C@@H](CC)C(=O)N(C)[C@...,-6.031517,1.260366,2.172649,3.013872,3.733320,4.586889,5.399569,6.176406,...,0.567029,0.400623,0.475788,0.516266,0.353488,52.551241,713.790090,1832.795103,0.451477,1.345542
3,8133,CCC[C@@H]1NC(=O)CN(CC)C(=O)[C@H](CC(C)C)NC(=O)...,-5.355561,1.265772,2.186831,2.995030,3.733071,4.548541,5.362531,6.157497,...,0.575816,0.391394,0.465474,0.444791,0.411105,54.943002,776.072156,2056.663536,0.450816,1.321370
4,8119,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.073658,1.262152,2.177810,2.998624,3.747792,4.570792,5.345531,6.076177,...,0.544048,0.342655,0.553639,0.636460,0.287638,45.060837,582.506417,2560.016441,0.330056,1.477737
5,8143,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.406714,1.263467,2.177773,3.010879,3.756409,4.537040,5.305276,6.013423,...,0.521440,0.411553,0.528651,0.525749,0.441578,46.170332,590.730936,2052.174504,0.399489,1.495978
6,6496,CC(=O)N1CCC[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N[...,-4.860000,1.262625,2.178379,3.001984,3.751093,4.587410,5.377711,6.116195,...,0.644354,0.296155,0.470177,0.475961,0.342756,49.058557,593.936407,1983.399773,0.466531,1.288893
7,6423,CC(=O)N1CCC[C@@H]1C(=O)N(C)[C@@H](CC(C)C)C(=O)...,-6.300000,1.262958,2.182461,2.999257,3.741833,4.604033,5.454285,6.214735,...,0.616065,0.311878,0.435148,0.417772,0.254081,49.869778,644.135872,2411.114189,0.424098,1.107002
8,8345,CCOC(=O)[C@@H]1CSCC(=O)N[C@@H](CC)C(=O)N(CC(C)...,-7.534591,1.260968,2.174923,3.012771,3.726420,4.548957,5.352239,6.159423,...,0.532833,0.379510,0.478903,0.524793,0.340577,45.177438,575.947872,2255.562804,0.368514,1.344273
9,1121,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-6.200000,1.263934,2.202480,3.003131,3.755747,4.669339,5.455309,6.206667,...,0.569544,0.319304,0.527342,0.614802,0.396168,50.379698,712.333697,3347.440319,0.354317,1.538312


In [128]:
df_ordered = merged_df.merge(df[['ID']], on='ID', how='right')
df_ordered = df_ordered.reindex(df.index)
df_ordered

,ID,SMILES,Permeability,TDB1u,TDB2u,TDB3u,TDB4u,TDB5u,TDB6u,TDB7u,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,1120,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-6.300000,1.256129,2.176604,3.014198,3.723798,4.557097,5.391202,6.103738,...,0.517052,0.333526,0.520259,0.510811,0.420493,49.703108,739.996624,3953.646545,0.275867,1.451563
1,1118,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](Cc2...,-5.350000,1.256449,2.178478,3.022149,3.744684,4.529544,5.323638,6.067260,...,0.544737,0.347757,0.647006,0.459842,0.383878,47.483083,643.440777,2871.212438,0.338740,1.490726
2,1121,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-6.200000,1.263934,2.202480,3.003131,3.755747,4.669339,5.455309,6.206667,...,0.569544,0.319304,0.527342,0.614802,0.396168,50.379698,712.333697,3347.440319,0.354317,1.538312
3,8133,CCC[C@@H]1NC(=O)CN(CC)C(=O)[C@H](CC(C)C)NC(=O)...,-5.355561,1.265772,2.186831,2.995030,3.733071,4.548541,5.362531,6.157497,...,0.575816,0.391394,0.465474,0.444791,0.411105,54.943002,776.072156,2056.663536,0.450816,1.321370
4,8143,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.406714,1.263467,2.177773,3.010879,3.756409,4.537040,5.305276,6.013423,...,0.521440,0.411553,0.528651,0.525749,0.441578,46.170332,590.730936,2052.174504,0.399489,1.495978
5,8119,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.073658,1.262152,2.177810,2.998624,3.747792,4.570792,5.345531,6.076177,...,0.544048,0.342655,0.553639,0.636460,0.287638,45.060837,582.506417,2560.016441,0.330056,1.477737
6,6496,CC(=O)N1CCC[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N[...,-4.860000,1.262625,2.178379,3.001984,3.751093,4.587410,5.377711,6.116195,...,0.644354,0.296155,0.470177,0.475961,0.342756,49.058557,593.936407,1983.399773,0.466531,1.288893
7,8168,CCOC(=O)[C@@H]1CSCC(=O)N[C@@H](CC)C(=O)N(C)[C@...,-6.031517,1.260366,2.172649,3.013872,3.733320,4.586889,5.399569,6.176406,...,0.567029,0.400623,0.475788,0.516266,0.353488,52.551241,713.790090,1832.795103,0.451477,1.345542
8,8345,CCOC(=O)[C@@H]1CSCC(=O)N[C@@H](CC)C(=O)N(CC(C)...,-7.534591,1.260968,2.174923,3.012771,3.726420,4.548957,5.352239,6.159423,...,0.532833,0.379510,0.478903,0.524793,0.340577,45.177438,575.947872,2255.562804,0.368514,1.344273
9,6423,CC(=O)N1CCC[C@@H]1C(=O)N(C)[C@@H](CC(C)C)C(=O)...,-6.300000,1.262958,2.182461,2.999257,3.741833,4.604033,5.454285,6.214735,...,0.616065,0.311878,0.435148,0.417772,0.254081,49.869778,644.135872,2411.114189,0.424098,1.107002


In [129]:
df_ordered.to_csv('features/Descriptors/Test_3d_padel_curated_MDCK.csv', index=False)

In [130]:
#2d Padel descriptors
df_train = pd.read_csv('features/Descriptors/Train_2d_padel_curated_MDCK.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_padel_curated_MDCK.csv')
df_test = df_test.dropna()
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 1444)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 1444)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.150684 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3449
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 255
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spli

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3890,0.4981,0.6237,0.2606,0.5133,0.4845,0.4143,0.5238,0.6436,0.4207,0.7371,0.5620
DecisionTreeRegressor,0.4938,0.5159,0.7027,0.0616,0.4811,0.4415,0.4517,0.5317,0.6721,0.3684,0.6119,0.5262
RandomForestRegressor,0.3331,0.4692,0.5771,0.3670,0.6136,0.6254,0.4263,0.5518,0.6529,0.4038,0.6680,0.4986
GradientBoostingRegressor,0.3883,0.4936,0.6231,0.2621,0.5864,0.5577,0.4396,0.5353,0.6630,0.3853,0.6287,0.4601
AdaBoostRegressor,0.3213,0.4465,0.5668,0.3894,0.6268,0.6169,0.4862,0.5602,0.6973,0.3201,0.5693,0.4380
XGBRegressor,0.2819,0.4264,0.5310,0.4642,0.6961,0.6395,0.4041,0.5132,0.6357,0.4349,0.6760,0.5537
ExtraTreesRegressor,0.2854,0.4184,0.5343,0.4575,0.6807,0.6543,0.3990,0.5087,0.6317,0.4420,0.6851,0.5262
LinearRegression,0.6591,0.5839,0.8119,-0.2527,0.5359,0.5682,0.6412,0.5974,0.8007,0.1034,0.6246,0.7273
KNeighborsRegressor,0.3601,0.4519,0.6001,0.3155,0.5698,0.5291,0.4589,0.5470,0.6774,0.3583,0.6218,0.4325
SVR,0.4066,0.5029,0.6377,0.2272,0.4823,0.5094,0.4276,0.4890,0.6539,0.4021,0.6681,0.5675


In [131]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.181702344056397, -5.313238713016328, -5.95...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.313238713016328, -5.567742830653738, -5.2...","[-5.376856679124589, -5.526691384204971, -5.63...","[0.14555366830585018, 0.19450464980409402, 0.2..."
1,DecisionTreeRegressor,"[-4.94, -5.32, -5.984295768000001, -5.28903688...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.58, -4.94, -4.73, -5.289036881, -5.185752...","[-5.406000000000001, -5.656000000000001, -5.51...","[0.6453712110096018, 0.535820865588491, 0.4932..."
2,RandomForestRegressor,"[-5.3651666495133306, -5.314070908362498, -5.8...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.189572378913333, -5.75767186221, -5.82712...","[-5.347228752023801, -5.717783976145801, -5.85...","[0.15319032905157012, 0.17777719908977407, 0.0..."
3,GradientBoostingRegressor,"[-5.185243291526317, -4.871761145150415, -6.21...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.857216853722791, -5.74131748463349, -5.20...","[-5.3485168750633525, -5.870370176877826, -5.8...","[0.4482698005134859, 0.35500767216040646, 0.34..."
4,AdaBoostRegressor,"[-5.509113140714285, -5.208468812615385, -6.18...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.02, -5.845550411249999, -5.58303064113333...","[-5.190122730952001, -5.8208382117715685, -5.8...","[0.28362438400734674, 0.2060559442970082, 0.20..."
5,XGBRegressor,"[-5.1857853, -4.8836474, -6.0389395, -5.034064...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.4093223, -5.41427, -4.9545627, -5.315364,...","[-5.586074, -5.845536, -5.6030626, -5.4576864,...","[0.25722784, 0.30775592, 0.33314005, 0.2222955..."
6,ExtraTreesRegressor,"[-5.202840635329998, -4.911609916082497, -6.05...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.500849999999999, -6.018480601440001, -5.8...","[-5.481714062969999, -5.9400391819785, -5.8852...","[0.21850689622122357, 0.2957661602181418, 0.05..."
7,LinearRegression,"[-5.429377144467243, -4.962984046493772, -4.98...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.726509870239312, -5.734432545055338, -9.0...","[-6.4422197252639375, -5.811777985023915, -8.1...","[0.2924572527355262, 0.5893225799543569, 0.539..."
8,KNeighborsRegressor,"[-5.613333333333333, -5.3999999999999995, -5.6...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333334, -5.2...","[-5.6546666666666665, -5.666000000000001, -5.5...","[0.14230796026770795, 0.14081350945290821, 0.2..."
9,SVR,"[-5.41138733424168, -5.400042448608094, -5.744...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.674173919907601, -5.916390511081038, -5.6...","[-5.677894327886973, -5.861468488665199, -5.65...","[0.186278283226074, 0.2407125307712907, 0.1054..."


In [132]:
result_df.to_csv('results/Descriptors/Results_2D_padel_MDCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_padel_MDCK.csv')

In [133]:
#2d padel descriptors const removal
df_train = pd.read_csv('features/Descriptors/Train_2d_padel_curated_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train, const_col = remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_padel_curated_MDCK.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = X_test.drop(const_col,axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 992)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 992)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.120942 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3449
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 255
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3890,0.4981,0.6237,0.2606,0.5133,0.4845,0.4143,0.5238,0.6436,0.4207,0.7371,0.5620
DecisionTreeRegressor,0.5161,0.5478,0.7184,0.0192,0.4715,0.4666,0.4294,0.5093,0.6553,0.3995,0.6400,0.5565
RandomForestRegressor,0.3400,0.4771,0.5831,0.3538,0.6009,0.6095,0.4273,0.5543,0.6537,0.4025,0.6663,0.4601
GradientBoostingRegressor,0.3914,0.5015,0.6256,0.2561,0.5794,0.5453,0.4292,0.5362,0.6551,0.3998,0.6442,0.4904
AdaBoostRegressor,0.2956,0.4226,0.5437,0.4382,0.6700,0.6129,0.4445,0.5447,0.6667,0.3785,0.6298,0.4380
XGBRegressor,0.2819,0.4264,0.5310,0.4642,0.6961,0.6395,0.4041,0.5132,0.6357,0.4349,0.6760,0.5537
ExtraTreesRegressor,0.3142,0.4417,0.5605,0.4029,0.6428,0.6056,0.4035,0.5032,0.6352,0.4357,0.6791,0.5455
LinearRegression,0.6591,0.5839,0.8119,-0.2527,0.5359,0.5682,0.6412,0.5974,0.8007,0.1034,0.6246,0.7273
KNeighborsRegressor,0.3601,0.4519,0.6001,0.3155,0.5698,0.5291,0.4589,0.5470,0.6774,0.3583,0.6218,0.4325
SVR,0.4066,0.5029,0.6377,0.2272,0.4823,0.5094,0.4276,0.4890,0.6539,0.4020,0.6681,0.5675


In [134]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.181702344056397, -5.313238713016328, -5.95...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.313238713016328, -5.567742830653738, -5.2...","[-5.376856679124589, -5.526691384204971, -5.63...","[0.14555366830585018, 0.19450464980409402, 0.2..."
1,DecisionTreeRegressor,"[-4.94, -5.32, -6.05, -4.82, -7.698970004, -5....",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.61, -4.94, -4.82, -5.289036881, -5.185752...","[-5.454000000000001, -5.7620000000000005, -5.5...","[0.5970795591878856, 0.6003798797428173, 0.642..."
2,RandomForestRegressor,"[-5.3356262076649985, -5.263841081106665, -5.8...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.310254730924998, -5.927677645478573, -5.7...","[-5.370251634670334, -5.744969694999716, -5.83...","[0.16562219905593567, 0.21690384399370072, 0.0..."
3,GradientBoostingRegressor,"[-5.428073852805545, -4.964434365381989, -6.21...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.893353701612168, -5.628303931150666, -5.1...","[-5.385420727546547, -5.855857608349488, -5.86...","[0.4878756221580647, 0.3523993824382789, 0.390..."
4,AdaBoostRegressor,"[-5.5037200492, -5.101282249625, -6.1423799976...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.4025, -5.775950829400001, -5.629868588571...","[-5.30172985977851, -5.820459324147274, -5.819...","[0.2961434057236757, 0.20228508856293845, 0.20..."
5,XGBRegressor,"[-5.1857853, -4.8836474, -6.0389395, -5.034064...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.4093223, -5.41427, -4.9545627, -5.315364,...","[-5.586074, -5.845536, -5.6030626, -5.4576864,...","[0.25722784, 0.30775592, 0.33314005, 0.2222955..."
6,ExtraTreesRegressor,"[-5.085147274950001, -5.065079750719997, -5.99...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.536749999999997, -6.023079932670002, -5.8...","[-5.424579531909167, -5.908770792078499, -5.94...","[0.22511230469462726, 0.3219017836500588, 0.05..."
7,LinearRegression,"[-5.429377144467245, -4.962984046493777, -4.98...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.72650987023931, -5.734432545055342, -9.09...","[-6.442219725263939, -5.811777985023918, -8.16...","[0.2924572527355256, 0.5893225799543581, 0.539..."
8,KNeighborsRegressor,"[-5.613333333333333, -5.3999999999999995, -5.6...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333334, -5.2...","[-5.6546666666666665, -5.666000000000001, -5.5...","[0.14230796026770795, 0.14081350945290821, 0.2..."
9,SVR,"[-5.411438115444501, -5.399962764094401, -5.74...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.674204657515688, -5.9164450054685185, -5....","[-5.677906535109213, -5.861491174466585, -5.65...","[0.18627467322971664, 0.24071872280783782, 0.1..."


In [135]:
result_df.to_csv('results/Descriptors/Results_2D_padel_const_rem_MDCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_padel_const_rem_MDCK.csv')

In [136]:
#2d padel descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_2d_padel_curated_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train, const_col = remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_padel_curated_MDCK.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = X_test.drop(const_col,axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 642)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 642)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.127908 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2115
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 158
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3761,0.5041,0.6133,0.2852,0.5379,0.5294,0.4440,0.5503,0.6663,0.3791,0.6770,0.6722
DecisionTreeRegressor,0.6402,0.6086,0.8001,-0.2166,0.5461,0.4939,0.4588,0.5859,0.6774,0.3584,0.6237,0.4986
RandomForestRegressor,0.3707,0.5089,0.6089,0.2954,0.5605,0.5828,0.4424,0.5648,0.6651,0.3813,0.6513,0.5482
GradientBoostingRegressor,0.4335,0.5166,0.6584,0.1761,0.5616,0.5126,0.4094,0.5285,0.6399,0.4275,0.6778,0.4986
AdaBoostRegressor,0.2740,0.4077,0.5234,0.4793,0.6973,0.6391,0.4744,0.5656,0.6887,0.3367,0.5906,0.5482
XGBRegressor,0.5230,0.5652,0.7232,0.0061,0.4340,0.4357,0.3982,0.5155,0.6310,0.4432,0.6913,0.6006
ExtraTreesRegressor,0.2782,0.4294,0.5274,0.4713,0.6922,0.6403,0.4182,0.5155,0.6467,0.4152,0.6583,0.5152
LinearRegression,0.9826,0.7372,0.9913,-0.8675,0.3884,0.4132,0.9013,0.6834,0.9493,-0.2603,0.5725,0.7163
KNeighborsRegressor,0.3931,0.4924,0.6270,0.2528,0.5280,0.5086,0.4520,0.5534,0.6723,0.3679,0.6281,0.4455
SVR,0.4242,0.5162,0.6513,0.1937,0.4461,0.4680,0.4320,0.4864,0.6572,0.3960,0.6666,0.5647


In [137]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.241730271618002, -5.284679741187142, -5.40...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.284679741187142, -5.762060833192084, -6.4...","[-5.38455750105418, -5.615187868064276, -6.084...","[0.1579647159314029, 0.25790608367777706, 0.22..."
1,DecisionTreeRegressor,"[-4.58, -4.61, -6.05, -5.752026734, -7.6989700...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.25, -6.25, -4.61, -5.289036881, -5.752026...","[-5.508, -6.1103378892, -5.8609454990000005, -...","[0.7498373156892099, 0.30493971839110395, 0.66..."
2,RandomForestRegressor,"[-5.217895890599998, -5.412462212983332, -5.81...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.301558930573334, -5.848343700191334, -5.9...","[-5.391550177843238, -5.816318666709705, -6.06...","[0.16924793877042696, 0.14318938979269444, 0.1..."
3,GradientBoostingRegressor,"[-5.412115265013379, -5.006439412407397, -6.29...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.666166324134487, -6.026053121727395, -5.3...","[-5.598152178794464, -5.982706899853454, -5.88...","[0.43224622351203473, 0.31192509917442696, 0.4..."
4,AdaBoostRegressor,"[-5.22125, -5.2036044744, -6.117403133833332, ...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.22125, -6.234999999999999, -6.190243322, ...","[-5.235142799007906, -5.862959308731229, -5.97...","[0.22413299365707737, 0.25649221437106134, 0.2..."
5,XGBRegressor,"[-5.45672, -4.8887424, -5.7698536, -5.5672617,...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.0418353, -6.4486685, -5.3985257, -5.38439...","[-5.6947837, -6.1200485, -5.921075, -5.721923,...","[0.42349374, 0.2964004, 0.49319342, 0.5180346,..."
6,ExtraTreesRegressor,"[-5.215490368809998, -5.176692430202499, -5.92...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.357497274950001, -6.1981266241599995, -6....","[-5.3185181518005, -5.997445321879668, -6.0389...","[0.23800908091919235, 0.2750920781944004, 0.09..."
7,LinearRegression,"[-4.914571052165563, -4.6111440374432, -5.1784...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-7.306023365950994, -5.303900153894148, -10....","[-6.559706292762629, -5.611145025884625, -8.76...","[0.5306450855256852, 0.900368235100485, 1.0394..."
8,KNeighborsRegressor,"[-5.066666666666666, -5.3999999999999995, -5.8...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333333, -5.8...","[-5.713333333333333, -5.713333333333333, -5.62...","[0.1707890186425604, 0.1707890186425604, 0.282..."
9,SVR,"[-5.391286520219778, -5.347386254529625, -5.71...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.681983575526109, -5.901791670644574, -5.7...","[-5.689514278812022, -5.830663429970727, -5.71...","[0.18892579314338112, 0.25567865962539166, 0.1..."


In [138]:
result_df.to_csv('results/Descriptors/Results_2D_padel_LVR_MDCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_padel_const_LVR_MDCK.csv')

In [139]:
#2d All descriptors
df_train_padel = pd.read_csv('features/Descriptors/Train_2d_padel_curated_MDCK.csv')
df_train_rdkit = pd.read_csv('features/Descriptors/Train_2d_RDKit_des_MDCK.csv')
df_train_mordred = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc_MDCK.csv')

df_2d_train = df_train_rdkit.merge(df_train_mordred, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_train_padel, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_2d_train

,ID,SMILES,Permeability,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,AMW,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb
0,1114,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-4.940000,14.922930,14.922930,0.059501,-1.162678,0.238950,27.097561,1139.494,...,6.469974,162.717774,1.984363,57.439206,25.502169,31.937037,33296.0,140.0,7.952,416.0
1,1113,CC(C)C[C@@H]1NC(=O)[C@@H]2CCCN2C(=O)[C@@H](Cc2...,-5.820000,15.128047,15.128047,0.075261,-1.145488,0.238950,27.097561,1139.494,...,6.469974,162.720191,1.984393,57.435556,25.501133,31.934422,33296.0,140.0,7.952,416.0
2,1117,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H]2CCCN...,-5.650000,14.908525,14.908525,0.051146,-1.177248,0.238950,27.097561,1139.494,...,6.469974,162.717400,1.984359,57.440346,25.502649,31.937697,33296.0,140.0,7.952,416.0
3,1119,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-6.250000,14.724823,14.724823,0.030635,-1.196597,0.240803,26.125000,1115.472,...,6.406411,156.626458,1.957831,56.630904,25.293944,31.336960,31648.0,138.0,7.786,396.0
4,2428,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N(C)[C@@H](C)C(=...,-5.510000,14.918925,14.918925,0.008728,-1.166833,0.250056,27.358974,1091.406,...,6.492137,155.242276,1.990286,60.716736,28.525011,32.191725,29794.0,130.0,5.187,394.0
5,2446,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N2CCC[C@@H]2C(=O...,-6.400000,14.912312,14.912312,0.031806,-1.168185,0.250056,27.358974,1091.406,...,6.492137,155.232056,1.990155,60.685463,28.512533,32.172930,29704.0,130.0,5.187,394.0
6,2445,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N2CCC[C@@H]2C(=O...,-4.820000,14.926201,14.926201,0.060156,-1.148185,0.252488,27.358974,1089.434,...,6.404116,155.232056,1.990155,57.684402,25.511472,32.172930,29704.0,130.0,7.084,394.0
7,2427,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N(C)[C@@H](C)C(=...,-4.610000,14.950175,14.950175,0.063054,-1.159021,0.252488,27.358974,1089.434,...,6.404116,155.242276,1.990286,57.714858,25.523133,32.191725,29794.0,130.0,7.084,394.0
8,8145,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.744727,14.598899,14.598899,0.013996,-1.178122,0.218577,28.506849,1048.403,...,6.466915,143.466261,1.965291,59.644932,28.221894,28.405281,25439.0,124.0,7.222,358.0
9,1107,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](C)N...,-4.610000,14.543197,14.543197,0.095971,-1.130815,0.222694,27.567568,1047.438,...,6.085737,142.514887,1.925877,56.524332,25.255908,31.268424,25829.0,128.0,8.158,360.0


In [140]:
df_2d_train.to_csv('features/Descriptors/Train_2d_all_descriptors_MDCK.csv', index=False)

In [141]:
df_test_padel = pd.read_csv('features/Descriptors/Test_2d_padel_curated_MDCK.csv')
df_test_rdkit = pd.read_csv('features/Descriptors/Test_2d_RDKit_des_MDCK.csv')
df_test_mordred = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc_MDCK.csv')

df_2d_test = df_test_rdkit.merge(df_test_mordred, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_test_padel, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_2d_test

,ID,SMILES,Permeability,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,AMW,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb
0,1120,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-6.300000,15.211581,15.211581,0.031713,-1.206465,0.159924,25.279070,1199.634,...,6.243799,168.348068,1.957536,56.909620,25.395733,31.513887,37307.0,146.0,10.656,424.0
1,1118,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](Cc2...,-5.350000,14.834083,14.834083,0.037197,-1.184826,0.240803,26.125000,1115.472,...,6.406411,156.622606,1.957783,56.617512,25.292701,31.324812,31648.0,138.0,7.786,396.0
2,1121,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-6.200000,14.571211,14.571211,0.032067,-1.213008,0.153969,27.307692,1083.386,...,6.601542,155.443185,1.992861,56.706886,25.570597,31.136289,30176.0,124.0,9.100,392.0
3,8133,CCC[C@@H]1NC(=O)CN(CC)C(=O)[C@H](CC(C)C)NC(=O)...,-5.355561,14.507460,14.507460,0.015782,-1.174377,0.218370,27.694444,1034.376,...,6.500783,141.646739,1.967316,59.761039,28.238710,28.504056,24767.0,120.0,6.761,352.0
4,8143,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.406714,14.432367,14.432367,0.010248,-1.181980,0.206072,27.492958,1022.365,...,6.465978,138.417801,1.949546,59.630333,28.216709,28.395866,23619.0,120.0,6.424,344.0
5,8119,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.073658,14.337271,14.337271,0.011275,-1.179506,0.205195,26.642857,1008.338,...,6.500703,136.601049,1.951444,59.784728,28.235280,28.531690,22971.0,116.0,5.752,338.0
6,6496,CC(=O)N1CCC[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N[...,-4.860000,14.798396,14.798396,0.082911,-1.653049,0.157962,27.464789,1002.309,...,6.339573,139.131996,1.959606,56.808725,28.380436,28.428289,22927.0,118.0,7.404,354.0
7,8168,CCOC(=O)[C@@H]1CSCC(=O)N[C@@H](CC)C(=O)N(C)[C@...,-6.031517,14.228063,14.228063,0.008269,-1.181029,0.216153,26.159420,996.327,...,6.464993,133.378413,1.933020,59.341751,28.115454,28.210938,22236.0,115.0,5.397,330.0
8,8345,CCOC(=O)[C@@H]1CSCC(=O)N[C@@H](CC)C(=O)N(CC(C)...,-7.534591,14.235704,14.235704,0.008614,-1.181469,0.216153,26.159420,996.327,...,6.464993,133.381424,1.933064,59.345483,28.116662,28.212683,22278.0,115.0,5.397,330.0
9,6423,CC(=O)N1CCC[C@@H]1C(=O)N(C)[C@@H](CC(C)C)C(=O)...,-6.300000,14.765338,14.765338,0.082043,-1.652577,0.137104,27.528571,988.282,...,6.371851,137.310770,1.961582,56.609941,28.393162,28.216778,22212.0,114.0,7.691,348.0


In [142]:
df_2d_test.to_csv('features/Descriptors/Test_2d_all_descriptors_MDCK.csv', index=False)

In [143]:
#2d All descriptors
df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors_MDCK.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_all_descriptors_MDCK.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
# X_test = X_test.select_dtypes(include=['number'])
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 3097)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 3097)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.111437 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8601
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 643
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spli

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.4099,0.5167,0.6402,0.2211,0.4759,0.4402,0.4216,0.5234,0.6493,0.4105,0.7188,0.5785
DecisionTreeRegressor,0.5081,0.5140,0.7128,0.0344,0.4303,0.4612,0.3868,0.5053,0.6219,0.4591,0.7021,0.5344
RandomForestRegressor,0.3347,0.4884,0.5786,0.3638,0.6098,0.6179,0.4080,0.5468,0.6387,0.4295,0.7005,0.4490
GradientBoostingRegressor,0.4037,0.5185,0.6354,0.2328,0.5667,0.5197,0.4270,0.5313,0.6535,0.4029,0.6456,0.5262
AdaBoostRegressor,0.3347,0.4398,0.5785,0.3640,0.6163,0.5934,0.4206,0.5174,0.6485,0.4118,0.6629,0.4849
XGBRegressor,0.3791,0.4879,0.6157,0.2794,0.5641,0.5040,0.4614,0.5474,0.6793,0.3548,0.5982,0.4931
ExtraTreesRegressor,0.2946,0.4264,0.5428,0.4401,0.6740,0.6291,0.4182,0.5101,0.6467,0.4152,0.6568,0.4959
LinearRegression,0.6417,0.6156,0.8011,-0.2196,0.5506,0.5075,0.6044,0.5843,0.7774,0.1549,0.6205,0.7548
KNeighborsRegressor,0.3735,0.4726,0.6112,0.2901,0.5652,0.5392,0.4921,0.5664,0.7015,0.3119,0.5695,0.3931
SVR,0.4113,0.5090,0.6413,0.2183,0.4738,0.5003,0.4351,0.4950,0.6596,0.3916,0.6575,0.5592


In [144]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.17886263740318, -5.324280272622234, -5.959...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.324280272622234, -5.565454977675508, -5.2...","[-5.411886786765182, -5.562464625481077, -5.66...","[0.1456050227116578, 0.17964258932313062, 0.30..."
1,DecisionTreeRegressor,"[-5.65, -5.32, -6.4, -4.73, -6.4, -5.984295768...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.94, -5.65, -4.73, -5.289036881, -5.185752...","[-5.6, -5.784000000000001, -5.6587663138, -5.6...","[0.5797585704411792, 0.42687703147393624, 1.07..."
2,RandomForestRegressor,"[-5.356170970249996, -5.365321796246666, -5.82...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.365565068393336, -5.81604680131, -5.63995...","[-5.509210377441766, -5.7376182062397, -5.8198...","[0.15709987483445678, 0.14145384800368485, 0.1..."
3,GradientBoostingRegressor,"[-5.320484872988993, -5.124919181268304, -6.21...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.827908400984254, -5.755483665191757, -5.0...","[-5.396910157659742, -5.897314106949049, -5.80...","[0.47940176054150757, 0.3199029645187068, 0.58..."
4,AdaBoostRegressor,"[-5.4140063063333335, -5.32, -6.16893911738461...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.65, -6.203333333333333, -6.08945907300000...","[-5.509474598515714, -5.9084619047720235, -5.8...","[0.18185336659768375, 0.2334790294142701, 0.20..."
5,XGBRegressor,"[-5.4613853, -5.352883, -6.1830983, -5.155133,...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.81602, -5.577811, -4.806387, -5.257579, -...","[-5.382535, -5.8049517, -5.505273, -5.5633516,...","[0.5230336, 0.3851455, 0.49738005, 0.3403484, ..."
6,ExtraTreesRegressor,"[-5.3393, -4.881009353709998, -6.1994917570325...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.512178012569998, -5.9925857351125025, -5....","[-5.41571432816, -5.915274865436501, -5.892654...","[0.2416310042163394, 0.225921014396662, 0.1337..."
7,LinearRegression,"[-5.03772376678926, -4.7925398148873946, -5.12...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.686159901399619, -5.930823298174342, -8.5...","[-6.038785240617474, -5.877772648375559, -8.13...","[0.47582168494049637, 0.6865943132612236, 0.47..."
8,KNeighborsRegressor,"[-5.066666666666666, -5.3999999999999995, -5.8...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333333, -5.2...","[-5.713333333333333, -5.713333333333333, -5.20...","[0.1707890186425604, 0.1707890186425604, 0.213..."
9,SVR,"[-5.315312549485516, -5.410555754867083, -5.76...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.544645855615043, -5.862090427428702, -5.6...","[-5.60355561531266, -5.8314324811301415, -5.67...","[0.16621425893702915, 0.2364353119861693, 0.12..."


In [145]:
result_df.to_csv('results/Descriptors/Results_2D_All_desc_MDCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_All_desc_MDCK.csv')

In [146]:
#2d All descriptors const rem
df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors_MDCK.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train, const_col =  remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_all_descriptors_MDCK.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 2274)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 2274)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.096148 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8601
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 643
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spli

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.4099,0.5167,0.6402,0.2211,0.4759,0.4402,0.4216,0.5234,0.6493,0.4105,0.7188,0.5785
DecisionTreeRegressor,0.7516,0.6139,0.8670,-0.4285,0.3820,0.4325,0.4950,0.5707,0.7036,0.3078,0.5581,0.4601
RandomForestRegressor,0.3331,0.4818,0.5771,0.3670,0.6126,0.6090,0.4199,0.5528,0.6480,0.4128,0.6783,0.4435
GradientBoostingRegressor,0.4222,0.5163,0.6498,0.1976,0.5386,0.5070,0.4069,0.5234,0.6379,0.4310,0.6722,0.5234
AdaBoostRegressor,0.3592,0.4502,0.5993,0.3173,0.5873,0.5371,0.4466,0.5278,0.6683,0.3754,0.6241,0.4325
XGBRegressor,0.3791,0.4879,0.6157,0.2794,0.5641,0.5040,0.4614,0.5474,0.6793,0.3548,0.5982,0.4931
ExtraTreesRegressor,0.3073,0.4332,0.5544,0.4159,0.6580,0.5992,0.4134,0.5145,0.6430,0.4219,0.6652,0.5262
LinearRegression,0.6417,0.6156,0.8011,-0.2196,0.5506,0.5075,0.6044,0.5843,0.7774,0.1549,0.6205,0.7548
KNeighborsRegressor,0.3735,0.4726,0.6112,0.2901,0.5652,0.5392,0.4921,0.5664,0.7015,0.3119,0.5695,0.3931
SVR,0.4113,0.5090,0.6413,0.2183,0.4738,0.5003,0.4351,0.4950,0.6596,0.3916,0.6575,0.5592


In [147]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.17886263740318, -5.324280272622234, -5.959...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.324280272622234, -5.565454977675508, -5.2...","[-5.411886786765182, -5.562464625481077, -5.66...","[0.1456050227116578, 0.17964258932313062, 0.30..."
1,DecisionTreeRegressor,"[-5.65, -5.32, -6.05, -4.73, -7.698970004, -5....",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.94, -5.65, -4.73, -5.289036881, -5.185752...","[-5.49, -5.754, -5.5622822246, -5.862161628400...","[0.5326912801989534, 0.47504105085771275, 1.05..."
2,RandomForestRegressor,"[-5.340934707187996, -5.396886978792221, -5.81...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.363317946127905, -5.811217583318572, -5.5...","[-5.461616911215349, -5.740654089677216, -5.76...","[0.13616789114563754, 0.1428206337697975, 0.10..."
3,GradientBoostingRegressor,"[-5.408782052039135, -5.1244027298995105, -6.2...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.826190735676328, -5.757353190020055, -5.0...","[-5.400559299529254, -5.906001045902572, -5.76...","[0.46257376838985265, 0.3130235357604289, 0.46..."
4,AdaBoostRegressor,"[-5.337027134666666, -5.237332392000001, -6.14...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.372950034470588, -6.224, -5.98668855725, ...","[-5.369601846812905, -5.848828896714176, -5.93...","[0.2682463357163755, 0.2760953365369655, 0.193..."
5,XGBRegressor,"[-5.4613853, -5.352883, -6.1830983, -5.155133,...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.81602, -5.577811, -4.806387, -5.257579, -...","[-5.382535, -5.8049517, -5.505273, -5.5633516,...","[0.5230336, 0.3851455, 0.49738005, 0.3403484, ..."
6,ExtraTreesRegressor,"[-5.305447274950001, -4.9000499999999985, -6.2...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.527299999999999, -6.007247274949999, -5.7...","[-5.4553072616415, -5.956181547328, -5.8250799...","[0.24800475130775518, 0.25618526849102746, 0.1..."
7,LinearRegression,"[-5.037723766789253, -4.792539814887399, -5.12...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.686159901399612, -5.930823298174338, -8.5...","[-6.038785240617473, -5.877772648375559, -8.13...","[0.4758216849404947, 0.6865943132612251, 0.471..."
8,KNeighborsRegressor,"[-5.066666666666666, -5.3999999999999995, -5.8...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333333, -5.2...","[-5.713333333333333, -5.713333333333333, -5.20...","[0.1707890186425604, 0.1707890186425604, 0.213..."
9,SVR,"[-5.3153361978013, -5.410537290915246, -5.7655...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.544683332237691, -5.8621453706321915, -5....","[-5.603568856420774, -5.831457651894695, -5.67...","[0.16620822017865106, 0.23644277378544987, 0.1..."


In [148]:
result_df.to_csv('results/Descriptors/Results_2D_All_desc_const_rem_MDCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_All_desc_const_rem_MDCK.csv')

In [149]:
#2d All descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors_MDCK.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train, const_col = remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_all_descriptors_MDCK.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 1534)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 1534)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.078623 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5543
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 417
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spli

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3933,0.5254,0.6271,0.2526,0.5069,0.4386,0.4238,0.5390,0.6510,0.4074,0.7227,0.5978
DecisionTreeRegressor,0.7100,0.6350,0.8426,-0.3493,0.4248,0.3161,0.5541,0.6485,0.7444,0.2252,0.5156,0.4601
RandomForestRegressor,0.3734,0.5188,0.6111,0.2903,0.5538,0.5613,0.4274,0.5589,0.6538,0.4023,0.6781,0.5096
GradientBoostingRegressor,0.4156,0.5130,0.6446,0.2102,0.5592,0.5024,0.4229,0.5157,0.6503,0.4087,0.6619,0.6171
AdaBoostRegressor,0.2997,0.4128,0.5475,0.4304,0.6594,0.6374,0.4199,0.5106,0.6480,0.4129,0.6615,0.5372
XGBRegressor,0.5808,0.5488,0.7621,-0.1038,0.4007,0.4593,0.4503,0.5597,0.6710,0.3703,0.6267,0.6336
ExtraTreesRegressor,0.2707,0.4225,0.5203,0.4855,0.7030,0.6214,0.4210,0.5100,0.6489,0.4113,0.6539,0.5207
LinearRegression,0.7161,0.6443,0.8462,-0.3610,0.5009,0.4670,0.7401,0.6689,0.8603,-0.0349,0.6151,0.7328
KNeighborsRegressor,0.3719,0.4712,0.6099,0.2932,0.5672,0.5496,0.4758,0.5532,0.6898,0.3347,0.5938,0.4242
SVR,0.4283,0.5200,0.6545,0.1859,0.4389,0.4848,0.4335,0.4890,0.6584,0.3938,0.6628,0.5647


In [150]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.193297965477244, -5.535778042677406, -5.62...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.280975908332179, -5.606666342554522, -6.3...","[-5.440945763707487, -5.656541463991928, -6.01...","[0.21467829601382854, 0.2633408594459569, 0.23..."
1,DecisionTreeRegressor,"[-4.73, -4.58, -6.05, -5.585026652, -7.6989700...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.65, -6.25, -4.73, -5.289036881, -5.585026...","[-5.256, -6.164, -5.8934853178000015, -5.77102...","[0.6359119435896765, 0.17199999999999988, 0.99..."
2,RandomForestRegressor,"[-5.259955065509996, -5.438957058236665, -5.68...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.391750643003808, -5.8739033408766685, -5....","[-5.529480148290924, -5.859056492409467, -6.03...","[0.18550763138877954, 0.14017505706890707, 0.1..."
3,GradientBoostingRegressor,"[-5.6233297161113125, -5.040288634903594, -6.1...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.619707864281258, -6.171218440395044, -5.4...","[-5.636032655158989, -6.1199005740464525, -6.2...","[0.3297307356375742, 0.1939326759889041, 0.622..."
4,AdaBoostRegressor,"[-5.413333333333333, -5.211587863333333, -5.65...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.168332978187499, -6.160859153599999, -6.0...","[-5.427730870615834, -5.849124505100001, -6.10...","[0.20500176705414763, 0.23500906606944566, 0.0..."
5,XGBRegressor,"[-5.4807277, -5.1154923, -5.8696904, -5.702743...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.905909, -6.276088, -5.561126, -5.591366, ...","[-5.592835, -6.1083403, -6.070587, -5.917414, ...","[0.3811348, 0.12946336, 0.27048436, 0.68297535..."
6,ExtraTreesRegressor,"[-5.212347274949999, -5.153131544539998, -6.02...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.3139, -6.031693724420002, -5.949513952359...","[-5.363313601572665, -5.944100352017668, -6.02...","[0.2258846254569773, 0.2949060136438629, 0.119..."
7,LinearRegression,"[-4.764205949456107, -4.453115786363093, -5.18...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.4815263273160495, -5.551793313972038, -9....","[-7.015143775715265, -5.568001290373657, -8.31...","[1.2070957156418478, 0.9093877853377796, 1.012..."
8,KNeighborsRegressor,"[-5.066666666666666, -5.3999999999999995, -5.8...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333333, -5.2...","[-5.713333333333333, -5.6546666666666665, -5.2...","[0.1707890186425604, 0.14230796026770795, 0.18..."
9,SVR,"[-5.3116995933048265, -5.383185674997991, -5.7...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.576204313249687, -5.864939468536933, -5.6...","[-5.630772942016698, -5.809290640312082, -5.72...","[0.14846822682901972, 0.251050152746994, 0.132..."


In [151]:
result_df.to_csv('results/Descriptors/Results_2D_All_desc_LVR_MDCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_All_desc_LVR_MDCK.csv')

In [152]:
#2d All descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors_MDCK.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train, const_col = remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_all_descriptors_MDCK.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 1534)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 1534)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.126431 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5543
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 417
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spli

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3933,0.5254,0.6271,0.2526,0.5069,0.4386,0.4238,0.5390,0.6510,0.4074,0.7227,0.5978
DecisionTreeRegressor,0.7100,0.6350,0.8426,-0.3493,0.4248,0.3161,0.5541,0.6485,0.7444,0.2252,0.5156,0.4601
RandomForestRegressor,0.3734,0.5188,0.6111,0.2903,0.5538,0.5613,0.4274,0.5589,0.6538,0.4023,0.6781,0.5096
GradientBoostingRegressor,0.4156,0.5130,0.6446,0.2102,0.5592,0.5024,0.4229,0.5157,0.6503,0.4087,0.6619,0.6171
AdaBoostRegressor,0.2997,0.4128,0.5475,0.4304,0.6594,0.6374,0.4199,0.5106,0.6480,0.4129,0.6615,0.5372
XGBRegressor,0.5808,0.5488,0.7621,-0.1038,0.4007,0.4593,0.4503,0.5597,0.6710,0.3703,0.6267,0.6336
ExtraTreesRegressor,0.2707,0.4225,0.5203,0.4855,0.7030,0.6214,0.4210,0.5100,0.6489,0.4113,0.6539,0.5207
LinearRegression,0.7161,0.6443,0.8462,-0.3610,0.5009,0.4670,0.7401,0.6689,0.8603,-0.0349,0.6151,0.7328
KNeighborsRegressor,0.3719,0.4712,0.6099,0.2932,0.5672,0.5496,0.4758,0.5532,0.6898,0.3347,0.5938,0.4242
SVR,0.4283,0.5200,0.6545,0.1859,0.4389,0.4848,0.4335,0.4890,0.6584,0.3938,0.6628,0.5647


In [153]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.193297965477244, -5.535778042677406, -5.62...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.280975908332179, -5.606666342554522, -6.3...","[-5.440945763707487, -5.656541463991928, -6.01...","[0.21467829601382854, 0.2633408594459569, 0.23..."
1,DecisionTreeRegressor,"[-4.73, -4.58, -6.05, -5.585026652, -7.6989700...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.65, -6.25, -4.73, -5.289036881, -5.585026...","[-5.256, -6.164, -5.8934853178000015, -5.77102...","[0.6359119435896765, 0.17199999999999988, 0.99..."
2,RandomForestRegressor,"[-5.259955065509996, -5.438957058236665, -5.68...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.391750643003808, -5.8739033408766685, -5....","[-5.529480148290924, -5.859056492409468, -6.03...","[0.18550763138877968, 0.14017505706890723, 0.1..."
3,GradientBoostingRegressor,"[-5.6233297161113125, -5.040288634903594, -6.1...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.619707864281258, -6.171218440395044, -5.4...","[-5.636032655158989, -6.1199005740464525, -6.2...","[0.3297307356375742, 0.1939326759889041, 0.622..."
4,AdaBoostRegressor,"[-5.413333333333333, -5.211587863333333, -5.65...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.168332978187499, -6.160859153599999, -6.0...","[-5.427730870615834, -5.849124505100001, -6.10...","[0.20500176705414763, 0.23500906606944566, 0.0..."
5,XGBRegressor,"[-5.4807277, -5.1154923, -5.8696904, -5.702743...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.905909, -6.276088, -5.561126, -5.591366, ...","[-5.592835, -6.1083403, -6.070587, -5.917414, ...","[0.3811348, 0.12946336, 0.27048436, 0.68297535..."
6,ExtraTreesRegressor,"[-5.212347274949999, -5.153131544539998, -6.02...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.3138999999999985, -6.031693724420002, -5....","[-5.363313601572665, -5.944100352017668, -6.02...","[0.22588462545697738, 0.2949060136438629, 0.11..."
7,LinearRegression,"[-4.764205949456107, -4.453115786363093, -5.18...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.4815263273160495, -5.551793313972038, -9....","[-7.015143775715265, -5.568001290373657, -8.31...","[1.2070957156418478, 0.9093877853377796, 1.012..."
8,KNeighborsRegressor,"[-5.066666666666666, -5.3999999999999995, -5.8...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333333, -5.2...","[-5.713333333333333, -5.6546666666666665, -5.2...","[0.1707890186425604, 0.14230796026770795, 0.18..."
9,SVR,"[-5.3116995933048265, -5.383185674997991, -5.7...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.576204313249687, -5.864939468536933, -5.6...","[-5.630772942016698, -5.809290640312082, -5.72...","[0.14846822682901972, 0.251050152746994, 0.132..."


In [154]:
def features(df, target_column='Permeability', threshold=0.9):
    correlation_matrix = df.corr()
    
    features_to_drop = set()
    
    for feature in correlation_matrix.columns:
        if feature == target_column:
            continue 
        target_corr = correlation_matrix[target_column][feature]
        
        for other_feature in correlation_matrix.columns:
            if other_feature == feature or other_feature == target_column:
                continue
            
            if abs(correlation_matrix[feature][other_feature]) > threshold:
                other_target_corr = correlation_matrix[target_column][other_feature]

                if abs(other_target_corr) < abs(target_corr):
                    features_to_drop.add(other_feature)
                else:
                    features_to_drop.add(feature)
    selected_features = [col for col in df.columns if col not in features_to_drop and col != target_column]
    
    return selected_features

In [155]:
def remove_low_variance_columns(df, threshold=0.005):
    # df = df.drop(['ID','SMILES','Permeability'],axis=1)
    variances = df.var()
    
    low_variance_columns = variances[variances < threshold].index.tolist()
    
    df_cleaned = df.drop(columns=low_variance_columns)
    
    return df_cleaned, low_variance_columns

In [156]:
df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors_MDCK.csv')
df_train =df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
X_train = df_train[selected_features] 
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_all_descriptors_MDCK.csv')
df_test =df_test.dropna()
X_test =  df_test[X_train.columns]
y_test =  df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 117)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 117)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.114308 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 506
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 37
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits w

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


0.16733882346672302


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3748,0.4948,0.6122,0.2878,0.5398,0.5426,0.4175,0.4948,0.6462,0.4162,0.7307,0.5840
DecisionTreeRegressor,0.8636,0.6501,0.9293,-0.6412,0.3041,0.3343,0.4001,0.5037,0.6326,0.4405,0.6822,0.6584
RandomForestRegressor,0.3372,0.4778,0.5807,0.3592,0.6076,0.6102,0.3907,0.5047,0.6251,0.4536,0.7189,0.5620
GradientBoostingRegressor,0.3976,0.4884,0.6306,0.2443,0.5946,0.5192,0.3693,0.4832,0.6077,0.4836,0.7266,0.6612
AdaBoostRegressor,0.2868,0.4182,0.5355,0.4549,0.6812,0.5996,0.4199,0.5085,0.6480,0.4128,0.6622,0.4876
XGBRegressor,0.3626,0.4799,0.6022,0.3108,0.5804,0.5264,0.3581,0.4617,0.5984,0.4993,0.7358,0.6915
ExtraTreesRegressor,0.2842,0.4491,0.5331,0.4599,0.6861,0.6242,0.4008,0.4845,0.6331,0.4396,0.6838,0.6033
LinearRegression,0.8407,0.6964,0.9169,-0.5978,0.5642,0.5049,1.3907,0.7864,1.1793,-0.9447,0.5372,0.6749
KNeighborsRegressor,0.3928,0.5055,0.6267,0.2536,0.5195,0.4762,0.4258,0.5078,0.6525,0.4046,0.6611,0.5255
SVR,0.3948,0.4856,0.6283,0.2497,0.5055,0.5270,0.4371,0.5047,0.6611,0.3888,0.6561,0.5179


In [157]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.417325333566333, -5.6148450703870205, -5.6...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.417325333566333, -5.417325333566333, -6.3...","[-5.488209414025517, -5.519455192884415, -6.12...","[0.20355339378837395, 0.22883480997439007, 0.1..."
1,DecisionTreeRegressor,"[-5.65, -5.32, -6.05, -5.585026652, -7.6989700...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.25, -6.25, -5.65, -5.289036881, -5.585026...","[-5.958, -5.62548837, -5.6877926416, -5.441229...","[0.24636558201177358, 0.5401143861800538, 0.29..."
2,RandomForestRegressor,"[-5.4180054735779954, -5.188139700039998, -5.5...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.426146186532, -5.804883892846666, -5.8601...","[-5.5894362406198, -5.786507413171968, -5.9962...","[0.12921485717016817, 0.10280655690932591, 0.1..."
3,GradientBoostingRegressor,"[-5.606933433790057, -5.097190470740296, -5.97...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.820095826128992, -6.1847317427127555, -6....","[-5.840796048863632, -5.9796347296574215, -6.3...","[0.2340824756293014, 0.26407962925425044, 0.33..."
4,AdaBoostRegressor,"[-5.511884436749999, -4.94, -6.170252537409092...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.1885952525, -5.710761347692308, -5.676028...","[-5.438356459098053, -5.8157835185170335, -5.9...","[0.26159803172522517, 0.18535620832121846, 0.2..."
5,XGBRegressor,"[-5.6188526, -5.5541134, -5.784735, -5.435798,...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.947677, -6.1050835, -5.8008127, -5.644912...","[-5.8864694, -5.8466787, -6.022241, -5.457515,...","[0.2296419, 0.30284426, 0.32961917, 0.16435511..."
6,ExtraTreesRegressor,"[-5.445804998999998, -5.147127429112498, -5.82...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.6025949108500015, -6.122393603649999, -6....","[-5.607795232191, -6.006444182309332, -6.14486...","[0.21765226229635284, 0.2696501510256584, 0.08..."
7,LinearRegression,"[-4.790585760497281, -4.83176895031884, -5.233...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-10.0, -5.918363325633983, -6.96469703938048...","[-9.675291624412043, -5.824696844499735, -7.07...","[0.6494167511759116, 0.9424054261931016, 1.198..."
8,KNeighborsRegressor,"[-5.613333333333333, -5.3999999999999995, -5.8...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333334, -5.9...","[-5.6259999999999994, -5.6546666666666665, -5....","[0.16096100286853474, 0.14230796026770784, 0.1..."
9,SVR,"[-5.481751179956121, -5.264961303592915, -5.66...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.609319772973054, -5.991595985583899, -5.7...","[-5.5851870656538285, -5.870481347690559, -5.6...","[0.069210040334638, 0.27142064691190526, 0.073..."


In [158]:
result_df.to_csv('results/Descriptors/Results_2D_All_desc_LVR_remove_corr_features_MDCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_All_desc_LVRremove_corr_features_MDCK.csv')

In [159]:
#3d RDKit descriptors
df_train = pd.read_csv('features/Descriptors/Train_3d_RDKit_desc_MDCK.csv')
df_train = df_train.fillna(0)
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_3d_RDKit_desc_MDCK.csv')
df_test = df_test.fillna(0)
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 11)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 11)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.108083 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 15
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 1
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with 

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.5707,0.6234,0.7554,-0.0846,-0.1359,-0.1264,0.6224,0.6657,0.7889,0.1297,0.5669,0.4938
DecisionTreeRegressor,0.9946,0.8268,0.9973,-0.8902,0.1653,0.1587,0.8170,0.7172,0.9039,-0.1425,0.1419,0.1515
RandomForestRegressor,0.5983,0.6177,0.7735,-0.1370,0.0851,0.0372,0.5982,0.6453,0.7734,0.1636,0.4200,0.2094
GradientBoostingRegressor,0.6725,0.6759,0.8201,-0.2782,0.1000,0.0806,0.7038,0.6880,0.8389,0.0159,0.2160,0.2452
AdaBoostRegressor,0.6167,0.6555,0.7853,-0.1721,0.0520,-0.0106,0.6757,0.6871,0.8220,0.0551,0.2580,0.2259
XGBRegressor,0.7996,0.7050,0.8942,-0.5197,-0.0128,0.0132,0.6300,0.6616,0.7938,0.1190,0.3679,0.3361
ExtraTreesRegressor,0.6062,0.6320,0.7786,-0.1521,0.1689,0.1592,0.6802,0.6678,0.8247,0.0489,0.2885,0.2479
LinearRegression,0.6679,0.6795,0.8173,-0.2694,0.1009,0.0715,0.9672,0.8101,0.9835,-0.3525,-0.0302,0.0386
KNeighborsRegressor,0.6613,0.6740,0.8132,-0.2569,0.0892,0.1047,0.7144,0.6680,0.8452,0.0011,0.2142,0.1983
SVR,0.5395,0.6084,0.7345,-0.0253,0.2001,0.2184,0.6230,0.6485,0.7893,0.1288,0.3821,0.1846


In [160]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.551606744545717, -5.672979576159265, -5.55...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.551606744545717, -5.551606744545717, -5.5...","[-5.732715898412391, -5.667140648413334, -5.54...","[0.1993677206308215, 0.16553490350184352, 0.10..."
1,DecisionTreeRegressor,"[-4.82, -6.4, -5.32, -5.22, -5.22, -5.58502665...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.65, -6.3, -5.65, -6.321123049, -4.82, -5....","[-5.276000000000001, -5.218, -5.402, -5.986109...","[0.4628433860389495, 0.5632903336646207, 0.522..."
2,RandomForestRegressor,"[-5.474970703149999, -5.939291568709995, -5.43...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.4276937856199945, -5.7070402273699985, -5...","[-5.5007154512979985, -5.6325601659719995, -5....","[0.14183512920650856, 0.1804745975931289, 0.02..."
3,GradientBoostingRegressor,"[-5.65061693355481, -6.0694584113874015, -5.40...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.358902710164112, -5.577329042768232, -5.7...","[-5.353732266200677, -5.482762440314313, -5.41...","[0.19122827283691385, 0.4612642160901979, 0.32..."
4,AdaBoostRegressor,"[-5.869230769230769, -5.869230769230769, -5.48...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.531620521666667, -5.802652272307693, -5.6...","[-5.481295594331327, -5.702859370400587, -5.53...","[0.10175230801654228, 0.23033753575208887, 0.0..."
5,XGBRegressor,"[-5.5658355, -5.6945286, -5.4065337, -4.932633...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.5287833, -5.729893, -5.651846, -6.464418,...","[-5.4274573, -5.5588236, -5.2612424, -5.930664...","[0.31325763, 0.33557042, 0.25263372, 0.2725702..."
6,ExtraTreesRegressor,"[-5.423910012939997, -5.801987143980001, -5.35...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.497969144419994, -5.755670794779998, -5.3...","[-5.366340789715998, -5.725900910125999, -5.28...","[0.2404920393967082, 0.20348453967391603, 0.11..."
7,LinearRegression,"[-4.945319503908539, -5.432263773902517, -6.04...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.297294114018479, -5.10149661562284, -5.25...","[-4.267034796893176, -5.280348088190523, -5.10...","[0.2116151725913957, 0.25423615767819424, 0.36..."
8,KNeighborsRegressor,"[-5.316666666666667, -6.184008911333334, -5.89...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.493333333333333, -5.571587863333334, -5.8...","[-5.365333333333334, -5.657367732933333, -5.74...","[0.07626270385975048, 0.16252025590782396, 0.1..."
9,SVR,"[-5.227052082737401, -5.828439814355429, -5.59...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.723406728424486, -5.498092431404659, -5.6...","[-5.510157118723937, -5.634145443544243, -5.52...","[0.18707117031075732, 0.1470399721901797, 0.16..."


In [161]:
result_df.to_csv('results/Descriptors/Results_3D_RDKit_desc_MDCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_3D_RDKit_desc_MDCK.csv')

In [162]:
#3d Padel descriptors
df_train = pd.read_csv('features/Descriptors/Train_3d_padel_curated_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_3d_padel_curated_MDCK.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 431)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 431)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.103083 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1695
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 113
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.4670,0.5272,0.6834,0.1125,0.3657,0.4367,0.4394,0.5216,0.6629,0.3855,0.6745,0.6749
DecisionTreeRegressor,1.0782,0.8057,1.0384,-1.0492,0.0104,0.0557,0.5608,0.5715,0.7489,0.2158,0.5285,0.5620
RandomForestRegressor,0.5050,0.5608,0.7107,0.0402,0.3204,0.3190,0.4232,0.4843,0.6505,0.4083,0.6736,0.6253
GradientBoostingRegressor,0.6370,0.6230,0.7981,-0.2106,0.1852,0.2375,0.4185,0.4197,0.6469,0.4148,0.6503,0.6612
AdaBoostRegressor,0.5021,0.5704,0.7086,0.0458,0.3334,0.3587,0.4162,0.4320,0.6451,0.4180,0.6908,0.6474
XGBRegressor,0.6240,0.6730,0.7899,-0.1860,0.2484,0.2543,0.4470,0.4413,0.6686,0.3749,0.6212,0.6915
ExtraTreesRegressor,0.4922,0.5811,0.7016,0.0646,0.3801,0.3554,0.4046,0.4715,0.6361,0.4343,0.6778,0.6116
LinearRegression,1.3096,0.9443,1.1444,-1.4889,0.1293,0.1655,0.6580,0.6148,0.8112,0.0799,0.5957,0.6749
KNeighborsRegressor,0.6154,0.6135,0.7845,-0.1697,0.2673,0.3707,0.4933,0.5980,0.7023,0.3102,0.5602,0.4270
SVR,0.5004,0.5690,0.7074,0.0489,0.2690,0.2218,0.4220,0.4966,0.6496,0.4100,0.7526,0.7328


In [163]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.246797503393029, -5.161982855255484, -5.02...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.496304172780948, -5.149596643381521, -5.5...","[-5.340693483156313, -5.200453389249764, -5.85...","[0.1267119874406526, 0.10107930409385539, 0.19..."
1,DecisionTreeRegressor,"[-6.25, -5.32, -6.400000000000001, -5.51, -5.3...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.65, -5.32, -6.25, -5.185752404, -5.289036...","[-5.4698073762, -5.523807376200001, -6.9820679...","[0.3165272389285214, 0.4588183696781475, 0.841..."
2,RandomForestRegressor,"[-5.23876252166, -5.3184487849, -6.02496194288...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.30368974484, -5.452169522199999, -5.75990...","[-5.395350430486, -5.466657778819999, -6.01328...","[0.12144223408632773, 0.08101951433005579, 0.1..."
3,GradientBoostingRegressor,"[-5.061758958091732, -5.285702377594746, -6.10...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.403355204366959, -5.478632308288314, -5.7...","[-5.351341940901965, -5.392880581564958, -6.26...","[0.1424228925893085, 0.16422308790425977, 0.31..."
4,AdaBoostRegressor,"[-5.2198750516666665, -5.32, -6.10077249675, -...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.016860338857143, -5.520768271333334, -5.7...","[-5.311301370504762, -5.387760993953524, -6.04...","[0.29770864402612207, 0.17252446160203558, 0.1..."
5,XGBRegressor,"[-5.074493, -5.243013, -6.2975116, -5.659368, ...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.436323, -5.3146443, -5.2889743, -5.432026...","[-5.2401934, -5.2186317, -6.169871, -5.7114906...","[0.15122977, 0.18593703, 0.49911273, 0.3392453..."
6,ExtraTreesRegressor,"[-5.239672687159997, -5.5772076578, -6.0622649...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.167810533690004, -5.554167208880001, -5.9...","[-5.333338132174002, -5.524451015235999, -6.22...","[0.17563720997771626, 0.07403153589805866, 0.1..."
7,LinearRegression,"[-4.994591079308595, -4.592438164716916, -4.24...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.30480930192413, -4.146222199389969, -7.99...","[-5.2148533399550745, -4.574085165324054, -7.6...","[0.5023519647231425, 0.452731294024605, 0.5369..."
8,KNeighborsRegressor,"[-5.444909164999999, -5.3999999999999995, -5.3...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.503333333333334, -5.613333333333333, -5.5...","[-5.508, -5.713333333333333, -5.61533333333333...","[0.0290287214094362, 0.1707890186425604, 0.157..."
9,SVR,"[-5.239530117062856, -5.303417277945989, -5.56...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.482317738929968, -5.290207538047728, -5.7...","[-5.484684077298157, -5.400686269139815, -5.73...","[0.11068793933091971, 0.1446802329802815, 0.08..."


In [164]:
result_df.to_csv('results/Descriptors/Results_3D_padel_desc_MDCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_3D_padel_desc_MDCK.csv')

In [165]:
df_train_rdkit = pd.read_csv('features/Descriptors/Train_3d_RDKit_desc_MDCK.csv')
df_train_rdkit = df_train_rdkit.fillna(0)
df_train_padel = pd.read_csv('features/Descriptors/Train_3d_padel_curated_MDCK.csv')

df_3d_descriptors = df_train_rdkit.merge(df_train_padel, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_3d_descriptors

,ID,SMILES,Permeability,3d_rdkit_1,3d_rdkit_2,3d_rdkit_3,3d_rdkit_4,3d_rdkit_5,3d_rdkit_6,3d_rdkit_7,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,1114,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-4.940000,24313.755201,30312.376753,46681.937558,0.520839,0.649338,6.667315,0.000027,...,0.490993,0.446412,0.502821,0.521474,0.367154,50.115915,697.880017,2474.945996,0.406107,1.391449
1,1113,CC(C)C[C@@H]1NC(=O)[C@@H]2CCCN2C(=O)[C@@H](Cc2...,-5.820000,20729.202872,33683.486578,45247.431206,0.458130,0.744429,6.612865,0.000036,...,0.530772,0.398556,0.543790,0.527114,0.372276,47.613373,628.467138,2289.825818,0.393991,1.443180
2,1117,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H]2CCCN...,-5.650000,20665.252633,38087.030609,54414.093500,0.379778,0.699948,7.046731,0.000034,...,0.527383,0.387331,0.523372,0.543333,0.399556,48.105086,653.233321,2640.699755,0.372071,1.466261
3,1119,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-6.250000,23316.296843,27342.847641,44212.952948,0.527363,0.618435,6.521161,0.000027,...,0.528976,0.349388,0.516241,0.483660,0.323140,49.516594,715.115788,3493.983049,0.317545,1.323041
4,2428,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N(C)[C@@H](C)C(=...,-5.510000,17548.732916,32062.570730,44521.068339,0.394167,0.720166,6.566914,0.000041,...,0.506249,0.395804,0.498185,0.483571,0.313967,46.999494,637.788214,2722.371988,0.353079,1.295723
5,2446,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N2CCC[C@@H]2C(=O...,-6.400000,19844.954604,31315.006377,42161.230503,0.470692,0.742744,6.538558,0.000037,...,0.465091,0.397955,0.535696,0.559095,0.348772,46.003992,641.858241,3155.799517,0.294568,1.443562
6,2445,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N2CCC[C@@H]2C(=O...,-4.820000,20353.342160,32484.579946,45010.234764,0.452194,0.721715,6.701328,0.000035,...,0.493305,0.428676,0.535662,0.498539,0.389916,53.087085,798.688967,3320.153449,0.382972,1.424116
7,2427,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N(C)[C@@H](C)C(=...,-4.610000,20484.932132,27064.776904,40541.969906,0.505277,0.667574,6.358461,0.000033,...,0.483498,0.392982,0.564432,0.505432,0.322603,48.069815,689.212021,3344.170973,0.314720,1.392467
8,8145,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.744727,21947.274238,24319.601060,39677.159260,0.553146,0.612937,6.402193,0.000028,...,0.565467,0.377819,0.494304,0.444191,0.445117,47.071607,591.915175,1902.720960,0.414930,1.383611
9,1107,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](C)N...,-4.610000,19402.624738,30740.712404,46314.848977,0.418929,0.663733,6.785633,0.000034,...,0.633354,0.305158,0.532685,0.549858,0.399249,48.688812,594.973414,2015.324533,0.450032,1.481792


In [166]:
nan_rows = df_3d_descriptors[df_3d_descriptors.isna().any(axis=1)]
nan_rows

,ID,SMILES,Permeability,3d_rdkit_1,3d_rdkit_2,3d_rdkit_3,3d_rdkit_4,3d_rdkit_5,3d_rdkit_6,3d_rdkit_7,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds


In [167]:
df_3d_descriptors.to_csv('features/Descriptors/Train_3d_all_descriptors_MDCK.csv', index=False)

In [168]:
df_test_rdkit = pd.read_csv('features/Descriptors/Test_3d_RDKit_desc_MDCK.csv')
df_test_rdkit = df_test_rdkit.fillna(0)
df_test_padel = pd.read_csv('features/Descriptors/Test_3d_padel_curated_MDCK.csv')

df_3d_descriptors = df_test_rdkit.merge(df_test_padel, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_3d_descriptors

,ID,SMILES,Permeability,3d_rdkit_1,3d_rdkit_2,3d_rdkit_3,3d_rdkit_4,3d_rdkit_5,3d_rdkit_6,3d_rdkit_7,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,1120,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-6.300000,19802.437352,46153.644559,61555.468385,0.321701,0.749790,7.290132,0.000038,...,0.517052,0.333526,0.520259,0.510811,0.420493,49.703108,739.996624,3953.646545,0.275867,1.451563
1,1118,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](Cc2...,-5.350000,19361.572110,31647.602403,40164.602400,0.482056,0.787948,6.392793,0.000041,...,0.544737,0.347757,0.647006,0.459842,0.383878,47.483083,643.440777,2871.212438,0.338740,1.490726
2,1121,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-6.200000,15403.864354,37701.556329,45366.507814,0.339543,0.831044,6.741393,0.000054,...,0.569544,0.319304,0.527342,0.614802,0.396168,50.379698,712.333697,3347.440319,0.354317,1.538312
3,8133,CCC[C@@H]1NC(=O)CN(CC)C(=O)[C@H](CC(C)C)NC(=O)...,-5.355561,19224.894748,23386.993074,35959.413902,0.534628,0.650372,6.162796,0.000034,...,0.575816,0.391394,0.465474,0.444791,0.411105,54.943002,776.072156,2056.663536,0.450816,1.321370
4,8143,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.406714,21673.942645,26981.976759,44843.056097,0.483329,0.601698,6.762160,0.000028,...,0.521440,0.411553,0.528651,0.525749,0.441578,46.170332,590.730936,2052.174504,0.399489,1.495978
5,8119,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.073658,16488.706850,25907.929012,36349.761891,0.453613,0.712740,6.248809,0.000043,...,0.544048,0.342655,0.553639,0.636460,0.287638,45.060837,582.506417,2560.016441,0.330056,1.477737
6,6496,CC(=O)N1CCC[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N[...,-4.860000,12689.812239,33609.441742,41251.160369,0.307623,0.814751,6.608658,0.000064,...,0.644354,0.296155,0.470177,0.475961,0.342756,49.058557,593.936407,1983.399773,0.466531,1.288893
7,8168,CCOC(=O)[C@@H]1CSCC(=O)N[C@@H](CC)C(=O)N(C)[C@...,-6.031517,19158.684524,25259.129336,41368.456209,0.463123,0.610589,6.561346,0.000032,...,0.567029,0.400623,0.475788,0.516266,0.353488,52.551241,713.790090,1832.795103,0.451477,1.345542
8,8345,CCOC(=O)[C@@H]1CSCC(=O)N[C@@H](CC)C(=O)N(CC(C)...,-7.534591,20296.552958,28218.351312,46436.625133,0.437081,0.607674,6.902955,0.000030,...,0.532833,0.379510,0.478903,0.524793,0.340577,45.177438,575.947872,2255.562804,0.368514,1.344273
9,6423,CC(=O)N1CCC[C@@H]1C(=O)N(C)[C@@H](CC(C)C)C(=O)...,-6.300000,14710.113691,30244.589666,41331.345120,0.355907,0.731759,6.607160,0.000050,...,0.616065,0.311878,0.435148,0.417772,0.254081,49.869778,644.135872,2411.114189,0.424098,1.107002


In [169]:
nan_rows = df_3d_descriptors[df_3d_descriptors.isna().any(axis=1)]
nan_rows

,ID,SMILES,Permeability,3d_rdkit_1,3d_rdkit_2,3d_rdkit_3,3d_rdkit_4,3d_rdkit_5,3d_rdkit_6,3d_rdkit_7,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds


In [170]:
df_3d_descriptors.to_csv('features/Descriptors/Test_3d_all_descriptors_MDCK.csv', index=False)

In [171]:
#3d All descriptors
df_train = pd.read_csv('features/Descriptors/Train_3d_all_descriptors_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_3d_all_descriptors_MDCK.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models_3dall = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models_3dall, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 442)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 442)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.122593 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1710
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 114
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.4737,0.5311,0.6883,0.0997,0.3542,0.4258,0.4350,0.5202,0.6595,0.3917,0.6782,0.6749
DecisionTreeRegressor,1.1445,0.8521,1.0698,-1.1752,-0.0053,0.0697,0.4980,0.4926,0.7057,0.3037,0.5711,0.6198
RandomForestRegressor,0.5127,0.5612,0.7161,0.0255,0.3075,0.3214,0.3915,0.4687,0.6257,0.4525,0.7196,0.7052
GradientBoostingRegressor,0.6066,0.6282,0.7789,-0.1529,0.1967,0.2040,0.4216,0.4077,0.6493,0.4105,0.6487,0.6336
AdaBoostRegressor,0.5316,0.5703,0.7291,-0.0103,0.2674,0.3140,0.4201,0.4450,0.6481,0.4126,0.6798,0.7080
XGBRegressor,0.6723,0.6747,0.8199,-0.2777,0.2306,0.2663,0.4548,0.4643,0.6744,0.3640,0.6105,0.6915
ExtraTreesRegressor,0.4918,0.5704,0.7013,0.0654,0.3593,0.3506,0.4048,0.4723,0.6363,0.4339,0.6768,0.6722
LinearRegression,1.1566,0.8931,1.0754,-1.1981,0.1947,0.2327,0.5399,0.5819,0.7348,0.2450,0.6851,0.7300
KNeighborsRegressor,0.5525,0.6025,0.7433,-0.0501,0.3483,0.3923,0.4879,0.5897,0.6985,0.3177,0.5694,0.4876
SVR,0.4838,0.5603,0.6956,0.0805,0.3074,0.2659,0.4232,0.4990,0.6506,0.4082,0.7478,0.7300


In [172]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.246797503393029, -5.161982855255484, -5.02...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.496304172780948, -5.149596643381521, -5.5...","[-5.350719385070764, -5.18859748532164, -5.869...","[0.12630222505148736, 0.10128546198004572, 0.2..."
1,DecisionTreeRegressor,"[-4.73, -4.73, -6.3, -5.402304814, -5.34486156...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-6.25, -4.94, -4.73, -5.129011186, -5.402304...","[-5.613387552200001, -5.2262010672, -6.3162739...","[0.4541944293171029, 0.3577253490790411, 1.204..."
2,RandomForestRegressor,"[-5.148721896999998, -5.35970046089, -6.019120...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.28165899382, -5.4193078611, -5.7562036672...","[-5.422449812520001, -5.457008017566, -6.05980...","[0.15019383946322165, 0.058247178267046784, 0...."
3,GradientBoostingRegressor,"[-5.215391955604233, -5.26646079162257, -6.196...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.37310896653543, -5.388732657559867, -5.68...","[-5.311519086279125, -5.350012496152736, -6.26...","[0.13334234399881534, 0.20971114116986692, 0.3..."
4,AdaBoostRegressor,"[-5.339912808499999, -5.3478942386, -6.0957380...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.82, -5.65, -6.21, -5.3096230523333325, -5...","[-5.1971847129216115, -5.491373820446061, -6.0...","[0.3184449431520926, 0.1320553481548821, 0.206..."
5,XGBRegressor,"[-5.0274296, -5.2062883, -6.195439, -5.5775614...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.443049, -5.2251496, -5.3186226, -5.386756...","[-5.294223, -5.249493, -6.0947576, -5.8331823,...","[0.1582653, 0.16459233, 0.49150863, 0.6412933,..."
6,ExtraTreesRegressor,"[-5.203494767320002, -5.449526899809999, -5.82...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.103804798990002, -5.535612067779997, -6.0...","[-5.313949375084002, -5.507876729268, -6.17668...","[0.13641627555964603, 0.0590204838140248, 0.20..."
7,LinearRegression,"[-4.6699306736118835, -4.632615135263127, -4.1...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.420015004113733, -4.0, -7.682997522547392...","[-5.165879178872528, -4.380810182161771, -7.41...","[0.3835404978478627, 0.3626452942937592, 0.638..."
8,KNeighborsRegressor,"[-5.366666666666667, -5.3999999999999995, -5.3...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.503333333333334, -5.613333333333333, -5.5...","[-5.508, -5.713333333333333, -5.53533333333333...","[0.0290287214094362, 0.1707890186425604, 0.190..."
9,SVR,"[-5.231561287179898, -5.3269842494696595, -5.5...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.492880373623747, -5.293723337690618, -5.7...","[-5.484890283395411, -5.412472400013679, -5.71...","[0.10486903871959698, 0.14092657533932915, 0.0..."


In [173]:
result_df.to_csv('results/Descriptors/Results_3D_All_desc_MDCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_3D_All_desc_MDCK.csv')

In [174]:
#3d All descriptors const rem
df_train = pd.read_csv('features/Descriptors/Train_3d_all_descriptors_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train,  const_col =  remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_3d_all_descriptors_MDCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 442)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 442)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.105938 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1710
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 114
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.4737,0.5311,0.6883,0.0997,0.3542,0.4258,0.4350,0.5202,0.6595,0.3917,0.6782,0.6749
DecisionTreeRegressor,1.1445,0.8521,1.0698,-1.1752,-0.0053,0.0697,0.4980,0.4926,0.7057,0.3037,0.5711,0.6198
RandomForestRegressor,0.5127,0.5612,0.7161,0.0255,0.3075,0.3214,0.3915,0.4687,0.6257,0.4525,0.7196,0.7052
GradientBoostingRegressor,0.6066,0.6282,0.7789,-0.1529,0.1967,0.2040,0.4216,0.4077,0.6493,0.4105,0.6487,0.6336
AdaBoostRegressor,0.5316,0.5703,0.7291,-0.0103,0.2674,0.3140,0.4201,0.4450,0.6481,0.4126,0.6798,0.7080
XGBRegressor,0.6723,0.6747,0.8199,-0.2777,0.2306,0.2663,0.4548,0.4643,0.6744,0.3640,0.6105,0.6915
ExtraTreesRegressor,0.4918,0.5704,0.7013,0.0654,0.3593,0.3506,0.4048,0.4723,0.6363,0.4339,0.6768,0.6722
LinearRegression,1.1566,0.8931,1.0754,-1.1981,0.1947,0.2327,0.5399,0.5819,0.7348,0.2450,0.6851,0.7300
KNeighborsRegressor,0.5525,0.6025,0.7433,-0.0501,0.3483,0.3923,0.4879,0.5897,0.6985,0.3177,0.5694,0.4876
SVR,0.4838,0.5603,0.6956,0.0805,0.3074,0.2659,0.4232,0.4990,0.6506,0.4082,0.7478,0.7300


In [175]:
result_df.to_csv('results/Descriptors/Results_3D_All_desc_const_rem_MDCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_3D_All_desc_const_rem_MDCK.csv')

In [176]:
#3d All descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_3d_all_descriptors_MDCK.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train,  const_col =  remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_3d_all_descriptors_MDCK.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 370)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 370)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.133709 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1530
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 102
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.4648,0.5433,0.6817,0.1167,0.3641,0.4366,0.4309,0.5101,0.6564,0.3975,0.6821,0.6612
DecisionTreeRegressor,1.1426,0.8283,1.0689,-1.1716,-0.1123,-0.0654,0.4322,0.4642,0.6574,0.3956,0.6346,0.7135
RandomForestRegressor,0.5187,0.5662,0.7202,0.0143,0.2838,0.2947,0.4016,0.4696,0.6337,0.4385,0.6912,0.6722
GradientBoostingRegressor,0.6393,0.6512,0.7996,-0.2150,0.1522,0.2101,0.3625,0.4001,0.6021,0.4931,0.7193,0.7576
AdaBoostRegressor,0.5436,0.5730,0.7373,-0.0332,0.2623,0.3058,0.4059,0.4297,0.6371,0.4324,0.7048,0.6887
XGBRegressor,0.7301,0.7126,0.8545,-0.3876,0.1352,0.1749,0.3722,0.3957,0.6101,0.4795,0.7035,0.6556
ExtraTreesRegressor,0.5078,0.5777,0.7126,0.0350,0.3308,0.3196,0.3939,0.4793,0.6276,0.4491,0.6936,0.6942
LinearRegression,1.2259,0.9155,1.1072,-1.3299,0.1733,0.2157,0.6012,0.5806,0.7753,0.1594,0.6551,0.6942
KNeighborsRegressor,0.6574,0.6349,0.8108,-0.2495,0.2652,0.3595,0.3893,0.5138,0.6239,0.4557,0.6908,0.6391
SVR,0.4862,0.5659,0.6973,0.0759,0.3072,0.2772,0.4105,0.4824,0.6407,0.4260,0.7339,0.6722


In [177]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.265608550646573, -5.387794840214481, -5.41...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.3769956652953965, -5.359131768717777, -5....","[-5.286726097784646, -5.242128383145757, -5.94...","[0.1399782150154841, 0.11476090700880245, 0.19..."
1,DecisionTreeRegressor,"[-4.73, -5.402304814, -6.3, -5.402304814, -5.2...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.65, -5.1, -4.94, -5.744727495, -5.5850266...","[-5.6624253736, -5.2392423748, -6.625777851400...","[0.21093931129705076, 0.3512209325034714, 1.16..."
2,RandomForestRegressor,"[-5.22928399031, -5.570744807419999, -6.020247...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.201601599300001, -5.413932942130002, -5.7...","[-5.390219472192001, -5.461879706212001, -6.18...","[0.1916183910012216, 0.0769381809803867, 0.213..."
3,GradientBoostingRegressor,"[-5.095356843874307, -5.779208702622519, -6.24...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.59715006662455, -5.370719424817681, -5.52...","[-5.413164158034335, -5.461369018299547, -6.37...","[0.1950790693414769, 0.12692979181752959, 0.45..."
4,AdaBoostRegressor,"[-5.16, -5.350850189, -6.189973983055555, -5.5...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.0796147524, -5.65, -5.760148273454546, -5...","[-5.218536372796883, -5.4489980178414505, -6.0...","[0.22384740695507388, 0.20530141380073358, 0.1..."
5,XGBRegressor,"[-5.1071396, -5.660103, -6.1757755, -5.7973194...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.6456203, -5.912598, -5.985625, -5.3570256...","[-5.172892, -5.50604, -6.2951784, -5.8479815, ...","[0.3062994, 0.38251373, 0.21393523, 0.8250141,..."
6,ExtraTreesRegressor,"[-5.392422345539999, -5.58932100413, -6.002171...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.114195309050003, -5.615109756070002, -5.9...","[-5.23293110328, -5.543775010222002, -6.240094...","[0.11963542983793014, 0.08732118164387061, 0.2..."
7,LinearRegression,"[-4.454275426601918, -4.733744078817778, -4.30...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.345455278757832, -4.0, -7.767311008993734...","[-5.2077928581519135, -4.351520216365174, -7.6...","[0.5660458710306174, 0.32951526794121827, 0.49..."
8,KNeighborsRegressor,"[-5.444909164999999, -5.3999999999999995, -5.3...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.503333333333334, -5.613333333333333, -5.5...","[-5.508, -5.626000000000001, -5.56000000000000...","[0.0290287214094362, 0.16096100286853404, 0.10..."
9,SVR,"[-5.243395207846869, -5.353024364440868, -5.60...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.477770594673899, -5.354512497888812, -5.8...","[-5.489178192237473, -5.467714869456471, -5.78...","[0.11264057435247544, 0.11446533283051702, 0.1..."


In [178]:
result_df.to_csv('results/Descriptors/Results_3D_All_desc_LVR_MDCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_3D_All_desc_LVR_MDCK.csv')

In [179]:
#2d and 3d descriptors all
df_train_2d = pd.read_csv('features/Descriptors/Train_2d_all_descriptors_MDCK.csv')
df_train_2d
df_train_3d = pd.read_csv('features/Descriptors/Train_3d_all_descriptors_MDCK.csv')
df_train_3d

df_2d_3d_train = df_train_2d.merge(df_train_3d, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_2d_3d_train.to_csv('features/Descriptors/Train_2d_3d_all_descriptors_MDCK.csv', index=False)
df_2d_3d_train

,ID,SMILES,Permeability,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,1114,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-4.940000,14.922930,14.922930,0.059501,-1.162678,0.238950,27.097561,1139.494,...,0.490993,0.446412,0.502821,0.521474,0.367154,50.115915,697.880017,2474.945996,0.406107,1.391449
1,1113,CC(C)C[C@@H]1NC(=O)[C@@H]2CCCN2C(=O)[C@@H](Cc2...,-5.820000,15.128047,15.128047,0.075261,-1.145488,0.238950,27.097561,1139.494,...,0.530772,0.398556,0.543790,0.527114,0.372276,47.613373,628.467138,2289.825818,0.393991,1.443180
2,1117,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H]2CCCN...,-5.650000,14.908525,14.908525,0.051146,-1.177248,0.238950,27.097561,1139.494,...,0.527383,0.387331,0.523372,0.543333,0.399556,48.105086,653.233321,2640.699755,0.372071,1.466261
3,1119,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-6.250000,14.724823,14.724823,0.030635,-1.196597,0.240803,26.125000,1115.472,...,0.528976,0.349388,0.516241,0.483660,0.323140,49.516594,715.115788,3493.983049,0.317545,1.323041
4,2428,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N(C)[C@@H](C)C(=...,-5.510000,14.918925,14.918925,0.008728,-1.166833,0.250056,27.358974,1091.406,...,0.506249,0.395804,0.498185,0.483571,0.313967,46.999494,637.788214,2722.371988,0.353079,1.295723
5,2446,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N2CCC[C@@H]2C(=O...,-6.400000,14.912312,14.912312,0.031806,-1.168185,0.250056,27.358974,1091.406,...,0.465091,0.397955,0.535696,0.559095,0.348772,46.003992,641.858241,3155.799517,0.294568,1.443562
6,2445,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N2CCC[C@@H]2C(=O...,-4.820000,14.926201,14.926201,0.060156,-1.148185,0.252488,27.358974,1089.434,...,0.493305,0.428676,0.535662,0.498539,0.389916,53.087085,798.688967,3320.153449,0.382972,1.424116
7,2427,CCN1CC(=O)N[C@@H](CC(C)C)C(=O)N(C)[C@@H](C)C(=...,-4.610000,14.950175,14.950175,0.063054,-1.159021,0.252488,27.358974,1089.434,...,0.483498,0.392982,0.564432,0.505432,0.322603,48.069815,689.212021,3344.170973,0.314720,1.392467
8,8145,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.744727,14.598899,14.598899,0.013996,-1.178122,0.218577,28.506849,1048.403,...,0.565467,0.377819,0.494304,0.444191,0.445117,47.071607,591.915175,1902.720960,0.414930,1.383611
9,1107,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](C)N...,-4.610000,14.543197,14.543197,0.095971,-1.130815,0.222694,27.567568,1047.438,...,0.633354,0.305158,0.532685,0.549858,0.399249,48.688812,594.973414,2015.324533,0.450032,1.481792


In [180]:
df_test_2d = pd.read_csv('features/Descriptors/Test_2d_all_descriptors_MDCK.csv')
df_test_2d
df_test_3d = pd.read_csv('features/Descriptors/Test_3d_all_descriptors_MDCK.csv')
df_test_3d

df_2d_3d_test = df_test_2d.merge(df_test_3d, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_2d_3d_test.to_csv('features/Descriptors/Test_2d_3d_all_descriptors_MDCK.csv', index=False)
df_2d_3d_test

,ID,SMILES,Permeability,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,1120,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C...,-6.300000,15.211581,15.211581,0.031713,-1.206465,0.159924,25.279070,1199.634,...,0.517052,0.333526,0.520259,0.510811,0.420493,49.703108,739.996624,3953.646545,0.275867,1.451563
1,1118,CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](Cc2...,-5.350000,14.834083,14.834083,0.037197,-1.184826,0.240803,26.125000,1115.472,...,0.544737,0.347757,0.647006,0.459842,0.383878,47.483083,643.440777,2871.212438,0.338740,1.490726
2,1121,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-6.200000,14.571211,14.571211,0.032067,-1.213008,0.153969,27.307692,1083.386,...,0.569544,0.319304,0.527342,0.614802,0.396168,50.379698,712.333697,3347.440319,0.354317,1.538312
3,8133,CCC[C@@H]1NC(=O)CN(CC)C(=O)[C@H](CC(C)C)NC(=O)...,-5.355561,14.507460,14.507460,0.015782,-1.174377,0.218370,27.694444,1034.376,...,0.575816,0.391394,0.465474,0.444791,0.411105,54.943002,776.072156,2056.663536,0.450816,1.321370
4,8143,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.406714,14.432367,14.432367,0.010248,-1.181980,0.206072,27.492958,1022.365,...,0.521440,0.411553,0.528651,0.525749,0.441578,46.170332,590.730936,2052.174504,0.399489,1.495978
5,8119,CCC[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](CC(C)C)...,-5.073658,14.337271,14.337271,0.011275,-1.179506,0.205195,26.642857,1008.338,...,0.544048,0.342655,0.553639,0.636460,0.287638,45.060837,582.506417,2560.016441,0.330056,1.477737
6,6496,CC(=O)N1CCC[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N[...,-4.860000,14.798396,14.798396,0.082911,-1.653049,0.157962,27.464789,1002.309,...,0.644354,0.296155,0.470177,0.475961,0.342756,49.058557,593.936407,1983.399773,0.466531,1.288893
7,8168,CCOC(=O)[C@@H]1CSCC(=O)N[C@@H](CC)C(=O)N(C)[C@...,-6.031517,14.228063,14.228063,0.008269,-1.181029,0.216153,26.159420,996.327,...,0.567029,0.400623,0.475788,0.516266,0.353488,52.551241,713.790090,1832.795103,0.451477,1.345542
8,8345,CCOC(=O)[C@@H]1CSCC(=O)N[C@@H](CC)C(=O)N(CC(C)...,-7.534591,14.235704,14.235704,0.008614,-1.181469,0.216153,26.159420,996.327,...,0.532833,0.379510,0.478903,0.524793,0.340577,45.177438,575.947872,2255.562804,0.368514,1.344273
9,6423,CC(=O)N1CCC[C@@H]1C(=O)N(C)[C@@H](CC(C)C)C(=O)...,-6.300000,14.765338,14.765338,0.082043,-1.652577,0.137104,27.528571,988.282,...,0.616065,0.311878,0.435148,0.417772,0.254081,49.869778,644.135872,2411.114189,0.424098,1.107002


In [181]:
#All 2d and 3d descriptors
df_train = pd.read_csv('features/Descriptors/Train_2d_3d_all_descriptors_MDCK.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_3d_all_descriptors_MDCK.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 3539)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 3539)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.094941 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10311
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 757
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spl

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3700,0.4928,0.6083,0.2968,0.5468,0.5270,0.3737,0.4398,0.6113,0.4775,0.7374,0.6281
DecisionTreeRegressor,0.8445,0.6943,0.9190,-0.6050,0.2886,0.2197,0.4436,0.5347,0.6661,0.3796,0.6396,0.4215
RandomForestRegressor,0.4081,0.5320,0.6388,0.2245,0.4846,0.4661,0.4094,0.5022,0.6398,0.4276,0.6913,0.6419
GradientBoostingRegressor,0.4615,0.5530,0.6794,0.1228,0.4759,0.4450,0.3693,0.4511,0.6077,0.4835,0.7320,0.6694
AdaBoostRegressor,0.3344,0.4652,0.5783,0.3644,0.6040,0.5869,0.3498,0.4356,0.5914,0.5109,0.7643,0.6226
XGBRegressor,0.3905,0.5083,0.6249,0.2578,0.5528,0.4120,0.4608,0.4876,0.6788,0.3556,0.6015,0.4298
ExtraTreesRegressor,0.3447,0.4654,0.5871,0.3448,0.5955,0.5499,0.4007,0.4763,0.6330,0.4397,0.6816,0.5207
LinearRegression,0.6023,0.6288,0.7761,-0.1447,0.4724,0.5239,0.3999,0.4555,0.6324,0.4408,0.7456,0.8843
KNeighborsRegressor,0.3980,0.4920,0.6309,0.2435,0.5158,0.5162,0.4301,0.5101,0.6558,0.3986,0.6511,0.5034
SVR,0.4149,0.5181,0.6441,0.2115,0.4668,0.4735,0.3905,0.4594,0.6249,0.4539,0.7474,0.7190


In [182]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.215861823469457, -5.251197413111948, -5.91...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.30436639792533, -5.494688351263029, -5.30...","[-5.3532860856156335, -5.359462651125263, -5.8...","[0.0864104636383826, 0.18952743988363308, 0.34..."
1,DecisionTreeRegressor,"[-5.65, -5.32, -6.4, -4.73, -6.321123049, -6.4...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.61, -5.65, -4.94, -5.289036881, -5.185752...","[-5.7620000000000005, -5.812, -5.8489909236000...","[0.6162596855222641, 0.5146416228794557, 1.053..."
2,RandomForestRegressor,"[-5.322499615549999, -5.332195337359998, -5.78...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.292130132720002, -5.73694841523, -5.65896...","[-5.426520200136, -5.600225717594001, -5.92779...","[0.12061704213694667, 0.07915745890583208, 0.1..."
3,GradientBoostingRegressor,"[-5.404278411896555, -5.116849859461513, -6.24...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.002590810908289, -5.486491670275284, -5.1...","[-5.466301361131698, -5.661579386088404, -5.76...","[0.3289466438067385, 0.1612939090713535, 0.455..."
4,AdaBoostRegressor,"[-5.28414528088889, -5.266604474399999, -6.004...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.147142857142858, -5.713151663333334, -5.5...","[-5.438195807200477, -5.67174814902, -5.906186...","[0.2721846802582524, 0.16408910515221672, 0.22..."
5,XGBRegressor,"[-5.38492, -5.6513515, -5.7458606, -5.329964, ...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.1734533, -5.9038725, -5.120241, -5.42298,...","[-5.466357, -5.667393, -5.6188054, -5.4685163,...","[0.31459057, 0.30587858, 0.50324976, 0.1854683..."
6,ExtraTreesRegressor,"[-5.275089099799999, -4.958615817509998, -6.20...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.547947274949996, -5.772567230360003, -5.9...","[-5.482764202564, -5.796431830916001, -5.88468...","[0.259274925334918, 0.1732671033887551, 0.1294..."
7,LinearRegression,"[-4.666061325115393, -4.58641244771306, -4.595...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.714668054255832, -5.376635336304349, -7.9...","[-6.0364269012455125, -5.198613400761761, -7.5...","[0.3539684863695013, 0.49333364346083175, 0.31..."
8,KNeighborsRegressor,"[-5.066666666666666, -5.3999999999999995, -5.8...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333333, -5.2...","[-5.713333333333333, -5.713333333333333, -5.28...","[0.1707890186425604, 0.1707890186425604, 0.189..."
9,SVR,"[-5.297581389384146, -5.414199735919868, -5.74...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.551198356601163, -5.75824272388648, -5.64...","[-5.589487707385979, -5.744942152724975, -5.67...","[0.1608416641904293, 0.21799077517156168, 0.11..."


In [183]:
result_df.to_csv('results/Descriptors/Results_2D_3D_All_desc_MDCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_3D_All_desc_MDCK.csv')

In [184]:
#All 2d and 3d descriptors const rem
df_train = pd.read_csv('features/Descriptors/Train_2d_3d_all_descriptors_MDCK.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train,  const_col =  remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_3d_all_descriptors_MDCK.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 2716)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 2716)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.102863 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10311
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 757
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spl

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3700,0.4928,0.6083,0.2968,0.5468,0.5270,0.3737,0.4398,0.6113,0.4775,0.7374,0.6281
DecisionTreeRegressor,0.8065,0.6899,0.8981,-0.5328,0.3194,0.2568,0.4590,0.5430,0.6775,0.3581,0.6070,0.6033
RandomForestRegressor,0.3938,0.5293,0.6275,0.2517,0.5118,0.4664,0.3954,0.4956,0.6288,0.4471,0.7114,0.6529
GradientBoostingRegressor,0.4205,0.5266,0.6485,0.2008,0.5127,0.4816,0.3698,0.4539,0.6081,0.4829,0.7346,0.6694
AdaBoostRegressor,0.4019,0.4940,0.6339,0.2363,0.5010,0.4796,0.3745,0.4523,0.6120,0.4763,0.7308,0.6474
XGBRegressor,0.3905,0.5083,0.6249,0.2578,0.5528,0.4120,0.4608,0.4876,0.6788,0.3556,0.6015,0.4298
ExtraTreesRegressor,0.3290,0.4536,0.5736,0.3748,0.6163,0.5515,0.3884,0.4593,0.6232,0.4569,0.6955,0.6088
LinearRegression,0.6023,0.6288,0.7761,-0.1447,0.4724,0.5239,0.3999,0.4555,0.6324,0.4408,0.7456,0.8843
KNeighborsRegressor,0.3980,0.4920,0.6309,0.2435,0.5158,0.5162,0.4301,0.5101,0.6558,0.3986,0.6511,0.5034
SVR,0.4148,0.5181,0.6441,0.2116,0.4668,0.4735,0.3905,0.4594,0.6249,0.4539,0.7474,0.7190


In [185]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-5.215861823469457, -5.251197413111948, -5.91...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.30436639792533, -5.494688351263029, -5.30...","[-5.3532860856156335, -5.359462651125263, -5.8...","[0.0864104636383826, 0.18952743988363308, 0.34..."
1,DecisionTreeRegressor,"[-5.65, -5.32, -6.4, -4.73, -7.698970004, -6.4...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.61, -5.65, -4.96, -5.289036881, -5.185752...","[-5.846, -5.978, -5.394945499, -5.720191555000...","[0.6420155761350341, 0.47410547349719545, 0.37..."
2,RandomForestRegressor,"[-5.308648015019995, -5.347294500619999, -5.77...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.326164768600003, -5.697017133289998, -5.7...","[-5.439803698620001, -5.602684791188, -5.92280...","[0.13480912233553652, 0.10109476987396705, 0.1..."
3,GradientBoostingRegressor,"[-5.360576196385327, -5.107849126194527, -6.23...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.0462128995375695, -5.47761216891229, -5.1...","[-5.489873862347201, -5.65962079232964, -5.778...","[0.3329908971086695, 0.20583932695677704, 0.48..."
4,AdaBoostRegressor,"[-5.4392643224, -5.32, -6.148608099733333, -5....",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-4.843003728666667, -6.033048981666667, -5.8...","[-5.321567549531136, -5.731274153179166, -5.96...","[0.312265826498948, 0.20803473051336144, 0.162..."
5,XGBRegressor,"[-5.38492, -5.6513515, -5.7458606, -5.329964, ...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.1734533, -5.9038725, -5.120241, -5.42298,...","[-5.466357, -5.667393, -5.6188054, -5.4685163,...","[0.31459057, 0.30587858, 0.50324976, 0.1854683..."
6,ExtraTreesRegressor,"[-5.3102418248499985, -4.939072497939998, -6.1...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.623684918709999, -5.718333048170001, -5.8...","[-5.528718209311999, -5.764450560206001, -5.92...","[0.2400655572897644, 0.2040893039019574, 0.085..."
7,LinearRegression,"[-4.666061325115393, -4.586412447713056, -4.59...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.714668054255824, -5.376635336304347, -7.9...","[-6.0364269012455125, -5.19861340076176, -7.59...","[0.3539684863695002, 0.49333364346083103, 0.31..."
8,KNeighborsRegressor,"[-5.066666666666666, -5.3999999999999995, -5.8...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.613333333333333, -5.613333333333333, -5.2...","[-5.713333333333333, -5.713333333333333, -5.28...","[0.1707890186425604, 0.1707890186425604, 0.189..."
9,SVR,"[-5.297617734943936, -5.414198849202094, -5.74...",0 -6.300000 1 -5.350000 2 -6.200000 3...,"[[-5.551252993762204, -5.758306287752248, -5.6...","[-5.589487134589357, -5.744969872935161, -5.67...","[0.160828938694295, 0.21800000304981687, 0.118..."


In [186]:
result_df.to_csv('results/Descriptors/Results_2D_3D_All_desc_const_rem_MDCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_3D_All_desc_const_rem_MDCK.csv')

In [187]:
#All 2d and 3d descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_2d_3d_all_descriptors_MDCK.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train,  const_col =  remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_3d_all_descriptors_MDCK.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (51, 1904)
y_train shape:  (51,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (13, 1904)
y_test shape:  (13,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.123855 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7073
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 519
[LightGBM] [Info] Start training from score -5.612293
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spli

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3794,0.5112,0.6159,0.2790,0.5323,0.5130,0.3523,0.4501,0.5936,0.5073,0.7637,0.6336
DecisionTreeRegressor,1.2177,0.8294,1.1035,-1.3144,0.1023,0.1736,0.3971,0.4988,0.6301,0.4448,0.6841,0.6391
RandomForestRegressor,0.4164,0.5380,0.6453,0.2085,0.4738,0.4607,0.4052,0.4929,0.6366,0.4333,0.6951,0.6584
GradientBoostingRegressor,0.4804,0.5792,0.6931,0.0869,0.3989,0.3395,0.3594,0.4344,0.5995,0.4975,0.7345,0.8017
AdaBoostRegressor,0.3943,0.5018,0.6279,0.2507,0.5073,0.5312,0.3655,0.4234,0.6046,0.4889,0.7266,0.6253
XGBRegressor,0.7148,0.6371,0.8454,-0.3585,0.2553,0.2885,0.4820,0.5213,0.6943,0.3260,0.5834,0.5482
ExtraTreesRegressor,0.3198,0.4696,0.5655,0.3922,0.6314,0.5572,0.4102,0.4623,0.6405,0.4264,0.6684,0.5537
LinearRegression,0.5570,0.6056,0.7463,-0.0586,0.4998,0.5474,0.4666,0.5418,0.6831,0.3476,0.7182,0.8099
KNeighborsRegressor,0.4310,0.5109,0.6565,0.1809,0.4830,0.5113,0.4492,0.5683,0.6703,0.3718,0.6278,0.6116
SVR,0.4308,0.5278,0.6563,0.1813,0.4340,0.4446,0.3905,0.4553,0.6249,0.4539,0.7458,0.7410


In [188]:
result_df.to_csv('results/Descriptors/Results_2D_3D_All_desc_LVR_MDCK.csv')
prediction_df.to_csv('results/Descriptors/Prediction_data_2D_3D_All_desc_LVR_MDCK.csv')

In [189]:
#Stacked architecture model
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr
import lightgbm as lgb
import xgboost as xgb
from tqdm import tqdm
def features(df, target_column='Permeability', threshold=0.9):
    correlation_matrix = df.corr()
    
    features_to_drop = set()
    
    for feature in correlation_matrix.columns:
        if feature == target_column:
            continue 
        target_corr = correlation_matrix[target_column][feature]
        
        for other_feature in correlation_matrix.columns:
            if other_feature == feature or other_feature == target_column:
                continue
            
            if abs(correlation_matrix[feature][other_feature]) > threshold:
                other_target_corr = correlation_matrix[target_column][other_feature]

                if abs(other_target_corr) < abs(target_corr):
                    features_to_drop.add(other_feature)
                else:
                    features_to_drop.add(feature)
    selected_features = [col for col in df.columns if col not in features_to_drop and col != target_column]
    
    return selected_features

In [190]:
def remove_low_variance_columns(df, threshold=0.005):
    variances = df.var()
    
    low_variance_columns = variances[variances < threshold].index.tolist()
    
    df_cleaned = df.drop(columns=low_variance_columns)
    
    return df_cleaned, low_variance_columns

In [191]:
from tqdm import tqdm
# 2D and 3D descriptors dataframes
df_desc_train = pd.read_csv('/home/users/akshay/PCPpred/MDCK/features/Descriptors/Train_2d_3d_all_descriptors_MDCK.csv')
df_train = df_desc_train.sort_values(by='ID')
df_train =df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_desc_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
df_desc_test = pd.read_csv('/home/users/akshay/PCPpred/MDCK/features/Descriptors/Test_2d_3d_all_descriptors_MDCK.csv')
df_desc_test = df_desc_test.sort_values(by='ID')
df_desc_test =df_desc_test.dropna()
df_desc_test =  df_desc_test[df_desc_train.columns]


# Fingerprints
df_fp_train = pd.read_csv('/home/users/akshay/PCPpred/MDCK/features/Fingerprints/Train/All_fingerprints_train_MDCK.csv')
df_train = df_fp_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_fp_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
df_fp_test = pd.read_csv('/home/users/akshay/PCPpred/MDCK/features/Fingerprints/Test/All_fingerprints_test_MDCK.csv')
df_fp_test = df_fp_test.sort_values(by='ID')
df_fp_test = df_fp_test.dropna()
df_fp_test =  df_fp_test[df_fp_train.columns]


#Smiles Embeddings
df_emb_train = pd.read_csv('/home/users/akshay/PCPpred/MDCK/features/Embeddings/Train_MoLFormer-XL-both-10pct_model_1_fine_tuned_embeddings_mdck.csv')
df_train = df_emb_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_emb_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
df_emb_test = pd.read_csv('/home/users/akshay/PCPpred/MDCK/features/Embeddings/Test_MoLFormer-XL-both-10pct_model_1_fine_tuned_embeddings_mdck.csv')
df_emb_test = df_emb_test.sort_values(by='ID')
df_emb_test = df_emb_test.dropna()
df_emb_test =  df_emb_test[df_emb_train.columns]

#ATomic features
df_atomic_train = pd.read_csv('/home/users/akshay/PCPpred/MDCK/features/Atomic/Train_all_atomic_desc_MDCK.csv')
df_train = df_atomic_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
df_atomic_train = pd.concat( [df_train[['ID','SMILES','Permeability']],df_train[selected_features] ], axis=1)
# df_atomic_train =pd.concat( [df_train['SMILES'], df_train.select_dtypes(include=['number'])], axis=1)
df_atomic_test = pd.read_csv('/home/users/akshay/PCPpred/MDCK/features/Atomic/Test_all_atomic_desc_MDCK.csv')
df_atomic_test = df_atomic_test.sort_values(by='ID')
df_atomic_test = df_atomic_test.dropna()
df_atomic_test =  df_atomic_test[df_atomic_train.columns]


print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Data Loading completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
df_fp_test = df_fp_test[df_fp_test['ID'].isin(df_desc_test['ID'])]
df_fp_train = df_fp_train[df_fp_train['ID'].isin(df_desc_train['ID'])]

df_emb_test = df_emb_test[df_emb_test['ID'].isin(df_desc_test['ID'])]
df_emb_train = df_emb_train[df_emb_train['ID'].isin(df_desc_train['ID'])]

df_atomic_test = df_atomic_test[df_atomic_test['ID'].isin(df_desc_test['ID'])]
df_atomic_train = df_atomic_train[df_atomic_train['ID'].isin(df_desc_train['ID'])]
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Data Processing completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_desc_train.shape)
print(df_desc_test.shape)
print(df_fp_train.shape)
print(df_fp_test.shape)
print(df_emb_train.shape)
print(df_emb_test.shape)
print(df_atomic_train.shape)
print(df_atomic_test.shape)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_desc_train)
print(df_desc_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_fp_train)
print(df_fp_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_emb_train)
print(df_emb_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_atomic_train)
print(df_atomic_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
target_column = 'Permeability'
def scale_features(df_train, df_test):
    scaler = StandardScaler()
    train_features = df_train.drop(columns=['ID', 'SMILES', target_column])
    test_features = df_test.drop(columns=['ID', 'SMILES', target_column])
    scaler.fit(train_features)
    train_scaled = pd.DataFrame(scaler.transform(train_features), columns=train_features.columns, index=df_train.index)
    test_scaled = pd.DataFrame(scaler.transform(test_features), columns=test_features.columns, index=df_test.index)
    df_train_scaled = pd.concat([df_train[['ID', 'SMILES', target_column]], train_scaled], axis=1)
    df_test_scaled = pd.concat([df_test[['ID', 'SMILES', target_column]], test_scaled], axis=1)
    return df_train_scaled, df_test_scaled

df_desc_train, df_desc_test = scale_features(df_desc_train, df_desc_test)
df_fp_train, df_fp_test = scale_features(df_fp_train, df_fp_test)
df_emb_train, df_emb_test = scale_features(df_emb_train, df_emb_test)
df_atomic_train, df_atomic_test = scale_features(df_atomic_train, df_atomic_test)
models_weak = [
    lgb.LGBMRegressor(objective='regression', metric='rmse', boosting_type='gbdt', num_leaves=31, learning_rate=0.05, random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    KNeighborsRegressor(),
    SVR(),   
    MLPRegressor(random_state=101, max_iter=500),
    DecisionTreeRegressor(random_state=101),
]

models_meta = [
    lgb.LGBMRegressor(objective='regression', metric='rmse', boosting_type='gbdt', num_leaves=31, learning_rate=0.05, random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(),
    KNeighborsRegressor(),
    SVR(),
    MLPRegressor(random_state=101)
]


XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Data Loading completed
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Data Processing completed
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
(51, 175)
(13, 175)
(51, 91)
(13, 91)
(51, 573)
(13, 573)
(51, 8)
(13, 8)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
      ID                                             SMILES  Permeability  \
47  1017  CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)NC(=O)[C...     -5.700000   
43  1018  CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...     -4.960000   
9   1107  CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](C)N...     -4.610000   
41  1109  C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](C)N(C)C(...     -6.120000   
11  1110  CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[

In [192]:
dataframes = [(df_desc_train, df_desc_test), (df_fp_train, df_fp_test), (df_emb_train, df_emb_test), (df_atomic_train, df_atomic_test)]
target_column = 'Permeability'


meta_features_train = []
meta_features_test = []

# Stage 1: Train weak learners with 5-fold cross-validation
for df_train, df_test in tqdm(dataframes, desc="Processing dataframe pairs"):
    X_weak = df_train.drop(columns=['ID', 'SMILES', target_column])
    X_eval = df_test.drop(columns=['ID', 'SMILES', target_column])
    y_weak = df_train[target_column]
    y_eval = df_test[target_column]

    kf = KFold(n_splits=5, shuffle=True, random_state=101)

    # Storing predictions for the current dataframe
    fold_meta_features_train = np.zeros((X_weak.shape[0], len(models_weak)))
    fold_meta_features_test = np.zeros((X_eval.shape[0], len(models_weak)))

    for i, model in tqdm(enumerate(models_weak), desc="Training models"):
        fold_predictions = np.zeros(X_weak.shape[0])
        test_predictions_folds = []

        for train_index, val_index in kf.split(X_weak):
            X_train, X_val = X_weak.iloc[train_index], X_weak.iloc[val_index]
            y_train, y_val = y_weak.iloc[train_index], y_weak.iloc[val_index]
            
            model.fit(X_train, y_train)

            # Predictions for validation set
            fold_predictions[val_index] =  np.clip( model.predict(X_val), -10, -4.0)

            # Predictions for test set
            test_predictions_fold =  np.clip( model.predict(X_eval), -10, -4.0)
            test_predictions_folds.append(test_predictions_fold)

        # Store predictions for the meta-learner
        fold_meta_features_train[:, i] = fold_predictions
        fold_meta_features_test[:, i] = np.mean(test_predictions_folds, axis=0)

    meta_features_train.append(fold_meta_features_train)
    meta_features_test.append(fold_meta_features_test)

# Convert lists to arrays for the meta-learner
meta_features_train = np.hstack(meta_features_train)
meta_features_test = np.hstack(meta_features_test)

print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print("Dimensions of meta_features_train:", meta_features_train.shape)
print("Dimensions of meta_features_test:", meta_features_test.shape)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Stage 1 completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')

# Stage 2: Train the meta-learner using predictions from weak learners
kf = KFold(n_splits=5, shuffle=True, random_state=101)
results = {}
predictions = []
for model in models_meta:
    model_name = model.__class__.__name__
    predictions_train = []
    actual_y_train = []
    
    test_predictions_folds = []

    for train_index, val_index in kf.split(meta_features_train):
        X_fold_train, X_fold_val = meta_features_train[train_index], meta_features_train[val_index]
        y_fold_train, y_fold_val = y_weak.iloc[train_index], y_weak.iloc[val_index]
        
        model.fit(X_fold_train, y_fold_train)

        y_pred_fold = model.predict(X_fold_val)
        y_pred_fold = np.clip(y_pred_fold, -10, -4.0)
        predictions_train.extend(y_pred_fold)
        actual_y_train.extend(y_fold_val)

        # Predictions for test set
        test_predictions_fold = model.predict(meta_features_test)
        test_predictions_fold = np.clip(test_predictions_fold, -10, -4.0)
        test_predictions_folds.append(test_predictions_fold)

    # Metrics
    predictions_test_mean = np.mean(test_predictions_folds, axis=0)
    predictions_test_std = np.std(test_predictions_folds, axis=0)

    mse_train = mean_squared_error(actual_y_train, predictions_train)
    mae_train = mean_absolute_error(actual_y_train, predictions_train)
    rmse_train = np.sqrt(mse_train)
    r2_train = r2_score(actual_y_train, predictions_train)
    pearson_train, _ = pearsonr(actual_y_train, predictions_train)
    spearman_train, _ = spearmanr(actual_y_train, predictions_train)

    mse_test = mean_squared_error(y_eval, predictions_test_mean)
    mae_test = mean_absolute_error(y_eval, predictions_test_mean)
    rmse_test = np.sqrt(mse_test)
    r2_test = r2_score(y_eval, predictions_test_mean)
    pearson_test, _ = pearsonr(y_eval, predictions_test_mean)
    spearman_test, _ = spearmanr(y_eval, predictions_test_mean)
    print(f'{model_name} Evaluation completed: Test R2 score: {r2_test}')

    predictions.append({
            'Model': model_name,
            'Y Train pred': predictions_train,
            'Y Test actual': y_eval,
            'Test prediction folds': test_predictions_folds,
            'Test Predictions Mean': predictions_test_mean,
            'Test Predictions Std': predictions_test_std,

        })

    results[model_name] = {
        'Train MSE (5 fold CV)': mse_train,
        'Train MAE (5 fold CV)': mae_train,
        'Train RMSE (5 fold CV)': rmse_train,
        'Train R2 (5 fold CV)': r2_train,
        'Train PCC (5 fold CV)': pearson_train,
        'Train SCC (5 fold CV)': spearman_train,
        'Test MSE': mse_test,
        'Test MAE': mae_test,
        'Test RMSE': rmse_test,
        'Test R2': r2_test,
        'Test PCC': pearson_test,
        'Test SCC': spearman_test,
    }

results_df = pd.DataFrame(results).T
prediction_df = pd.DataFrame(predictions)
results_df

Processing dataframe pairs:   0%|          | 0/4 [00:00<?, ?it/s]
Training models: 0it [00:00, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.154071 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 711
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 50
[LightGBM] [Info] Start training from score -5.596983
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be


Training models: 1it [01:21, 81.85s/it]
Training models: 2it [01:23, 34.80s/it]
Training models: 3it [01:24, 19.34s/it]
Training models: 4it [01:25, 11.91s/it]
Training models: 5it [05:51, 103.69s/it]
Training models: 6it [05:52, 68.88s/it] 
Training models: 7it [05:54, 46.87s/it]
Training models: 10it [05:55, 35.54s/it][A
Processing dataframe pairs:  25%|██▌       | 1/4 [05:55<17:46, 355.43s/it]
Training models: 0it [00:00, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.144421 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 23
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 4
[LightGBM] [Info] Start training from score -5.596983
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best


Training models: 1it [01:24, 84.18s/it]
Training models: 2it [01:25, 35.52s/it]
Training models: 3it [01:25, 19.43s/it]
Training models: 4it [01:26, 11.89s/it]
Training models: 5it [05:47, 101.88s/it]
Training models: 6it [05:48, 67.65s/it] 
Training models: 7it [05:50, 45.95s/it]
Training models: 10it [05:50, 35.05s/it][A
Processing dataframe pairs:  50%|█████     | 2/4 [11:45<11:45, 352.54s/it]
Training models: 0it [00:00, ?it/s]

...

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.086899 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 373
[LightGBM] [Info] Number of data points in the train set: 41, number of used features: 27
[LightGBM] [Info] Start training from score -5.612466
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


LGBMRegressor Evaluation completed: Test R2 score: 0.2544419563047171
DecisionTreeRegressor Evaluation completed: Test R2 score: 0.5316021536494377
RandomForestRegressor Evaluation completed: Test R2 score: 0.5457752898291747
GradientBoostingRegressor Evaluation completed: Test R2 score: 0.565220814260551
AdaBoostRegressor Evaluation completed: Test R2 score: 0.5387000928683086
XGBRegressor Evaluation completed: Test R2 score: 0.6540581259305868
ExtraTreesRegressor Evaluation completed: Test R2 score: 0.6055384536903479
LinearRegression Evaluation completed: Test R2 score: -1.853152892550331
KNeighborsRegressor Evaluation completed: Test R2 score: 0.3863124365849099
SVR Evaluation completed: Test R2 score: 0.4521509391159426
MLPRegressor Evaluation completed: Test R2 score: -0.2783009303442556


,Train MSE (5 fold CV),Train MAE (5 fold CV),Train RMSE (5 fold CV),Train R2 (5 fold CV),Train PCC (5 fold CV),Train SCC (5 fold CV),Test MSE,Test MAE,Test RMSE,Test R2,Test PCC,Test SCC
LGBMRegressor,0.450004,0.546764,0.670824,0.144748,0.397529,0.359991,0.533171,0.604327,0.730186,0.254442,0.538230,0.515863
DecisionTreeRegressor,0.585946,0.593001,0.765471,-0.113615,0.536902,0.573563,0.334966,0.462344,0.578762,0.531602,0.771365,0.622592
RandomForestRegressor,0.313919,0.421359,0.560285,0.403384,0.643468,0.701876,0.324830,0.420308,0.569938,0.545775,0.767629,0.705237
GradientBoostingRegressor,0.379533,0.498938,0.616063,0.278681,0.599373,0.633984,0.310924,0.414149,0.557605,0.565221,0.782800,0.710746
AdaBoostRegressor,0.317825,0.428440,0.563759,0.395961,0.636394,0.616678,0.329890,0.407433,0.574360,0.538700,0.764091,0.666669
XGBRegressor,0.382043,0.484740,0.618097,0.273911,0.615454,0.665120,0.247394,0.390254,0.497387,0.654058,0.894086,0.878791
ExtraTreesRegressor,0.357907,0.437345,0.598253,0.319783,0.569106,0.629647,0.282091,0.388988,0.531123,0.605538,0.856429,0.719011
LinearRegression,6.238700,2.060846,2.497739,-10.856910,0.041299,0.007510,2.040376,1.160133,1.428417,-1.853153,0.021836,-0.085400
KNeighborsRegressor,0.539000,0.573053,0.734166,-0.024392,0.270617,0.304157,0.438867,0.518875,0.662470,0.386312,0.644838,0.567495
SVR,0.468164,0.549263,0.684225,0.110235,0.357441,0.313308,0.391783,0.471189,0.625926,0.452151,0.714756,0.655650


In [193]:
results_df.to_csv('/home/users/akshay/PCPpred/MDCK/results/Stacked/Results_5_folds_stacked_archi_MDCK.csv')
prediction_df.to_csv('/home/users/akshay/PCPpred/MDCK/results/Stacked/Prediction_data_5_folds_stacked_archi_MDCK.csv')

In [196]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr
import lightgbm as lgb
import xgboost as xgb
from tqdm import tqdm
import joblib

# Ensure the models directory exists
os.makedirs('/home/users/akshay/PCPpred/MDCK/models_MDCK/', exist_ok=True)

# Assuming remove_low_variance_columns and features functions are defined elsewhere
# 2D and 3D descriptors dataframes
df_desc_train = pd.read_csv('/home/users/akshay/PCPpred/MDCK/features/Descriptors/Train_2d_3d_all_descriptors_MDCK.csv')
df_train = df_desc_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'], axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features_desc = features(train, "Permeability")
joblib.dump(selected_features_desc, '/home/users/akshay/PCPpred/MDCK/models_MDCK/selected_features_descriptors.joblib')
df_desc_train = pd.concat([df_train[['ID','SMILES','Permeability']], df_train[selected_features_desc]], axis=1)
df_desc_test = pd.read_csv('/home/users/akshay/PCPpred/MDCK/features/Descriptors/Test_2d_3d_all_descriptors_MDCK.csv')
df_desc_test = df_desc_test.sort_values(by='ID')
df_desc_test = df_desc_test.dropna()
df_desc_test = df_desc_test[df_desc_train.columns]

# Fingerprints
df_fp_train = pd.read_csv('/home/users/akshay/PCPpred/MDCK/features/Fingerprints/Train/All_fingerprints_train_MDCK.csv')
df_train = df_fp_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'], axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features_fp = features(train, "Permeability")
joblib.dump(selected_features_fp, '/home/users/akshay/PCPpred/MDCK/models_MDCK/selected_features_fingerprints.joblib')
df_fp_train = pd.concat([df_train[['ID','SMILES','Permeability']], df_train[selected_features_fp]], axis=1)
df_fp_test = pd.read_csv('/home/users/akshay/PCPpred/MDCK/features/Fingerprints/Test/All_fingerprints_test_MDCK.csv')
df_fp_test = df_fp_test.sort_values(by='ID')
df_fp_test = df_fp_test.dropna()
df_fp_test = df_fp_test[df_fp_train.columns]

# Smiles Embeddings
df_emb_train = pd.read_csv('/home/users/akshay/PCPpred/MDCK/features/Embeddings/Train_MoLFormer-XL-both-10pct_model_1_fine_tuned_embeddings_mdck.csv')
df_train = df_emb_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'], axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features_emb = features(train, "Permeability")
joblib.dump(selected_features_emb, '/home/users/akshay/PCPpred/MDCK/models_MDCK/selected_features_embeddings.joblib')
df_emb_train = pd.concat([df_train[['ID','SMILES','Permeability']], df_train[selected_features_emb]], axis=1)
df_emb_test = pd.read_csv('/home/users/akshay/PCPpred/MDCK/features/Embeddings/Test_MoLFormer-XL-both-10pct_model_1_fine_tuned_embeddings_mdck.csv')
df_emb_test = df_emb_test.sort_values(by='ID')
df_emb_test = df_emb_test.dropna()
df_emb_test = df_emb_test[df_emb_train.columns]

# Atomic features
df_atomic_train = pd.read_csv('/home/users/akshay/PCPpred/MDCK/features/Atomic/Train_all_atomic_desc_MDCK.csv')
df_train = df_atomic_train.sort_values(by='ID')
df_train = df_train.dropna()
train = df_train.drop(['ID','SMILES'], axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features_atomic = features(train, "Permeability")
joblib.dump(selected_features_atomic, '/home/users/akshay/PCPpred/MDCK/models_MDCK/selected_features_atomic.joblib')
df_atomic_train = pd.concat([df_train[['ID','SMILES','Permeability']], df_train[selected_features_atomic]], axis=1)
df_atomic_test = pd.read_csv('/home/users/akshay/PCPpred/MDCK/features/Atomic/Test_all_atomic_desc_MDCK.csv')
df_atomic_test = df_atomic_test.sort_values(by='ID')
df_atomic_test = df_atomic_test.dropna()
df_atomic_test = df_atomic_test[df_atomic_train.columns]

print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Data Loading completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')

# Filter dataframes to have consistent IDs
df_fp_test = df_fp_test[df_fp_test['ID'].isin(df_desc_test['ID'])]
df_fp_train = df_fp_train[df_fp_train['ID'].isin(df_desc_train['ID'])]
df_emb_test = df_emb_test[df_emb_test['ID'].isin(df_desc_test['ID'])]
df_emb_train = df_emb_train[df_emb_train['ID'].isin(df_desc_train['ID'])]
df_atomic_test = df_atomic_test[df_atomic_test['ID'].isin(df_desc_test['ID'])]
df_atomic_train = df_atomic_train[df_atomic_train['ID'].isin(df_desc_train['ID'])]

print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Data Processing completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_desc_train.shape)
print(df_desc_test.shape)
print(df_fp_train.shape)
print(df_fp_test.shape)
print(df_emb_train.shape)
print(df_emb_test.shape)
print(df_atomic_train.shape)
print(df_atomic_test.shape)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_desc_train)
print(df_desc_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_fp_train)
print(df_fp_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_emb_train)
print(df_emb_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print(df_atomic_train)
print(df_atomic_test)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')

target_column = 'Permeability'

def scale_features(df_train, df_test, feature_type):
    scaler = StandardScaler()
    train_features = df_train.drop(columns=['ID', 'SMILES', target_column])
    test_features = df_test.drop(columns=['ID', 'SMILES', target_column])
    scaler.fit(train_features)
    train_scaled = pd.DataFrame(scaler.transform(train_features), columns=train_features.columns, index=df_train.index)
    test_scaled = pd.DataFrame(scaler.transform(test_features), columns=test_features.columns, index=df_test.index)
    df_train_scaled = pd.concat([df_train[['ID', 'SMILES', target_column]], train_scaled], axis=1)
    df_test_scaled = pd.concat([df_test[['ID', 'SMILES', target_column]], test_scaled], axis=1)
    # Save the scaler
    joblib.dump(scaler, f'/home/users/akshay/PCPpred/MDCK/models_MDCK/scaler_{feature_type}.joblib')
    return df_train_scaled, df_test_scaled

df_desc_train, df_desc_test = scale_features(df_desc_train, df_desc_test, 'Descriptor')
df_fp_train, df_fp_test = scale_features(df_fp_train, df_fp_test, 'Fingerprints')
df_emb_train, df_emb_test = scale_features(df_emb_train, df_emb_test, 'Embeddings')
df_atomic_train, df_atomic_test = scale_features(df_atomic_train, df_atomic_test , 'Atomic')

models_weak = [
    lgb.LGBMRegressor(objective='regression', metric='rmse', boosting_type='gbdt', num_leaves=31, learning_rate=0.05, random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    KNeighborsRegressor(),
    SVR(),
    MLPRegressor(random_state=101, max_iter=500),
    DecisionTreeRegressor(random_state=101),
]

models_meta = [
    lgb.LGBMRegressor(objective='regression', metric='rmse', boosting_type='gbdt', num_leaves=31, learning_rate=0.05, random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(),
    KNeighborsRegressor(),
    SVR(),
    MLPRegressor(random_state=101)
]

dataframes = [(df_desc_train, df_desc_test), (df_fp_train, df_fp_test), (df_emb_train, df_emb_test), (df_atomic_train, df_atomic_test)]
data_names = ['descriptors', 'fingerprints', 'embeddings', 'atomic']

meta_features_train = []
meta_features_test = []

# Stage 1: Train weak learners with 5-fold cross-validation
for df_idx, (df_train, df_test) in enumerate(tqdm(dataframes, desc="Processing dataframe pairs")):
    X_weak = df_train.drop(columns=['ID', 'SMILES', target_column])
    X_eval = df_test.drop(columns=['ID', 'SMILES', target_column])
    y_weak = df_train[target_column]
    y_eval = df_test[target_column]

    kf = KFold(n_splits=5, shuffle=True, random_state=101)

    fold_meta_features_train = np.zeros((X_weak.shape[0], len(models_weak)))
    fold_meta_features_test = np.zeros((X_eval.shape[0], len(models_weak)))

    for i, model in tqdm(enumerate(models_weak), desc="Training models", total=len(models_weak)):
        fold_predictions = np.zeros(X_weak.shape[0])
        test_predictions_folds = []

        for fold_idx, (train_index, val_index) in enumerate(kf.split(X_weak)):
            X_train, X_val = X_weak.iloc[train_index], X_weak.iloc[val_index]
            y_train, y_val = y_weak.iloc[train_index], y_weak.iloc[val_index]
            
            model.fit(X_train, y_train)
            
            model_name = model.__class__.__name__
            joblib.dump(model, f'/home/users/akshay/PCPpred/MDCK/models_MDCK/weak_{data_names[df_idx]}_{model_name}_fold_{fold_idx}.joblib')

            fold_predictions[val_index] = np.clip(model.predict(X_val), -10, -4.0)

            test_predictions_fold = np.clip(model.predict(X_eval), -10, -4.0)
            test_predictions_folds.append(test_predictions_fold)

        fold_meta_features_train[:, i] = fold_predictions
        fold_meta_features_test[:, i] = np.mean(test_predictions_folds, axis=0)

    meta_features_train.append(fold_meta_features_train)
    meta_features_test.append(fold_meta_features_test)
    
    joblib.dump(fold_meta_features_train, f'/home/users/akshay/PCPpred/MDCK/models_MDCK/meta_features_train_{data_names[df_idx]}.joblib')
    joblib.dump(fold_meta_features_test, f'/home/users/akshay/PCPpred/MDCK/models_MDCK/meta_features_test_{data_names[df_idx]}.joblib')

meta_features_train = np.hstack(meta_features_train)
meta_features_test = np.hstack(meta_features_test)

joblib.dump(meta_features_train, '/home/users/akshay/PCPpred/MDCK/models_MDCK/meta_features_train_combined.joblib')
joblib.dump(meta_features_test, '/home/users/akshay/PCPpred/MDCK/models_MDCK/meta_features_test_combined.joblib')

print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print("Dimensions of meta_features_train:", meta_features_train.shape)
print("Dimensions of meta_features_test:", meta_features_test.shape)
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')
print('Stage 1 completed')
print('XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX')

# Stage 2: Train the meta-learner using predictions from weak learners
kf = KFold(n_splits=5, shuffle=True, random_state=101)
results = {}
predictions = []
for model in models_meta:
    model_name = model.__class__.__name__
    predictions_train = []
    actual_y_train = []
    
    test_predictions_folds = []

    for fold_idx, (train_index, val_index) in enumerate(kf.split(meta_features_train)):
        X_fold_train, X_fold_val = meta_features_train[train_index], meta_features_train[val_index]
        y_fold_train, y_fold_val = y_weak.iloc[train_index], y_weak.iloc[val_index]
        
        model.fit(X_fold_train, y_fold_train)
        
        joblib.dump(model, f'/home/users/akshay/PCPpred/MDCK/models_MDCK/meta_{model_name}_fold_{fold_idx}.joblib')

        y_pred_fold = model.predict(X_fold_val)
        y_pred_fold = np.clip(y_pred_fold, -10, -4.0)
        predictions_train.extend(y_pred_fold)
        actual_y_train.extend(y_fold_val)

        test_predictions_fold = model.predict(meta_features_test)
        test_predictions_fold = np.clip(test_predictions_fold, -10, -4.0)
        test_predictions_folds.append(test_predictions_fold)

    # Metrics
    predictions_test_mean = np.mean(test_predictions_folds, axis=0)
    predictions_test_std = np.std(test_predictions_folds, axis=0)

    mse_train = mean_squared_error(actual_y_train, predictions_train)
    mae_train = mean_absolute_error(actual_y_train, predictions_train)
    rmse_train = np.sqrt(mse_train)
    r2_train = r2_score(actual_y_train, predictions_train)
    pearson_train, _ = pearsonr(actual_y_train, predictions_train)
    spearman_train, _ = spearmanr(actual_y_train, predictions_train)

    mse_test = mean_squared_error(y_eval, predictions_test_mean)
    mae_test = mean_absolute_error(y_eval, predictions_test_mean)
    rmse_test = np.sqrt(mse_test)
    r2_test = r2_score(y_eval, predictions_test_mean)
    pearson_test, _ = pearsonr(y_eval, predictions_test_mean)
    spearman_test, _ = spearmanr(y_eval, predictions_test_mean)
    

    predictions.append({
        'Model': model_name,
        'Y Train pred': predictions_train,
        'Y Test actual': y_eval,
        'Test prediction folds': test_predictions_folds,
        'Test Predictions Mean': predictions_test_mean,
        'Test Predictions Std': predictions_test_mean,
    })

    results[model_name] = {
        'Train MSE (5 fold CV)': mse_train,
        'Train MAE (5 fold CV)': mae_train,
        'Train RMSE (5 fold CV)': rmse_train,
        'Train R2 (5 fold CV)': r2_train,
        'Train PCC (5 fold CV)': pearson_train,
        'Train SCC (5 fold CV)': spearman_train,
        'Test MSE': mse_test,
        'Test MAE': mae_test,
        'Test RMSE': rmse_test,
        'Test R2': r2_test,
        'Test PCC': pearson_test,
        'Test SCC': spearman_test,
    }

results_df = pd.DataFrame(results).T

XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Data Loading completed
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Data Processing completed
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
(51, 175)
(13, 175)
(51, 91)
(13, 91)
(51, 573)
(13, 573)
(51, 8)
(13, 8)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
      ID                                             SMILES  Permeability  \
47  1017  CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)NC(=O)[C...     -5.700000   
43  1018  CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)N(C)C(=O...     -4.960000   
9   1107  CC(C)C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](C)N...     -4.610000   
41  1109  C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](C)N(C)C(...     -6.120000   
11  1110  CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[

Training models:   0%|          | 0/10 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.071017 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 711
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 50
[LightGBM] [Info] Start training from score -5.596983
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be


Training models:  10%|█         | 1/10 [00:00<00:05,  1.66it/s]

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

...

[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 0
[LightGBM] [Info] Start training from score -5.596983
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the spl


Training models:  10%|█         | 1/10 [00:00<00:02,  3.25it/s]

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f


Training models:  60%|██████    | 6/10 [00:04<00:03,  1.32it/s]/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(

Processing dataframe pairs: 100%|██████████| 4/4 [00:27<00:00,  6.90s/it]
/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/sklearn/utils/depr

XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Dimensions of meta_features_train: (51, 40)
Dimensions of meta_features_test: (13, 40)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
Stage 1 completed
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000348 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 70
[LightGBM] [Info] Number of data points in the train set: 40, number of used features: 5
[LightGBM] [Info] Start training from score -5.596983
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: